# PVRPTW v27 — Methodology Upgrade

Implements §5–§10 of `methodology_v4.tex`.

**Changes vs v26:**
- **§5** Vehicle-type preprocessing — `_preprocess_vehicle_types`
- **§6** Improved Clarke–Wright construction — `_icw_construct`
  (deterministic `cw_warmstart_day` retained as ICW baseline & preprocessing $\bar Z$ source)
- **§7** Or-opt segment relocation — `_or_opt`, integrated into `_local_search`
- **§9** Exact column generation with DP-based pricing — `_pricing_dp`, `_solve_group_column_gen`;
  replaces the restricted-master heuristic and provides certified LP lower bound $Z_\text{LP}$
- **§10** Overall algorithm — modified main loop with 5-stage pipeline

**Preserved verbatim from v26:**
- Formulation constants (R1–R11, T1–T6, D1–D4)
- SHL / day / dispatch-group decomposition
- `_simulate`, `_route_miles`, `_dp_sorted`, `_dp_of_node`, `_fits_vehicle`, `_fits_any_vehicle`,
  `cw_warmstart_day` (baseline), `_two_opt`, `_identify_dispatch_groups`
- `_solve_group_milp` (Variant B compact MILP with DFJ)
- Output/summary code

**Integration:**
1. Run the **Section A** cell to load module-level constants (or paste into your existing config cell).
2. Run the **Section B** cell to define the new `solve()` (replaces the v26 `solve` in-memory).
3. Everything else in the notebook (`load_data`, `build_arcs`, KPIs, mapping, `baseline_routes`) is unchanged.

All Xpress calls use `import xpress as xp` consistent with your notebook.


In [1]:
import math, time, sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import xpress as xp

SHL_NAME   = "Plymouth" 
STAGE      = 4              # 1=pattern, 2=vehicles, 3=routing, 4=time windows

LOG        = True
TIME_LIMIT = 7200.0         # seconds per SHL, default

DATA_DIR = Path(
    "/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6"
)
OUT_DIR = DATA_DIR / "Analysis_output" / "vanilla_v26_complete_changes_allvehicles_last"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

# Monthly-demand source (the 12-month utilisation file)
UTIL_FILE = DATA_DIR / "Routine_Round_Utilisation_Data.xlsx"


DATA_DIR: /Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6
OUT_DIR:  /Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6/Analysis_output/vanilla_v26_complete_changes_allvehicles_last


In [2]:
# Constants and configuration  — v15 (revised)
# --------------------------------------------------------------------------------
# KEY CHANGES vs v14:
#   1. Fleet sizing: Cd = max(C_baseline, n_dispatch_groups) — data-driven, no manual padding
#   2. Electric Van max_range_mi: 100 -> 180 (placeholder was too tight; 180 mi is
#      a conservative real-world single-charge figure for a Vivaro-e class van)
#   3. IIS_DEBUG default False (presolve=0 was disabling presolve for ALL solves,
#      massively slowing the solver; only enable for targeted debugging)
# --------------------------------------------------------------------------------
DAYS        = [1, 2, 3, 4, 5]
DAY_NAMES   = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri"}
SLOT_TO_DAY = {"T1": 1, "T2": 2, "T3": 3, "T4": 4, "T5": 5}
PRODUCTS    = ["Blood", "Platelet", "Frozen", "Other"]
WEEKDAY_SLOTS = ["T1", "T2", "T3", "T4", "T5"]
SAT_SLOTS     = ["TSAT"]
SUN_SLOTS     = ["TSUN"]

U_DEFAULT   = 8.0    # v19: default shift ceiling; solve() computes per-SHL from perishability mix
U           = U_DEFAULT  # global alias for cells outside solve() (load_data, post-hoc, run)
KM_TO_MI    = 0.621371

EXCLUDE_CENTRES = ["Other Blood Service", "Welsh Blood Service"]

DEPOT_COORDS = {
    "Barnsley":   (53.5519, -1.4894), "Basildon":    (51.5640,  0.4745),
    "Birmingham": (52.4581, -1.9239), "Cambridge":   (52.1995,  0.1324),
    "Colindale":  (51.5901, -0.2784), "Filton":      (51.5280, -2.5640),
    "Lancaster":  (54.0475, -2.8157), "Liverpool":   (53.3454, -2.8574),
    "Manchester": (53.4631, -2.2236), "Newcastle":   (54.9869, -1.6085),
    "Oxford":     (51.7630, -1.2099), "Plymouth":    (50.4264, -4.0742),
    "Southampton": (50.9366, -1.4250), "Tooting":     (51.4323, -0.1590),
}

# Container capacities per Va-Q-Tec DAT48/14
BOX_CAPACITY_PER_PRODUCT = {
    "Blood":    12,
    "Platelet": 15,
    "Frozen":   10,
    "Other":    10,
}
BOX_CAPACITY_AGGREGATE = 47

# Vehicle types — v22 capacity rules:
#   * van / van-like        -> 35 boxes max
#   * car                   -> 10 boxes max
#   * Luton / BMV / 7.5T / lorry-class -> EXCLUDED (not suitable for
#     delivery into hospital locations; the old categoriser folded these
#     into the large-van class, silently inflating fleets)
VEHICLE_TYPES = {
    "Diesel Van": {
        "boxes_capacity":  35,
        "co2_kg_per_mile": 0.2791,
        "cost_per_mile":   0.45,
        "max_range_mi":    None,
    },
    "Car": {
        "boxes_capacity":  10,
        "co2_kg_per_mile": 0.2500,   # DEFRA-average car per-mile figure (placeholder; below van)
        "cost_per_mile":   0.35,
        "max_range_mi":    None,
    },
    "Electric Van": {
        "boxes_capacity":  35,       # a van by body type -> van capacity rule applies
        "co2_kg_per_mile": 0.058,
        "cost_per_mile":   0.38,
        "max_range_mi":    180.0,    # conservative Vivaro-e range
    },
}

def _load_fleet_composition(data_dir):
    fpath = data_dir / "fleet_clean.csv"
    if not fpath.exists():
        return None
    f = pd.read_csv(fpath)
    
    # Heavy / non-hospital delivery vehicles only
    EXCLUDE_KW = ("7.5t", "7.5 t", "luton", "bmv", "box van", "boxvan",
                  "tail lift", "taillift", "lorry", "truck", "hgv", "18t", "12t")
                  
    CAR_KW = ("car", "estate", "saloon", "hatchback", "suv",
              "astra", "corsa", "focus", "fiesta", "golf", "insignia",
              "octavia", "passat", "auris", "prius", "leaf")
                  
    VAN_KW = ("van", "transit", "sprinter", "vivaro", "vito", "caddy",
              "berlingo", "partner", "dispatch", "trafic", "crafter",
              "relay", "boxer", "ducato", "transporter", "expert",
              "combo", "kangoo", "delivery", "compact")
              
    def categorise(style):
        s = str(style or "").lower()
        if any(k in s for k in EXCLUDE_KW):
            return None                                  # unsuitable heavy vehicles
        if any(k in s for k in CAR_KW):
            return "Car"                                 # car-class vehicles (10 boxes max)
        if any(k in s for k in VAN_KW):
            return "Electric Van" if "electric" in s else "Diesel Van"
        if "electric" in s:                              # electric, body unspecified -> van-like
            return "Electric Van"
        return None                                      # unrecognised -> excluded + reported
        
    f["_route_class"] = f["Style"].apply(categorise)
    excluded = f[f["_route_class"].isna()]
    
    if len(excluded):
        _styles = excluded["Style"].astype(str).value_counts()
        print(f"  Fleet exclusions ({len(excluded)} vehicles not suitable / unrecognised):")
        for st, cnt in _styles.items():
            print(f"    x{cnt:<3d} {st!r}")
        print("    (add any wrongly-excluded van/car styles to VAN_KW / CAR_KW in cell 2)")
        
    routine = f[f["_route_class"].notna()]
    comp = {}
    for shl, sub in routine.groupby("SHL"):
        comp[str(shl)] = sub["_route_class"].value_counts().to_dict()
        
    print(f"  Real fleet composition loaded from fleet_clean.csv "
          f"({len(routine)} routine-eligible vehicles across {len(comp)} depots; "
          f"capacity rules: van=35 boxes, car=10 boxes).")
    return comp

FLEET_COMPOSITION_OVERRIDE = _load_fleet_composition(DATA_DIR) or {}

BSMS_MIN_VISITS_BY_TIER = {
    "Very High": 5,
    "High":      3,
    "Moderate":  2,
    "Low":       1,
    "Very Low":  1,
}

# Fleet-sizing knobs — FIX: Barnsley headroom restored to 2
FLEET_SIZE_OVERRIDE = {}

# ── Actual delivery fleet per SHL (from NHSBT_FLEET_LIST_June_2026.xlsx) ──
# Counts all delivery-capable vehicles: B1, B2/B3/I/K2, F/F1, BE1, B4, G, D/D2, BMV, H2
FLEET_FROM_DATA = {
    "Barnsley": 36, "Basildon": 7, "Birmingham": 22, "Cambridge": 17,
    "Colindale": 30, "Filton": 36, "Lancaster": 4, "Liverpool": 15,
    "Manchester": 24, "Newcastle": 17, "Oxford": 14, "Plymouth": 4,
    "Southampton": 13, "Tooting": 32,
}

DROP_UNREACHABLE = False

# Speed-calibration robustness
CLAMP_CALIB_SPEED       = True
CALIB_MPH_FLOOR         = 20.0
CALIB_MPH_CEIL          = 70.0
CALIB_MIN_LEG_MI        = 5.0
CALIB_MIN_IMPLIED_MPH   = 15.0

# Window-relaxation policy
ABSORB_RESIDUAL_INTO_DP = True
ABSORB_MAX_H            = 0.5
MAX_WAIT_H_DEFAULT      = 2.0    # v19: default; solve() computes per-SHL from dispatch gaps
MAX_WAIT_H              = MAX_WAIT_H_DEFAULT  # global alias for cells outside solve()

PATCH_EXCLUDE = {}
REPLAY_BASELINE_FEASIBILITY = True

# Solver knobs — FIX: IIS_DEBUG off by default (presolve=0 was killing performance)
SYMMETRY_BREAK = True
IIS_DEBUG      = False     # v18: IIS disabled entirely — no file output     # v15 FIX: was True; presolve=0 disabled presolve for ALL solves
PER_SHL_TIME_LIMIT = {"Tooting": 43200.0, "Barnsley": 43200.0, "Colindale": 43200.0, "Oxford": 21600.0, "Newcastle": 21600.0}
ARRIVAL_WINDOW_SLACK_H = 0.5

USE_PRODUCT_PERISHABILITY = True
PERISH_HOURS = {
    "Blood":    9.0,
    "Platelet": 8.0,
    "Frozen":   9.5,
    "Other":    8.0,
}

# Objective components
CARBON_PRICE_PER_KG = 0.135
ROUTE_PENALTY = 0.0
# Utilities and last-resort defaults
SERVICE_BASE_MIN    = 15.0   # minutes of unloading/handover per stop
SERVICE_PER_BOX_MIN = 0.0    # must be 0 so s_node == S (h2h dwell deduction); otherwise time windows break
S           = SERVICE_BASE_MIN / 60.0    # dwell deduction for h2h tau — tied to SERVICE_BASE_MIN for consistency
ROAD_FACTOR_DEFAULT   = 1.0
SPEED_KMH_DEFAULT     = 80.0
SPEED_MPH_DEFAULT     = SPEED_KMH_DEFAULT * KM_TO_MI
FALLBACK_MPH_DEFAULT  = SPEED_MPH_DEFAULT
PENALTY_LATE    = 1000.0
SHORT_ARC_MI    = 0.0
DISPLAY_SNAP_MIN = 1

COLORS = [
    "#E63946","#457B9D","#2A9D8F","#E9C46A","#F4A261",
    "#8338EC","#06D6A0","#FB5607","#3A86FF","#FFBE0B",
    "#FF006E","#8AC926","#1982C4","#6A4C93","#FF595E",
    "#52B788","#F77F00","#4CC9F0","#7400B8","#80B918",
]

def default_fleet_composition(Cd):
    # Build fleet from current VEHICLE_TYPES (supports E0/E2/E3 scenarios)
    types = list(VEHICLE_TYPES.keys())
    if not types:
        return {"Diesel Van": Cd}
    per_type = max(1, Cd // len(types))
    remainder = Cd % len(types)
    comp = {}
    for i, tn in enumerate(types):
        comp[tn] = per_type + (1 if i < remainder else 0)
    # Trim to Cd total
    total = sum(comp.values())
    while total > Cd and total > 0:
        for tn in reversed(list(comp.keys())):
            if comp[tn] > 0:
                comp[tn] -= 1
                total -= 1
                if total <= Cd:
                    break
    return comp

def resolve_fleet_composition(shl_name, Cd):
    real = FLEET_COMPOSITION_OVERRIDE.get(shl_name)
    if not real:
        return default_fleet_composition(Cd)
    total = sum(real.values())
    if total == 0:
        return default_fleet_composition(Cd)
    scaled = {t: max(0, round(cnt / total * Cd)) for t, cnt in real.items()}
    drift = Cd - sum(scaled.values())
    if drift != 0:
        biggest = max(scaled, key=lambda k: scaled[k])
        scaled[biggest] = max(0, scaled[biggest] + drift)
    for tname in VEHICLE_TYPES:
        scaled.setdefault(tname, 0)
    return scaled

def safe_prob_attr(prob, name, default=None):
    try:
        return getattr(prob.attributes, name)
    except Exception:
        return default

def _rowcount(prob, default=0):
    for nm in ("rows", "originalrows", "numcon"):
        try:
            val = getattr(prob.attributes, nm)
            if val is not None:
                return val
        except Exception:
            pass
    return default

print("Constants loaded (v19: SHL-adaptive MAX_WAIT / fleet / perishability, CW dispatch-group fix).")


def peak_concurrency(intervals):
    """Max overlapping count over a list of (start_h, end_h). Sweep line."""
    events = []
    for a, b in intervals:
        if a is None or b is None or b < a:
            continue
        events.append((a, 1))
        events.append((b, -1))
    events.sort(key=lambda e: (e[0], -e[1]))   # departures before returns at ties
    cur = peak = 0
    for _, delta in events:
        cur += delta
        peak = max(peak, cur)
    return peak
 
 
def peak_vehicles_optimised(routes_by_day):
    """Peak concurrent vehicles per day in the solved schedule."""
    out = {}
    for t, routes in routes_by_day.items():
        iv = [(r.get("route_depart_h"),
               (r.get("route_depart_h") or 0) + (r.get("dur_h") or 0))
              for r in routes]
        out[t] = peak_concurrency(iv)
    return out
 
 
def peak_vehicles_baseline(all_trips, tau=None, lnodes=None):
    """Peak concurrent vehicles per day in the historical schedule.
 
    Uses recorded departure and last arrival, plus the modelled return leg where
    tau/lnodes are supplied. Without a vehicle identifier in the trips data this
    is an UPPER BOUND on the historical fleet: it assumes no vehicle serves two
    overlapping rounds, which is true by construction, but it cannot detect a
    vehicle that returns and departs again — which is exactly what the optimiser
    does. State it as such.
    """
    lnode_idx = ({(pc, s): i for i, (pc, s) in enumerate(lnodes, start=1)}
                 if lnodes else {})
    by_day = defaultdict(list)
    for r in all_trips:
        t = r.get("_day")
        stops = r.get("stops", [])
        dep = r.get("route_depart_h")
        if dep is None and stops:
            dep = min(s.get("depart_h", s.get("arrival_h", 0.0)) for s in stops)
        last_arr = max((s.get("arrival_h", 0.0) for s in stops), default=None)
        ret = 0.0
        hosp = r.get("hospitals", [])
        if tau and hosp:
            ni = lnode_idx.get((hosp[-1], 0))
            if ni is not None:
                ret = tau.get((ni, 0), 0.0)
        if dep is not None and last_arr is not None:
            by_day[t].append((dep, last_arr + ret))
    return {t: peak_concurrency(iv) for t, iv in by_day.items()}
 
 
def fleet_report(shl_name, result, all_trips, tau=None, lnodes=None):
    """Report peak vehicle requirement on both sides, with definitions stated."""
    opt = peak_vehicles_optimised(result["routes_by_day"])
    bsl = peak_vehicles_baseline(all_trips, tau, lnodes)
    n_opt_rounds = sum(len(v) for v in result["routes_by_day"].values())
    n_bsl_rounds = len(all_trips)
    row = dict(
        SHL=shl_name,
        opt_rounds=n_opt_rounds,
        opt_peak_vehicles=max(opt.values()) if opt else None,
        opt_rounds_per_vehicle_day=round(
            n_opt_rounds / max(len(DAYS) * max(opt.values(), default=1), 1), 2),
        bsl_rounds=n_bsl_rounds,
        bsl_peak_vehicles_upper=max(bsl.values()) if bsl else None,
        C_baseline_trip_count=result.get("C_baseline"),
    )
    print(f"\n  FLEET REQUIREMENT — {shl_name}")
    print(f"    optimised: {row['opt_rounds']} rounds on "
          f"{row['opt_peak_vehicles']} vehicles "
          f"({row['opt_rounds_per_vehicle_day']} rounds/vehicle/day)")
    print(f"    baseline:  {row['bsl_rounds']} rounds, peak concurrency "
          f"<= {row['bsl_peak_vehicles_upper']} (UPPER BOUND — no vehicle IDs "
          f"in the trips data)")
    print(f"    C_baseline = {row['C_baseline_trip_count']} is the count of "
          f"distinct baseline TRIPS, not vehicles — do not compare it directly.")
    return row

  Fleet exclusions (148 vehicles not suitable / unrecognised):
    x71  '7.5t box van c/w tailllift'
    x29  '9 Seat MPV'
    x19  '7.5t Fridge box with Tail Lift (BMV)'
    x7   '17 seat minibus'
    x7   'LWB Delivery van with internal tail lift'
    x5   'FOR-TOURNEO-1.8T'
    x4   '3.5t box van c/w tailllift'
    x2   'Recruitment vehicle (With AC)'
    x1   '15 seat minibus'
    x1   '7.5t Box Van c/w Tail lift'
    x1   'MER-313-3.9T'
    x1   'FORD CUSTOM TOURNEO TITANIUM'
    (add any wrongly-excluded van/car styles to VAN_KW / CAR_KW in cell 2)
  Real fleet composition loaded from fleet_clean.csv (169 routine-eligible vehicles across 14 depots; capacity rules: van=35 boxes, car=10 boxes).
Constants loaded (v19: SHL-adaptive MAX_WAIT / fleet / perishability, CW dispatch-group fix).


In [3]:
from pathlib import Path
OUT = Path('/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6/Analysis_output/vanilla_v26_complete_vehicles_allkpis')
files = sorted(OUT.glob('kpi_*_pooled.csv'))
files = [f for f in files if 'stops' not in f.name and 'shl' not in f.name.lower() and 'summary' not in f.name and 'progression' not in f.name]
for f in files:
    import pandas as pd
    df = pd.read_csv(f)
    hb = df['Hidden_Breach'].sum() if 'Hidden_Breach' in df.columns else 0
    lt = (df['N_Late_Arrivals'] > 0).sum() if 'N_Late_Arrivals' in df.columns else 0
    print(f"{f.name:40s}  rounds={len(df):3d}  breach={int(hb):3d}  late={int(lt):3d}")
print(f"\nTotals across {len(files)} files")

kpi_barnsley_pooled.csv                   rounds=101  breach= 54  late=  5
kpi_basildon_pooled.csv                   rounds= 40  breach= 15  late=  0
kpi_birmingham_pooled.csv                 rounds= 85  breach= 40  late= 50
kpi_cambridge_pooled.csv                  rounds= 45  breach= 25  late=  0
kpi_colindale_pooled.csv                  rounds= 90  breach= 16  late=  2
kpi_filton_pooled.csv                     rounds= 80  breach= 10  late=  0
kpi_lancaster_pooled.csv                  rounds= 20  breach= 10  late=  0
kpi_liverpool_pooled.csv                  rounds= 45  breach=  6  late=  5
kpi_manchester_pooled.csv                 rounds= 50  breach=  0  late=  0
kpi_newcastle_pooled.csv                  rounds= 40  breach= 25  late=  5
kpi_oxford_pooled.csv                     rounds= 70  breach= 40  late=  5
kpi_plymouth_pooled.csv                   rounds= 40  breach=  5  late= 15
kpi_southampton_pooled.csv                rounds= 80  breach= 40  late=  5
kpi_tooting_pooled.csv   

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Monthly demand loader — computes per-hospital, per-product, per-DOW demand
# parameters from the 12-month Routine Round Utilisation Data, one set per month.
#
# Returns a dict: { "2025-06": {(pulse_code, product, day_int): median_demand}, ... }
# where day_int ∈ {1..5} (Mon=1 … Fri=5).
#
# This replaces the static demand_params.csv (which holds a single all-months
# median) with month-specific medians, enabling a 12-month rolling solve.
# ═══════════════════════════════════════════════════════════════════════════════

def load_monthly_demand(util_path, verbose=True):
    """Load the utilisation file and compute per-month demand parameters.
    
    Returns
    -------
    monthly_demand : dict[str, dict[tuple, float]]
        Keys are month strings ("2025-06", …). Values are dicts keyed by
        (pulse_code, product, day_int) → median demand (units).
    monthly_boxes : dict[str, dict[tuple, float]]
        Same structure but values are median box counts per delivery.
    month_labels : list[str]
        Sorted list of month strings.
    util_df : pd.DataFrame
        The raw utilisation DataFrame for downstream analysis.
    """
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df = pd.read_excel(util_path)
    
    df["Month"] = df["Period"].dt.to_period("M").astype(str)
    df["DOW_int"] = df["Day"].dt.dayofweek  # 0=Mon … 4=Fri, 5=Sat, 6=Sun
    df["Total_Units"] = df["Blood"] + df["Platelet"] + df["Frozen"] + df["Other"]
    
    month_labels = sorted(df["Month"].unique())
    
    # Product columns → product names matching PRODUCTS constant
    product_cols = {"Blood": "Blood", "Platelet": "Platelet", "Frozen": "Frozen", "Other": "Other"}
    
    monthly_demand = {}
    monthly_boxes  = {}
    
    for month in month_labels:
        mdf = df[df["Month"] == month]
        # Weekday filter (Mon–Fri only, DOW_int 0–4)
        wdf = mdf[mdf["DOW_int"] <= 4]
        
        demand_dict = {}
        boxes_dict  = {}
        
        for pc in wdf["Pulse Code"].unique():
            pc_df = wdf[wdf["Pulse Code"] == pc]
            for dow_int in range(5):  # 0=Mon … 4=Fri
                day_int = dow_int + 1  # 1=Mon … 5=Fri (model convention)
                dow_df = pc_df[pc_df["DOW_int"] == dow_int]
                if len(dow_df) == 0:
                    continue
                # Per-product median demand
                for col, prod in product_cols.items():
                    med = float(dow_df[col].median())
                    if med > 0:
                        demand_dict[(str(pc), prod, day_int)] = med
                # Box median
                box_med = float(dow_df["Boxes"].median())
                boxes_dict[(str(pc), day_int)] = box_med
        
        monthly_demand[month] = demand_dict
        monthly_boxes[month]  = boxes_dict
    
    if verbose:
        n_months = len(month_labels)
        n_hosp = df["Pulse Code"].nunique()
        n_rows = len(df)
        print(f"  Monthly demand loaded: {n_months} months, {n_hosp} hospitals, "
              f"{n_rows:,} delivery events")
        print(f"  Months: {month_labels[0]} … {month_labels[-1]}")
        # Summary stats
        for m in [month_labels[0], month_labels[-1]]:
            d = monthly_demand[m]
            n_keys = len(d)
            tot = sum(d.values())
            print(f"    {m}: {n_keys:,} demand entries, total median demand = {tot:,.0f} units")
    
    return monthly_demand, monthly_boxes, month_labels, df


def load_allmonths_demand(util_path, verbose=True):
    """Compute the all-months-pooled median (equivalent to demand_params.csv)."""
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df = pd.read_excel(util_path)
    
    df["DOW_int"] = df["Day"].dt.dayofweek
    wdf = df[df["DOW_int"] <= 4]
    
    product_cols = {"Blood": "Blood", "Platelet": "Platelet", "Frozen": "Frozen", "Other": "Other"}
    demand_dict = {}
    
    for pc in wdf["Pulse Code"].unique():
        pc_df = wdf[wdf["Pulse Code"] == pc]
        for dow_int in range(5):
            day_int = dow_int + 1
            dow_df = pc_df[pc_df["DOW_int"] == dow_int]
            if len(dow_df) == 0:
                continue
            for col, prod in product_cols.items():
                med = float(dow_df[col].median())
                if med > 0:
                    demand_dict[(str(pc), prod, day_int)] = med
    
    if verbose:
        print(f"  All-months pooled demand: {len(demand_dict):,} entries")
    return demand_dict


# Load on import if the file exists
_MONTHLY_DEMAND = None
_MONTHLY_BOXES  = None
_MONTH_LABELS   = None
_UTIL_DF        = None

if UTIL_FILE.exists():
    _MONTHLY_DEMAND, _MONTHLY_BOXES, _MONTH_LABELS, _UTIL_DF = \
        load_monthly_demand(UTIL_FILE)
else:
    print(f"  ⚠ Utilisation file not found: {UTIL_FILE}")
    print(f"    Monthly demand will not be available; solver will use demand_params.csv")

print("Monthly demand loader defined (v24).")


  Monthly demand loaded: 12 months, 210 hospitals, 108,192 delivery events
  Months: 2025-06 … 2026-05
    2025-06: 1,547 demand entries, total median demand = 9,257 units
    2026-05: 1,436 demand entries, total median demand = 8,157 units
Monthly demand loader defined (v24).


In [5]:
# ==============================================================================
# CELL 4 — load_data  (v28, corrected)
#
# WHAT CHANGED FROM THE PASTED VERSION
#   1. The P162 diagnostic was pasted INSIDE load_data with PC/SHL hardcoded to
#      "P162"/"Tooting", so it ran on every SHL and printed a Tooting-specific
#      report for Barnsley, Filton, etc.  It is now a separate module-level
#      function, diagnose_missing_hospital(), called on demand.
#   2. _DAYS_LOOKUP and _DAY_MAP were dropped in the merge; days_from_field()
#      referenced them and would have raised NameError.  Restored.
#   3. coord_lu was returned but never built (the merge replaced the block that
#      created it).  Rebuilt, now override-aware.
#   4. Blocks 2 and 3 were pasted BEFORE the variables they read (wdays,
#      n_required_visits, _norm_stop_days), guaranteeing NameError.  Moved to
#      their correct positions.
#   5. The old buggy ids-scoped verification was still present further down,
#      shadowing the corrected one.  Removed.
#   6. The dead `cs = coords_df[...]` block (the original coords-driven ids
#      screen) was still present.  Removed — it is the defect being fixed.
#   7. `import pandas as pd` was function-local; math/np were not imported at
#      all in the merged cell.  Imports are module-level, as in the original.
#
# ROOT CAUSE BEING FIXED
#   `ids` was built exclusively from coords_clean.csv.  A hospital absent from
#   that file, or flagged bad_coord, or with null haversine_miles, never entered
#   the model — silently.  P162 (Worthing, Tooting) was such a case.  The audit
#   that should have caught it filtered trips_clean down to `ids` before
#   comparing against `ids`, so it compared a set with itself and reported
#   success.  Demand is now sourced from trips_clean_fixed.csv (ground truth);
#   coords_clean.csv is a geometry lookup only.
# ==============================================================================

import math
import numpy as np
import pandas as pd


# ── Coverage policy ───────────────────────────────────────────────────────────
# Hard-fail when a hospital required by trips_clean has no usable coordinate.
# Set False ONLY to reproduce historic (silently-dropping) runs.
STRICT_COVERAGE = True

# Manual coordinate repairs for hospitals the geocoder could not resolve.
# Every entry MUST be validated against the master schedule's Miles column via
# validate_coord_override() before production use.
#   Pulse Code -> (Latitude, Longitude, "source note")
COORD_OVERRIDE = {
    # P162 Worthing Hospital, Park Avenue, Worthing, West Sussex, BN11 2DH.
    # UNVERIFIED postcode-centroid estimate, not an authoritative geocode.
    # Master schedule gives Miles=54.0; a correct coordinate should yield a
    # haversine of roughly 42-44 mi from the Tooting depot (road factor
    # ~1.23-1.29).  validate_coord_override() checks this.
    "P162": (50.8236, -0.3714, "BN11 2DH postcode centroid — UNVERIFIED"),
}


def validate_coord_override_strict(shl_name, data, master_path=None, tol=0.35,
                                   strict=True):
    """As validate_coord_override, but a missing master_path is an error, not a
    silent skip. Every override for this SHL must reconcile with the master
    schedule's Miles column to within `tol` (relative)."""
    ov = globals().get("COORD_OVERRIDE", {})
    if not ov:
        return True
    if master_path is None:
        master_path = globals().get("MASTER_PATH")
    if master_path is None:
        raise ValueError(
            "validate_coord_override_strict: no master_path and no MASTER_PATH "
            "global. An unvalidated COORD_OVERRIDE is exactly the defect that "
            "produced P162 — refusing to skip the check."
        )
 
    m = pd.read_excel(master_path)
    m.columns = [str(c).strip() for c in m.columns]
    m["Pulse Code"] = m["Pulse Code"].astype(str).str.strip()
 
    dlat, dlon = DEPOT_COORDS[shl_name]
    rf = data.get("road_factor_shl", ROAD_FACTOR_DEFAULT)
    ok_all = True
 
    for pc, (lat, lon, note) in ov.items():
        row = m[m["Pulse Code"] == pc]
        if len(row) == 0 or str(row.iloc[0].get("Centre")).strip() != shl_name:
            continue
        miles = row.iloc[0].get("Miles")
        if pd.isna(miles) or float(miles) <= 0:
            msg = f"{pc}: no master Miles to validate override against."
            if strict:
                raise ValueError("OVERRIDE VALIDATION — " + msg)
            print("  ⚠ " + msg); ok_all = False; continue
 
        R = 6371.0088
        p = math.radians
        dla, dlo = p(lat - dlat), p(lon - dlon)
        a = (math.sin(dla / 2) ** 2 +
             math.cos(p(dlat)) * math.cos(p(lat)) * math.sin(dlo / 2) ** 2)
        hav_mi = 2 * R * math.asin(math.sqrt(a)) * 0.621371
        pred = hav_mi * rf
        rel = abs(pred - float(miles)) / float(miles)
        verdict = "OK" if rel <= tol else "FAIL"
        print(f"  override {pc}: haversine={hav_mi:.2f} mi x rf={rf:.3f} "
              f"-> {pred:.1f} mi vs master {float(miles):.1f} mi "
              f"(rel err {rel:.1%}) [{verdict}]  {note}")
        if rel > tol:
            ok_all = False
            if strict:
                raise ValueError(
                    f"OVERRIDE VALIDATION FAILED — {pc}: predicted {pred:.1f} mi "
                    f"vs master {float(miles):.1f} mi, relative error {rel:.1%} "
                    f"> tol {tol:.0%}."
                )
    return ok_all

def load_data(shl_name, month_demand=None, month_boxes=None):
    """Load and prepare every model input for one SHL from the processed files:
    hospital coordinates, per-slot time windows, dispatch times, demand, observed
    travel times, per-SHL speed calibration, the real Boxes cross-check, and (when
    available) the worst-case BSMS category for the post-hoc min-service check.

    v28: the REQUIRED hospital set comes from trips_clean_no_private.csv, not from
    coords_clean.csv.  coords_clean.csv supplies geometry for those hospitals.
    A required hospital with no usable coordinate raises under STRICT_COVERAGE
    rather than disappearing from the model.
    """
    coords_df = pd.read_csv(DATA_DIR / "coords_clean.csv")
    demand_df = pd.read_csv(DATA_DIR / "demand_params.csv")
    tw_df     = pd.read_csv(DATA_DIR / "tw_params.csv")
    trips_df  = pd.read_csv(DATA_DIR / "trips_clean_fixed_v2.csv")

    for _df in (coords_df, demand_df, tw_df, trips_df):
        if "Pulse Code" in _df.columns:
            _df["Pulse Code"] = _df["Pulse Code"].astype(str).str.strip()

    dlat, dlon = DEPOT_COORDS[shl_name]

    # ══════════════════════════════════════════════════════════════════════════
    # 1. REQUIRED DEMAND SET — trips_clean_no_private.csv is ground truth
    # ══════════════════════════════════════════════════════════════════════════
    required_ids = set(
        trips_df.loc[trips_df["SHL"] == shl_name, "Pulse Code"].dropna()
    )
    required_ids.discard("nan")

    _coord_rows = {r["Pulse Code"]: r for _, r in coords_df.iterrows()}

    def _usable_coord(pc):
        """Return (lat, lon, source_tag) or None. COORD_OVERRIDE takes priority."""
        _ov = globals().get("COORD_OVERRIDE", {}).get(pc)
        if _ov is not None:
            return float(_ov[0]), float(_ov[1]), f"override: {_ov[2]}"
        r = _coord_rows.get(pc)
        if r is None:
            return None
        if str(r.get("Centre")).strip() in EXCLUDE_CENTRES:
            return None
        if bool(r.get("bad_coord")):
            return None
        if pd.isna(r.get("Latitude")) or pd.isna(r.get("Longitude")):
            return None
        return float(r["Latitude"]), float(r["Longitude"]), "coords_clean"

    _name_lu = {r["Pulse Code"]: str(r.get("Hospital Name", r["Pulse Code"]))
                for _, r in coords_df.iterrows()}

    ids = []; hlat = {}; hlon = {}; hname = {}
    _missing_coord = []
    _override_used = []

    for pc in sorted(required_ids):
        _c = _usable_coord(pc)
        if _c is None:
            _missing_coord.append(pc)
            continue
        ids.append(pc)
        hlat[pc], hlon[pc] = _c[0], _c[1]
        hname[pc] = _name_lu.get(pc, pc)
        if _c[2].startswith("override"):
            _override_used.append(pc)
            print(f"  NOTE {pc} ({hname[pc]}): coordinate from COORD_OVERRIDE "
                  f"[{_c[2].split(': ', 1)[1]}]")

    # coord_lu retained for return-signature compatibility; override-aware.
    coord_lu = {pc: (float(r["Latitude"]), float(r["Longitude"]),
                     str(r.get("Hospital Name", pc)))
                for pc, r in _coord_rows.items() if pd.notna(r.get("Latitude"))}
    for pc in _override_used:
        coord_lu[pc] = (hlat[pc], hlon[pc], hname[pc])

    # Hospitals present in coords_clean for this SHL but not required by trips.
    # Reported rather than silently modelled, so demand-set drift stays visible.
    _coord_only = sorted(
        {p for p, r in _coord_rows.items()
         if str(r.get("Centre")).strip() == shl_name} - required_ids
    )
    if _coord_only:
        print(f"  NOTE {shl_name}: {len(_coord_only)} hospital(s) in coords_clean "
              f"but not required by trips_clean — not modelled: "
              f"{_coord_only[:12]}{' ...' if len(_coord_only) > 12 else ''}")

    if _missing_coord:
        _msg = (f"{shl_name}: {len(_missing_coord)} hospital(s) required by "
                f"trips_clean_fixed_v2.csv have no usable coordinate: "
                f"{sorted(_missing_coord)}. Add them to coords_clean.csv, clear "
                f"their bad_coord flag, or supply COORD_OVERRIDE entries. "
                f"Run diagnose_missing_hospital(pc, '{shl_name}') for detail.")
        if globals().get("STRICT_COVERAGE", True):
            raise ValueError("COVERAGE FAILURE — " + _msg)
        print(f"  ⚠ WARNING (STRICT_COVERAGE=False) — {_msg}")

    # ══════════════════════════════════════════════════════════════════════════
    # 2. DAY-FIELD RESOLUTION
    # ══════════════════════════════════════════════════════════════════════════
    _DAYS_LOOKUP = {
        "mtwhf": "all", None: "all", "": "all", "nan": "all",
        "weekday": "all", "weekdays": "all", "mtwtf": "all",
        "mon-fri": "all", "mon - fri": "all",
        "mon": "mon", "monday": "mon", "tue": "tue", "tuesday": "tue",
        "wed": "wed", "wednesday": "wed", "thu": "thu", "thursday": "thu",
        "fri": "fri", "friday": "fri",
    }
    _DAY_MAP = {"mon": 1, "tue": 2, "wed": 3, "thu": 4, "fri": 5}

    def days_from_field(raw):
        """Resolve a Days_field value to the list of weekdays it covers. Handles:
        NHSBT compact single-letter code (M/T/W/H/F, H=Thursday); three-letter
        prefixes (Mon/Tue/...); slash-compound values leaked from the master
        schedule; and MTWHF/weekday shorthands. Returns all weekdays as a safe
        default only when the value is genuinely unrecognisable.
        """
        if raw is None or (isinstance(raw, float) and math.isnan(raw)):
            return DAYS
        s = str(raw).strip(); key = s.lower()
        if key in _DAYS_LOOKUP:
            m = _DAYS_LOOKUP[key]
            return DAYS if m == "all" else [_DAY_MAP[m]]
        if "weekday" in key or "monfri" in key or "mon-fri" in key:
            return DAYS
        for prefix, d in _DAY_MAP.items():
            if key.startswith(prefix): return [d]
        _letters = {"M": 1, "T": 2, "W": 3, "H": 4, "F": 5}
        _days = sorted({_letters[ch] for ch in s.upper() if ch in _letters})
        if _days:
            if "/" in s:
                print(f"  WARNING: compound Days_field {s!r} -- resolved days={_days}; "
                      f"verify times were split upstream in the pipeline.")
            return _days
        print(f"  WARNING: unrecognised Days_field value {s!r} -- treating as MTWHF.")
        return DAYS

    def _norm_stop_days(raw):
        """Convert a stop_days value ('MTWHF', 'M', 'TWHF', 'MW') to a sorted
        tuple of weekday ints. Falls back to all weekdays when unrecognised."""
        if raw is None or (isinstance(raw, float) and math.isnan(raw)):
            return tuple(DAYS)
        resolved = days_from_field(raw)
        return tuple(sorted(resolved)) if resolved else tuple(DAYS)

    # ══════════════════════════════════════════════════════════════════════════
    # 3. TIME WINDOWS FROM tw_params.csv
    # ══════════════════════════════════════════════════════════════════════════
    windows = {}; slot_windows = {}; wdays = {}; lead = {}
    co_times = {}; co_times_large = {}; dp_times = {}
    co_times_slot = {}; co_times_large_slot = {}
    dp_times_co = {}

    for pc in ids:
        tw_j = tw_df[tw_df["Pulse Code"] == pc]
        windows[pc] = {d: [] for d in DAYS}
        slot_windows[pc] = {d: [] for d in DAYS}
        wdays[pc] = []
        for _, row in tw_j.iterrows():
            slot = str(row["Slot"]).strip()
            if slot in SAT_SLOTS or slot in SUN_SLOTS: continue
            if slot not in WEEKDAY_SLOTS: continue
            slot_days = days_from_field(row.get("Days_field", None))
            oh = parse_time(str(row.get("win_open", "")))
            ch = parse_time(str(row.get("win_close", "")))
            if ch is None: ch = parse_time(str(row.get("AR", "")))
            if ch is None: ch = 16.0
            if oh is None: oh = 0.0
            is_large = bool(row.get("large_order", False))
            co_std   = parse_time(str(row.get("CO_standard", "") if is_large else row.get("CO", "")))
            co_large = parse_time(str(row.get("CO", ""))) if is_large else None
            dp_h_raw = parse_time(str(row.get("DP", "")))
            row_lead_h = row.get("lead_time_h")
            row_lead_h = float(row_lead_h) if pd.notna(row_lead_h) else None
            co_for_dp = co_large if (is_large and co_large is not None) else co_std
            if co_for_dp is not None:
                lead_for_row = row_lead_h if row_lead_h is not None else 2.0
                dp_h_co = co_for_dp + lead_for_row
            else:
                dp_h_co = None
            for day in slot_days:
                windows[pc][day].append((oh, ch))
                slot_windows[pc][day].append((slot, oh, ch))
                if day not in wdays[pc]: wdays[pc].append(day)
                if co_std is not None:
                    co_times[(pc, day)] = co_std
                    co_times_slot[(pc, day, slot)] = co_std
                if co_large is not None:
                    co_times_large[(pc, day)] = co_large
                    co_times_large_slot[(pc, day, slot)] = co_large
                # Real DP column is authoritative; CO+lead is used only when DP is
                # empty (it would be wrong for evening cut-offs delivered next morning).
                dp_h_effective = dp_h_raw if dp_h_raw is not None else dp_h_co
                if dp_h_effective is not None:
                    dp_times[(pc, day)] = min(dp_times.get((pc, day), dp_h_effective),
                                              dp_h_effective)
                if co_for_dp is not None:
                    dp_times_co[(pc, day, slot)] = dp_h_co
        wdays[pc] = sorted(wdays[pc])
        lt = tw_j["lead_time_h"].dropna().tolist()
        lead[pc] = float(np.median(lt)) if lt else 2.0

    _no_tw = [pc for pc in ids if not wdays.get(pc)]
    if _no_tw:
        print(f"  NOTE {shl_name}: {len(_no_tw)} hospital(s) have no weekday row in "
              f"tw_params.csv: {sorted(_no_tw)} — relying on trips augmentation below.")

    # Optional coverage escape hatch (default empty; retained for edge cases the
    # pipeline cleaner cannot resolve).
    _patch_excl = set(globals().get("PATCH_EXCLUDE", {}).get(shl_name, []))
    if _patch_excl:
        _before = len(ids); ids = [pc for pc in ids if pc not in _patch_excl]
        print(f"  WARNING PATCH_EXCLUDE[{shl_name}]: dropped {_before-len(ids)} hospital(s) "
              f"{sorted(_patch_excl)} for well-posed coverage.")

    # ══════════════════════════════════════════════════════════════════════════
    # 4. PER-SLOT DISPATCH TABLE
    # ══════════════════════════════════════════════════════════════════════════
    # When a hospital has SEVERAL rows for the same slot (a compound-day slot the
    # pipeline has split into one row per day-group), match the row whose
    # Days_field actually covers THIS day, so a Tue-Fri leg gets its real dispatch
    # time rather than the (typically later) Monday one.
    dp_times_slot = {}
    for pc in ids:
        for day in DAYS:
            for s, (slot_name, oh, ch) in enumerate(slot_windows[pc].get(day, [])):
                rows_slot = tw_df[(tw_df["Pulse Code"] == pc) & (tw_df["Slot"] == slot_name)]
                dp_h_slot = None
                for _, row in rows_slot.iterrows():
                    if day not in days_from_field(row.get("Days_field", None)):
                        continue
                    dp_h_slot = parse_time(str(row.get("DP", "")))
                    if dp_h_slot is not None: break
                if dp_h_slot is None:
                    for _, row in rows_slot.iterrows():
                        dp_h_slot = parse_time(str(row.get("DP", "")))
                        if dp_h_slot is not None: break
                if dp_h_slot is None: dp_h_slot = dp_times_co.get((pc, day, slot_name))
                if dp_h_slot is None: dp_h_slot = dp_times.get((pc, day))
                if dp_h_slot is not None: dp_times_slot[(pc, day, s)] = dp_h_slot

    # ══════════════════════════════════════════════════════════════════════════
    # 5. AUGMENT slot_windows FROM trips_clean_fixed.csv
    # ══════════════════════════════════════════════════════════════════════════
    # tw_params.csv may be missing delivery-slot entries. trips_clean_fixed.csv is
    # ground truth: each unique (Pulse Code, matched_slot, stop_days) within an SHL
    # is one required delivery visit per covered day.
    _trips_shl = trips_df[trips_df["SHL"] == shl_name].copy()
    _augmented = 0
    _aug_details = []

    for pc in ids:
        _pc_trips = _trips_shl[_trips_shl["Pulse Code"] == pc]
        if len(_pc_trips) == 0:
            continue

        # Key on (matched_slot, stop_days) so compound-day rounds do not collapse.
        _trip_slot_map = {}
        _nan_trips = []
        for _, _tr in _pc_trips.iterrows():
            _sl = _tr.get("matched_slot")
            _days_tup = _norm_stop_days(_tr.get("stop_days"))
            if pd.isna(_sl):
                _nan_trips.append({
                    "trip": _tr.get("Trip Name"),
                    "depart_min": _tr.get("depart_min"),
                    "arrival_min": _tr.get("arrival_min"),
                    "cutoff_min": _tr.get("cutoff_min"),
                    "days_tup": _days_tup,
                })
                continue
            _sl = str(_sl).strip()
            if _sl not in WEEKDAY_SLOTS:
                continue
            _key = (_sl, _days_tup)
            if _key not in _trip_slot_map:
                _trip_slot_map[_key] = {
                    "depart_min": _tr.get("depart_min"),
                    "arrival_min": _tr.get("arrival_min"),
                    "cutoff_min": _tr.get("cutoff_min"),
                }

        # NaN-slot rows take the next T-slot not claimed by any day-group.
        _used_slots = {sn for (sn, _dt) in _trip_slot_map.keys()}
        for _nt in _nan_trips:
            for _candidate in WEEKDAY_SLOTS:
                if _candidate not in _used_slots:
                    _trip_slot_map[(_candidate, _nt["days_tup"])] = {
                        "depart_min": _nt["depart_min"],
                        "arrival_min": _nt["arrival_min"],
                        "cutoff_min": _nt["cutoff_min"],
                    }
                    _used_slots.add(_candidate)
                    _aug_details.append(
                        f"{pc} NaN-slot trip {_nt['trip']!r} -> assigned {_candidate}")
                    break

        # Track (slot, day) so a compound-day slot present only for Mon does not
        # block the same slot being added for Tue-Fri.
        _existing_by_day = {d: {sn for (sn, _oh, _ch) in slot_windows[pc].get(d, [])}
                            for d in DAYS}

        for (_ms, _days_tup), _info in sorted(_trip_slot_map.items()):
            _dep_h = _info["depart_min"] / 60.0 if pd.notna(_info["depart_min"]) else None
            _arr_h = _info["arrival_min"] / 60.0 if pd.notna(_info["arrival_min"]) else None
            _co_h  = _info["cutoff_min"] / 60.0 if pd.notna(_info["cutoff_min"]) else None

            _oh_synth = _dep_h if _dep_h is not None else 0.0
            _ch_synth = _arr_h if _arr_h is not None else (
                (_dep_h + 4.0) if _dep_h is not None else 16.0)

            added_any_day = False
            for _d in _days_tup:
                if _ms in _existing_by_day.get(_d, set()):
                    continue                      # tw_params already covers it
                windows[pc][_d].append((_oh_synth, _ch_synth))
                slot_windows[pc][_d].append((_ms, _oh_synth, _ch_synth))
                _existing_by_day.setdefault(_d, set()).add(_ms)
                if _d not in wdays[pc]:
                    wdays[pc].append(_d)
                if _dep_h is not None:
                    _new_s = len(slot_windows[pc][_d]) - 1
                    dp_times_slot[(pc, _d, _new_s)] = _dep_h
                    if (pc, _d) not in dp_times or _dep_h < dp_times[(pc, _d)]:
                        dp_times[(pc, _d)] = _dep_h
                if _co_h is not None:
                    co_times_slot[(pc, _d, _ms)] = _co_h
                added_any_day = True

            if added_any_day:
                _augmented += 1
                _aug_details.append(
                    f"{pc} +{_ms} on days={list(_days_tup)} "
                    f"(DP={_dep_h if _dep_h is None else round(_dep_h, 2)}h "
                    f"AR={_ch_synth:.2f}h)")

        wdays[pc] = sorted(wdays[pc])

    if _augmented > 0:
        print(f"\n  ── v28 trips_clean augmentation: {_augmented} missing slot(s) added ──")
        for _ad in _aug_details:
            print(f"      {_ad}")
        print("  (tw_params.csv was missing these; trips_clean_fixed_v2.csv is ground truth)")
    else:
        print("\n  ── v28 trips_clean augmentation: slot_windows already complete ──")

    # ══════════════════════════════════════════════════════════════════════════
    # 6. WEEKDAY-WINDOW FILTER — logged, not silent
    # ══════════════════════════════════════════════════════════════════════════
    _no_window = [pc for pc in ids if not wdays.get(pc)]
    if _no_window:
        _msg = (f"{shl_name}: {len(_no_window)} required hospital(s) have no weekday "
                f"delivery window after trips augmentation: {sorted(_no_window)}. "
                f"Check tw_params.csv Slot/Days_field and trips_clean_fixed_v2.csv "
                f"matched_slot for these codes.")
        if globals().get("STRICT_COVERAGE", True):
            raise ValueError("COVERAGE FAILURE — " + _msg)
        print(f"  ⚠ WARNING (STRICT_COVERAGE=False) — {_msg}")
    ids = [pc for pc in ids if wdays.get(pc)]

    n_required_visits = {(pc, t): len(slot_windows[pc].get(t, [])) for pc in ids for t in DAYS}
    n_max = {(pc, d): max(1, len(windows[pc].get(d, []))) for pc in ids for d in DAYS}

    dual = [(pc, t, cnt) for (pc, t), cnt in n_required_visits.items() if cnt > 1]
    if dual:
        dual_pcs = sorted(set(pc for pc, t, _ in dual))
        print(f"\n  Dual-slot hospitals: {dual_pcs}")
        for pc in dual_pcs:
            example_t = min(t for (p, t, _) in dual if p == pc)
            for s, (sn, oh, ch) in enumerate(slot_windows[pc].get(example_t, [])):
                dp_s = dp_times_slot.get((pc, example_t, s))
                co_s = dp_times_co.get((pc, example_t, sn))
                src_tag = ("CO+lead" if (co_s is not None and dp_s is not None
                                         and abs(dp_s - co_s) < 1e-6) else "DP col")
                print(f"    {pc} s={s} ({sn}): DP={dp_s} [{src_tag}] AR={ch:.2f}h")
    else:
        print("\n  All hospitals have exactly 1 delivery slot per day.")

    # ══════════════════════════════════════════════════════════════════════════
    # 7. COVERAGE VERIFICATION — scoped to required_ids, NOT to ids
    # ══════════════════════════════════════════════════════════════════════════
    # PRE-v28 BUG: the check filtered trips_clean down to `ids` before comparing
    # against `ids`, so a hospital missing from `ids` was excluded from its own
    # coverage check and the audit reported success.
    _absent = sorted(required_ids - set(ids))
    if _absent:
        print(f"\n  ⚠ v28 COVERAGE FAILURE: {len(_absent)} hospital(s) required by "
              f"trips_clean_fixed_v2.csv are NOT modelled: {_absent}")

    _trips_visit_count = {}
    for _pc_v in sorted(required_ids):
        _pc_v_sub = _trips_shl[_trips_shl["Pulse Code"] == _pc_v]
        _by_day_slots = {d: set() for d in DAYS}
        for _, _rr in _pc_v_sub.iterrows():
            _sl = _rr.get("matched_slot")
            _sl_str = str(_sl).strip() if pd.notna(_sl) else "NAN"
            for _d in _norm_stop_days(_rr.get("stop_days")):
                _by_day_slots[_d].add(_sl_str)
        _n_trips_v = max((len(s) for s in _by_day_slots.values()), default=0)
        _n_slots_v = (max(n_required_visits.get((_pc_v, t), 0) for t in DAYS)
                      if _pc_v in ids else 0)
        if _n_trips_v != _n_slots_v:
            _trips_visit_count[_pc_v] = (_n_trips_v, _n_slots_v)

    if _trips_visit_count:
        print("\n  ⚠ v28 RESIDUAL MISMATCH (trips_clean rounds vs model slots):")
        for _pc_r, (_nt_r, _ns_r) in sorted(_trips_visit_count.items()):
            _tag = "  <-- NOT MODELLED" if _ns_r == 0 else ""
            print(f"      {_pc_r}: {_nt_r} rounds in CSV, {_ns_r} model slots{_tag}")
        if globals().get("STRICT_COVERAGE", True) and _absent:
            raise ValueError(
                f"COVERAGE FAILURE — {shl_name}: hospitals required by "
                f"trips_clean_fixed_v2.csv are absent from the model: {_absent}")
    else:
        print(f"\n  ✓ v28 verification: all {len(required_ids)} required hospital "
              f"visit counts match trips_clean_fixed_v2.csv")

    co_derived_count = len(dp_times_co); total_slots = sum(n_required_visits.values())
    print(f"  CO-derived DP: {co_derived_count}/{total_slots} slot-days use CO+lead_time "
          f"(others fall back to raw DP column or aggregate minimum)")

    # ══════════════════════════════════════════════════════════════════════════
    # 8. DEMAND
    # ══════════════════════════════════════════════════════════════════════════
    demand = {}; DOW = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}
    if month_demand is not None:
        for (pc, prod, day_int), val in month_demand.items():
            if pc in ids:
                demand[(pc, prod, day_int)] = val
        print(f"  Demand: {len(demand)} entries from monthly utilisation data")
    else:
        for pc in ids:
            for _, row in demand_df[demand_df["Pulse Code"] == pc].iterrows():
                dow = int(row["dow"])
                if dow in DOW:
                    demand[(pc, str(row["product"]), DOW[dow])] = float(row["median_demand"])

    _no_demand = sorted({pc for pc in ids} - {k[0] for k in demand})
    if _no_demand:
        print(f"  ⚠ NOTE {shl_name}: {len(_no_demand)} modelled hospital(s) have NO demand "
              f"record: {_no_demand} — they will be visited but carry zero load.")

    visit_days = {pc: wdays[pc] for pc in ids}

    # ══════════════════════════════════════════════════════════════════════════
    # 9. DEPARTURE HORIZON
    # ══════════════════════════════════════════════════════════════════════════
    # T_DEP/T_MAX come from tw_params DP (the authoritative Master Schedule
    # source), NOT trips_clean depart_min, whose round-sheet times can differ.
    tw_dp_vals = []
    for pc in ids:
        tw_pc = tw_df[(tw_df["Pulse Code"] == pc) & tw_df["Slot"].isin(WEEKDAY_SLOTS)]
        for _, row in tw_pc.iterrows():
            dp_val = parse_time(str(row.get("DP", "")))
            if dp_val is not None:
                tw_dp_vals.append(dp_val)
    ts = trips_df[trips_df["SHL"] == shl_name]["depart_min"].dropna()
    if tw_dp_vals:
        T_DEP = min(tw_dp_vals)
        T_MAX_depart = max(tw_dp_vals)
    elif len(ts) > 0:
        T_DEP = float(ts.min()) / 60.0
        T_MAX_depart = float(ts.max()) / 60.0
    else:
        T_DEP = 6.0
        T_MAX_depart = T_DEP
    T_MAX = T_MAX_depart + U

    # ══════════════════════════════════════════════════════════════════════════
    # 10. DISTANCES AND ROAD FACTOR
    # ══════════════════════════════════════════════════════════════════════════
    sched_miles = {}; road_factor_shl = ROAD_FACTOR_DEFAULT
    shl_coords = coords_df[coords_df["Centre"] == shl_name].copy()
    if "Miles" in shl_coords.columns:
        for _, row in shl_coords.iterrows():
            pc = str(row["Pulse Code"]).strip(); mi = row.get("Miles")
            if pd.notna(mi) and float(mi) > 0: sched_miles[pc] = float(mi)
        factors = []
        for _, row in shl_coords.iterrows():
            hav = row.get("haversine_miles"); obs = row.get("Miles")
            if pd.notna(hav) and pd.notna(obs) and float(hav) > 0.5 and float(obs) > 0:
                factors.append(float(obs) / float(hav))
        if factors:
            road_factor_shl = float(np.median(factors))
            print(f"  Master Schedule Miles: {len(sched_miles)}/{len(shl_coords)} hospitals with a "
                  f"real depot-hospital distance | road_factor_shl (median real Miles/Haversine "
                  f"over valid pairs, n={len(factors)}) = {road_factor_shl:.3f}")
        else:
            print(f"  WARNING: no valid Miles/Haversine pairs for {shl_name} -- falling back to "
                  f"ROAD_FACTOR_DEFAULT={ROAD_FACTOR_DEFAULT}.")
    else:
        print("  WARNING: 'Miles' column not present in coords_clean.csv; the model will fall "
              f"back to an unadjusted Haversine distance (ROAD_FACTOR_DEFAULT={ROAD_FACTOR_DEFAULT}).")

    # Hospitals modelled via COORD_OVERRIDE have no coords_clean row and so no
    # Miles entry; their depot distance falls back to haversine * road_factor_shl.
    _no_miles = sorted(set(ids) - set(sched_miles))
    if _no_miles:
        print(f"  NOTE {shl_name}: {len(_no_miles)} modelled hospital(s) have no schedule "
              f"Miles: {_no_miles} — depot distance uses haversine x {road_factor_shl:.3f}.")

    # ══════════════════════════════════════════════════════════════════════════
    # 11. MASTER-SCHEDULE INTERVAL TAU
    # ══════════════════════════════════════════════════════════════════════════
    # AR - DP is the schedule's own transit claim for that slot (an upper bound
    # when the hospital is mid-round, exact when it is first stop). Min over the
    # hospital's slots gives the tightest data-provided depot transit, so every
    # scheduled slot is reachable by construction.
    ms_tau = {}
    _tw_ms = tw_df[(tw_df["Centre"] == shl_name) & tw_df["Slot"].isin(SLOT_TO_DAY)]
    for pc_ms, grp_ms in _tw_ms.groupby(_tw_ms["Pulse Code"].astype(str).str.strip()):
        gaps = []
        for _, row in grp_ms.iterrows():
            g = time_diff_min(str(row.get("DP", "")), str(row.get("AR", "")))
            if g is not None:
                if g < 0: g += 1440          # overnight wrap (DP 23:00 -> AR 01:00)
                if g > 0: gaps.append(g)
        if gaps:
            ms_tau[pc_ms] = min(gaps) / 60.0
    print(f"  Master-Schedule interval tau: {len(ms_tau)}/{len(ids)} modelled hospitals with a "
          f"schedule-stated depot transit (min over slots of AR-DP); Emer/AdHoc not used.")
    emer_tau = ms_tau  # downstream key name kept for compatibility

    tw_shl = tw_df[(tw_df["Centre"] == shl_name) & tw_df["Slot"].isin(SLOT_TO_DAY)].copy()
    tw_shl["gap"] = tw_shl.apply(
        lambda r: time_diff_min(str(r.get("DP", "")), str(r.get("AR", ""))), axis=1)
    gv = tw_shl["gap"].dropna()
    mean_gap_h = float(gv.median()) / 60.0 if len(gv) > 0 else None

    shl_trips = trips_df[trips_df["SHL"] == shl_name]
    C_baseline = shl_trips["Trip Name"].nunique() if len(shl_trips) > 0 else max(len(ids), 2)

    # ══════════════════════════════════════════════════════════════════════════
    # 12. SPEED CALIBRATION
    # ══════════════════════════════════════════════════════════════════════════
    CALIB_MPH_MIN = 20.0; CALIB_MPH_MAX = 70.0
    DWELL_ELAPSED_THRESHOLD_H = 2.0
    sched_tau_candidates = {}; sched_obs = []
    pc_to_node = {pc: i + 1 for i, pc in enumerate(ids)}
    shl_t = shl_trips[shl_trips["Pulse Code"].isin(ids)].copy()
    shl_t = uniquify_trip_names(shl_t, shl_name)
    shl_t = shl_t.sort_values(["Trip Name", "Delivery Sequence"])
    n_excluded_dwell = 0
    for _, grp in shl_t.groupby("Trip Name"):
        rows = grp.reset_index(drop=True)
        if len(rows) == 0: continue
        dep_col = rows["depart_min"].dropna()
        if len(dep_col) == 0: continue
        round_dep = float(dep_col.iloc[0])
        row = rows.iloc[0]; pc = str(row["Pulse Code"]); ni = pc_to_node.get(pc)
        if ni is None: continue
        arr = row.get("arrival_min")
        if pd.isna(arr): continue
        t_h = (float(arr) - round_dep) / 60.0
        if t_h <= 0: continue
        d_mi_calib = sched_miles.get(pc)
        if d_mi_calib is None or d_mi_calib <= 0:
            d_mi_calib = road_mi(dlat, dlon, hlat[pc], hlon[pc], road_factor_shl)
        implied_mph = d_mi_calib / t_h if t_h > 0 else 0
        is_slow = implied_mph < CALIB_MPH_MIN
        is_long_elapsed = t_h > DWELL_ELAPSED_THRESHOLD_H
        is_dwell_suspect = is_slow and is_long_elapsed
        is_too_fast = implied_mph > CALIB_MPH_MAX
        is_short_overhead = globals().get("CLAMP_CALIB_SPEED", False) and (
            (d_mi_calib < globals().get("CALIB_MIN_LEG_MI", 0.0)) or
            (implied_mph < globals().get("CALIB_MIN_IMPLIED_MPH", 0.0)))
        if is_dwell_suspect or is_short_overhead:
            n_excluded_dwell += 1
            continue
        if is_too_fast:
            continue
        sched_obs.append((d_mi_calib, t_h))
        sched_tau_candidates.setdefault(ni, []).append(t_h)
    sched_tau = {ni: min(ts_) for ni, ts_ in sched_tau_candidates.items()}
    speeds = [d / t for d, t in sched_obs if t > 0.01]
    calib_mph_raw = float(np.median(speeds)) if speeds else SPEED_MPH_DEFAULT
    if globals().get("CLAMP_CALIB_SPEED", False):
        _flo = globals().get("CALIB_MPH_FLOOR", CALIB_MPH_MIN)
        _cei = globals().get("CALIB_MPH_CEIL", CALIB_MPH_MAX)
        calib_mph = float(min(max(calib_mph_raw, _flo), _cei))
    else:
        calib_mph = calib_mph_raw
    if abs(calib_mph - calib_mph_raw) > 1e-9:
        print(f"  WARNING speed clamp: raw median {calib_mph_raw:.1f} mph -> {calib_mph:.1f} mph "
              f"(plausible range [{globals().get('CALIB_MPH_FLOOR', CALIB_MPH_MIN):.0f},"
              f"{globals().get('CALIB_MPH_CEIL', CALIB_MPH_MAX):.0f}])")
    print(f"  Speed calibration: {len(sched_obs)} valid legs kept, "
          f"{n_excluded_dwell} excluded (dwell-suspect or short/overhead), "
          f"calib_mph={calib_mph:.1f}  sched_tau: {len(sched_tau)} depot->hospital nodes (min t_h)")

    sched_tau_h2h = {}
    for _, grp in shl_t.groupby("Trip Name"):
        rows_h = grp.sort_values("Delivery Sequence").reset_index(drop=True)
        for idx_r in range(1, len(rows_h)):
            prev = rows_h.iloc[idx_r - 1]; curr = rows_h.iloc[idx_r]
            ni = pc_to_node.get(str(prev["Pulse Code"]))
            nj = pc_to_node.get(str(curr["Pulse Code"]))
            if ni is None or nj is None: continue
            arr_i = prev.get("arrival_min"); arr_j = curr.get("arrival_min")
            if pd.isna(arr_i) or pd.isna(arr_j): continue
            interval_h = (float(arr_j) - float(arr_i)) / 60.0
            pure_travel_h = max(0.0, interval_h - S)
            if pure_travel_h > 0:
                sched_tau_h2h[(ni, nj)] = min(sched_tau_h2h.get((ni, nj), 999.0), pure_travel_h)
    print(f"  h2h schedule tau: {len(sched_tau_h2h)} inter-hospital arcs (interval - S={S}h dwell)")
    print(f"\n  {shl_name}: {len(ids)} hospitals  T_DEP={T_DEP:.2f}h  T_MAX={T_MAX:.2f}h"
          + (f"  mean_gap={mean_gap_h*60:.0f}min" if mean_gap_h else ""))
    print(f"  C_baseline={C_baseline} ({len(sched_obs)} sched legs; calib speed {calib_mph:.1f} mph)")

    # ══════════════════════════════════════════════════════════════════════════
    # 13. REAL BOX COUNTS AND BSMS TIERS
    # ══════════════════════════════════════════════════════════════════════════
    real_boxes_by_hospital = {}
    try:
        boxes_df = pd.read_csv(DATA_DIR / "boxes_by_hospital.csv")
        boxes_df["Pulse Code"] = boxes_df["Pulse Code"].astype(str).str.strip()
        for _, row in boxes_df.iterrows():
            pc = row["Pulse Code"]
            if pc in ids and pd.notna(row.get("mean_boxes")):
                real_boxes_by_hospital[pc] = float(row["mean_boxes"])
        print(f"  Real Boxes cross-check: {len(real_boxes_by_hospital)}/{len(ids)} hospitals with "
              f"an observed mean-boxes-per-delivery figure.")
    except FileNotFoundError:
        print("  NOTE: boxes_by_hospital.csv not found -- real Boxes cross-check unavailable.")

    # Each hospital's tier is the worst across the four survey product dimensions;
    # the weekly-visit floor implied by that tier is looked up in
    # BSMS_MIN_VISITS_BY_TIER.
    bsms_tier = {}; bsms_min_visits = {}
    try:
        bsms_df = pd.read_csv(DATA_DIR / "bsms_worst_tier.csv")
        bsms_df["Pulse Code"] = bsms_df["Pulse Code"].astype(str).str.strip()
        for _, row in bsms_df.iterrows():
            pc = row["Pulse Code"]; tier = row.get("bsms_worst_tier")
            if pc in ids and pd.notna(tier):
                bsms_tier[pc] = str(tier)
                floor = BSMS_MIN_VISITS_BY_TIER.get(str(tier))
                if floor is not None: bsms_min_visits[pc] = int(floor)
        n_covered = len(bsms_tier)
        print(f"  BSMS worst-case tier: {n_covered}/{len(ids)} hospitals categorised "
              f"({len(ids) - n_covered} without a survey record -- no min-service floor imposed).")
    except FileNotFoundError:
        print("  NOTE: bsms_worst_tier.csv not found -- BSMS minimum-service-level check unavailable.")

    return dict(
        ids=ids, hlat=hlat, hlon=hlon, hname=hname, dlat=dlat, dlon=dlon,
        windows=windows, slot_windows=slot_windows, n_required_visits=n_required_visits,
        wdays=wdays, lead=lead, demand=demand, n_max=n_max, visit_days=visit_days,
        T_DEP=T_DEP, T_MAX=T_MAX, mean_gap_h=mean_gap_h, C_baseline=C_baseline,
        co_times=co_times, co_times_large=co_times_large,
        co_times_slot=co_times_slot, co_times_large_slot=co_times_large_slot,
        dp_times=dp_times, dp_times_slot=dp_times_slot, dp_times_co=dp_times_co,
        sched_tau=sched_tau, sched_tau_h2h=sched_tau_h2h, emer_tau=emer_tau,
        calib_mph=calib_mph, sched_miles=sched_miles, road_factor_shl=road_factor_shl,
        trips_df=trips_df, coord_lu=coord_lu,
        real_boxes_by_hospital=real_boxes_by_hospital,
        bsms_tier=bsms_tier, bsms_min_visits=bsms_min_visits,
        required_ids=sorted(required_ids), missing_coord=sorted(_missing_coord),
    )


# ==============================================================================
# DIAGNOSTIC — call on demand, e.g. diagnose_missing_hospital("P162", "Tooting")
# ==============================================================================

def diagnose_missing_hospital(pc, shl_name):
    """Explain why `pc` is absent from the model for `shl_name`, screen by screen."""
    pc = str(pc).strip()
    coords_df = pd.read_csv(DATA_DIR / "coords_clean.csv")
    tw_df     = pd.read_csv(DATA_DIR / "tw_params.csv")
    trips_df  = pd.read_csv(DATA_DIR / "trips_clean_fixed_v2.csv")
    demand_df = pd.read_csv(DATA_DIR / "demand_params.csv")
    for _df in (coords_df, tw_df, trips_df, demand_df):
        if "Pulse Code" in _df.columns:
            _df["Pulse Code"] = _df["Pulse Code"].astype(str).str.strip()

    print("=" * 72)
    print(f"DIAGNOSTIC — {pc} @ {shl_name}")
    print("=" * 72)

    row = coords_df[coords_df["Pulse Code"] == pc]
    print(f"\n[0] present in coords_clean.csv        : {len(row) > 0}")
    if len(row) == 0:
        print("    >>> absent from coords_clean.csv entirely.")
        print("    >>> Pre-v28 this alone removed it from the model, silently.")
    else:
        r = row.iloc[0]
        for f in ("Centre", "Latitude", "Longitude", "bad_coord", "haversine_miles", "Miles"):
            print(f"    {f:16} = {r.get(f)!r}")
        s1 = str(r.get("Centre")).strip() == shl_name
        s2 = str(r.get("Centre")).strip() not in EXCLUDE_CENTRES
        s3 = not bool(r.get("bad_coord"))
        s4 = pd.notna(r.get("haversine_miles"))
        print(f"\n[1] Centre == '{shl_name}'{' ' * max(0, 22 - len(shl_name))}: {s1}")
        print(f"[2] Centre not in EXCLUDE_CENTRES     : {s2}")
        print(f"[3] not bad_coord                     : {s3}")
        print(f"[4] haversine_miles notna             : {s4}")
        if not all([s1, s2, s3, s4]):
            failed = [n for n, ok in zip(
                ["Centre mismatch", "excluded centre", "bad_coord=True",
                 "haversine_miles is null"], [s1, s2, s3, s4]) if not ok]
            print(f"\n    >>> fails: {failed}")

    print(f"\n[5] COORD_OVERRIDE entry               : "
          f"{globals().get('COORD_OVERRIDE', {}).get(pc)}")

    tw_rows = tw_df[tw_df["Pulse Code"] == pc]
    wd_rows = tw_rows[tw_rows["Slot"].astype(str).str.strip().isin(WEEKDAY_SLOTS)]
    print(f"[6] rows in tw_params.csv              : {len(tw_rows)} "
          f"({len(wd_rows)} weekday-slot)")

    tr_rows = trips_df[(trips_df["Pulse Code"] == pc) & (trips_df["SHL"] == shl_name)]
    print(f"[7] rows in trips_clean_fixed_v2.csv      : {len(tr_rows)}")
    if len(tr_rows):
        print(tr_rows[["Trip Name", "matched_slot", "stop_days",
                       "depart_min", "arrival_min", "cutoff_min"]].to_string(index=False))
        print("    -> the augmentation block synthesises a weekday window from these,")
        print("       so the weekday filter would NOT drop it. Any drop is upstream.")

    print(f"[8] rows in demand_params.csv          : "
          f"{len(demand_df[demand_df['Pulse Code'] == pc])}")


def audit_coverage_all_shls():
    """Every hospital required by trips_clean but lacking a usable coordinate,
    across all SHLs. Run before a full re-solve."""
    coords_df = pd.read_csv(DATA_DIR / "coords_clean.csv")
    trips_df  = pd.read_csv(DATA_DIR / "trips_clean_fixed_v2.csv")
    for _df in (coords_df, trips_df):
        _df["Pulse Code"] = _df["Pulse Code"].astype(str).str.strip()
    ok = coords_df[
        (~coords_df["Centre"].isin(EXCLUDE_CENTRES)) &
        (~coords_df["bad_coord"].astype(bool)) &
        coords_df["haversine_miles"].notna()
    ]
    ov = set(globals().get("COORD_OVERRIDE", {}))
    print("=" * 72)
    print("COVERAGE AUDIT — required by trips_clean, no usable coordinate")
    print("=" * 72)
    any_missing = False
    for shl in sorted(trips_df["SHL"].dropna().unique()):
        if shl not in DEPOT_COORDS:
            continue
        need = set(trips_df.loc[trips_df["SHL"] == shl, "Pulse Code"])
        have = set(ok.loc[ok["Centre"] == shl, "Pulse Code"]) | ov
        miss = sorted(need - have)
        if miss:
            any_missing = True
            print(f"  {shl:12} missing {len(miss)}: {miss}")
    if not any_missing:
        print("  (none — every required hospital has a usable coordinate)")


def validate_coord_override(shl_name, data, master_path=None, tol=0.35):
    """Cross-check every COORD_OVERRIDE coordinate for this SHL against the
    master schedule's Miles column. An unverified override is a silent
    data-quality risk of exactly the kind that produced the P162 defect."""
    ov = globals().get("COORD_OVERRIDE", {})
    if not ov:
        return
    if master_path is None:
        print("  (validator: no master_path given — skipping Miles cross-check)")
        return

    m = pd.read_excel(master_path, sheet_name="schedule", header=1)
    m.columns = [str(c).strip() for c in m.columns]
    m["Pulse Code"] = m["Pulse Code"].astype(str).str.strip()

    dlat, dlon = DEPOT_COORDS[shl_name]
    rf = data.get("road_factor_shl", ROAD_FACTOR_DEFAULT)

    for pc, (lat, lon, note) in ov.items():
        row = m[m["Pulse Code"] == pc]
        if len(row) == 0 or str(row.iloc[0].get("Centre")).strip() != shl_name:
            continue
        miles = row.iloc[0].get("Miles")
        if pd.isna(miles) or float(miles) <= 0:
            print(f"  ⚠ {pc}: no master Miles to validate override against.")
            continue
        R = 6371.0088
        p = math.radians
        dla, dlo = p(lat - dlat), p(lon - dlon)
        a = (math.sin(dla / 2) ** 2 +
             math.cos(p(dlat)) * math.cos(p(lat)) * math.sin(dlo / 2) ** 2)
        hav_mi = 2 * R * math.asin(math.sqrt(a)) * KM_TO_MI
        implied = float(miles) / hav_mi if hav_mi > 0 else float("inf")
        ok = abs(implied - rf) <= tol * rf
        print(f"  {'OK' if ok else '⚠ SUSPECT'} {pc}: haversine={hav_mi:.1f} mi, "
              f"master Miles={float(miles):.1f}, implied road factor={implied:.2f} "
              f"(SHL median {rf:.2f}) [{note}]")
        if not ok and globals().get("STRICT_COVERAGE", True):
            raise ValueError(
                f"COORD_OVERRIDE for {pc} is inconsistent with the master schedule "
                f"Miles column (implied road factor {implied:.2f} vs SHL median "
                f"{rf:.2f}). Verify the coordinate before proceeding.")


print("load_data defined (v28: trips-sourced demand set, strict coverage, "
      "override-aware coords, corrected verification scoping).")


import pandas as pd
import math
 
 
# ── GUARD 1 ──────────────────────────────────────────────────────────────────
def assert_load_coverage(data, shl_name=None, verbose=True):
    """Every hospital required by the trips file must be in `ids`, must have a
    weekday window, and must have at least one required visit on some day."""
    shl_name = shl_name or data.get("shl_name", "?")
    required = set(data["required_ids"])
    ids = set(data["ids"])
 
    missing = sorted(required - ids)
    if missing:
        raise AssertionError(
            f"GUARD 1 FAILED — {shl_name}: {len(missing)} required hospital(s) "
            f"absent from the model after load_data: {missing}. "
            f"Run diagnose_missing_hospital('{missing[0]}', '{shl_name}')."
        )
 
    nrv = data["n_required_visits"]
    novisit = sorted(pc for pc in ids
                     if sum(nrv.get((pc, t), 0) for t in DAYS) == 0)
    if novisit:
        raise AssertionError(
            f"GUARD 1 FAILED — {shl_name}: {len(novisit)} hospital(s) are in "
            f"`ids` but have zero required visits on every day: {novisit}. "
            f"These occupy a logical node and are never visited. Check "
            f"slot_windows / matched_slot in the trips file."
        )
 
    if verbose:
        total_slots = sum(nrv.values())
        print(f"  ✓ GUARD 1 {shl_name}: {len(ids)}/{len(required)} required "
              f"hospitals modelled, {total_slots} slot-days, none with zero visits.")
    return True
 
 
# ── GUARD 2 ──────────────────────────────────────────────────────────────────
def assert_node_coverage(data, lnodes, lnode_idx, shl_name=None, verbose=True):
    """Every (hospital, slot) that is required on some day must have a logical
    node. Catches max_slots being computed from a stale n_required_visits."""
    shl_name = shl_name or data.get("shl_name", "?")
    nrv = data["n_required_visits"]
    have = {(pc, s) for (pc, s) in lnodes}
    need = {(pc, s)
            for pc in data["ids"]
            for t in DAYS
            for s in range(nrv.get((pc, t), 0))}
    missing = sorted(need - have)
    if missing:
        raise AssertionError(
            f"GUARD 2 FAILED — {shl_name}: {len(missing)} required "
            f"(hospital, slot) pair(s) have no logical node: {missing}."
        )
    if verbose:
        print(f"  ✓ GUARD 2 {shl_name}: {len(lnodes)} logical nodes cover all "
              f"{len(need)} required (hospital, slot) pairs.")
    return True
 
 
# ── GUARD 3 — THE MISSING ONE ────────────────────────────────────────────────
def assert_solution_coverage(result, data, shl_name=None, strict=True,
                             verbose=True):
    """Every required node-day must appear EXACTLY ONCE across the final routes.
 
    Call this immediately after solve() returns, before any KPI/export step.
    Nothing in the current pipeline does this: coverage is repaired inside the
    construction heuristics, but the route set that is actually returned is
    never re-checked.
    """
    shl_name = shl_name or data.get("shl_name", "?")
    lnodes = result["lnodes"]
    nrv = data["n_required_visits"]
    visit_days = data["visit_days"]
    routes_by_day = result["routes_by_day"]
 
    problems = []
    for t in DAYS:
        required = {i for i in range(1, len(lnodes) + 1)
                    if (t in visit_days[lnodes[i - 1][0]])
                    and (lnodes[i - 1][1] < nrv.get((lnodes[i - 1][0], t), 0))}
        served = []
        for r in routes_by_day.get(t, []):
            served.extend(r["logical_nodes"])
        served_set = set(served)
 
        missing = sorted(required - served_set)
        extra = sorted(served_set - required)
        dupes = sorted({nd for nd in served if served.count(nd) > 1})
 
        if missing or extra or dupes:
            day = DAY_NAMES.get(t, t)
            if missing:
                problems.append(
                    f"    day {day}: MISSING {len(missing)} node(s) — "
                    + ", ".join(f"{lnodes[i-1][0]}(s={lnodes[i-1][1]},node={i})"
                                for i in missing[:12])
                    + (" ..." if len(missing) > 12 else ""))
            if dupes:
                problems.append(
                    f"    day {day}: DUPLICATED — "
                    + ", ".join(f"{lnodes[i-1][0]}(node={i})" for i in dupes[:12]))
            if extra:
                problems.append(
                    f"    day {day}: NOT REQUIRED but served — "
                    + ", ".join(f"{lnodes[i-1][0]}(node={i})" for i in extra[:12]))
 
    if problems:
        msg = (f"GUARD 3 FAILED — {shl_name}: the returned route set does not "
               f"cover the required node-days exactly once.\n" + "\n".join(problems))
        if strict:
            raise AssertionError(msg)
        print("  ⚠ " + msg)
        return False
 
    if verbose:
        n_req = sum(1 for t in DAYS for i in range(1, len(lnodes) + 1)
                    if (t in visit_days[lnodes[i - 1][0]])
                    and (lnodes[i - 1][1] < nrv.get((lnodes[i - 1][0], t), 0)))
        print(f"  ✓ GUARD 3 {shl_name}: all {n_req} required node-days served "
              f"exactly once.")
    return True
 
 
# ── GUARD 4 ──────────────────────────────────────────────────────────────────
def assert_export_coverage(stops_csv_path, data, shl_name=None, verbose=True):
    """The exported stop-level file must contain every required hospital.
    This is the check that, had it existed, would have caught P162 in the
    published exports."""
    shl_name = shl_name or data.get("shl_name", "?")
    stops = pd.read_csv(stops_csv_path)
    stops["Code"] = stops["Code"].astype(str).str.strip()
    exported = set(stops.loc[stops.get("SHL", shl_name) == shl_name, "Code"]) \
        if "SHL" in stops.columns else set(stops["Code"])
    required = set(data["required_ids"])
    missing = sorted(required - exported)
    if missing:
        raise AssertionError(
            f"GUARD 4 FAILED — {shl_name}: {len(missing)} required hospital(s) "
            f"absent from {stops_csv_path}: {missing}."
        )
    if verbose:
        print(f"  ✓ GUARD 4 {shl_name}: stop export contains all "
              f"{len(required)} required hospitals.")
    return True
 
 
# ── Targeted pre-flight for one hospital ─────────────────────────────────────
def preflight_hospital(pc, shl_name, data=None):
    """Trace one hospital all the way from the trips file to node reachability.
    Use as: preflight_hospital('P162', 'Tooting')
    """
    pc = str(pc).strip()
    if data is None:
        data = load_data(shl_name)
 
    print("=" * 72)
    print(f"PRE-FLIGHT — {pc} @ {shl_name}")
    print("=" * 72)
 
    in_req = pc in set(data["required_ids"])
    in_ids = pc in set(data["ids"])
    print(f"[1] in required_ids (trips file)      : {in_req}")
    print(f"[2] in ids (modelled)                 : {in_ids}")
    if not in_ids:
        print("    >>> lost at load. Run diagnose_missing_hospital() and check "
              "COORD_OVERRIDE / bad_coord / haversine_miles.")
        return data
 
    sw = data["slot_windows"].get(pc, {})
    nrv = data["n_required_visits"]
    print(f"[3] weekday days with a window        : {sorted(data['wdays'].get(pc, []))}")
    for t in DAYS:
        slots = sw.get(t, [])
        print(f"    day {DAY_NAMES.get(t, t)}: n_required_visits="
              f"{nrv.get((pc, t), 0)}  slots={[(s, round(o,2), round(c,2)) for s,o,c in slots]}")
    if sum(nrv.get((pc, t), 0) for t in DAYS) == 0:
        print("    >>> zero required visits: it will hold a logical node and "
              "never be routed. This is the silent-drop mode GUARD 1 catches.")
        return data
 
    # Reachability from the depot on the stated window
    dlat, dlon = DEPOT_COORDS[shl_name]
    lat, lon = data["hlat"][pc], data["hlon"][pc]
    R = 6371.0088
    p = math.radians
    dla, dlo = p(lat - dlat), p(lon - dlon)
    a = (math.sin(dla / 2) ** 2 +
         math.cos(p(dlat)) * math.cos(p(lat)) * math.sin(dlo / 2) ** 2)
    hav_mi = 2 * R * math.asin(math.sqrt(a)) * 0.621371
    rf = data.get("road_factor_shl", ROAD_FACTOR_DEFAULT)
    mph = data.get("calib_mph", 25.0)
    road_mi = hav_mi * rf
    tau_h = road_mi / mph
    sched = data.get("sched_miles", {}).get(pc)
    print(f"\n[4] geometry: haversine={hav_mi:.2f} mi, road_factor={rf:.3f} "
          f"-> {road_mi:.1f} mi; master Miles={sched}")
    if sched:
        print(f"    implied road factor to match master = {sched / hav_mi:.3f} "
              f"(calibrated {rf:.3f})")
    print(f"    tau(depot->{pc}) ≈ {tau_h:.2f} h at {mph:.1f} mph")
 
    for t in DAYS:
        for s, (sn, oh, ch) in enumerate(sw.get(t, [])):
            dp_h = data.get("dp_times_slot", {}).get(
                (pc, t, s), data.get("dp_times", {}).get((pc, t), data["T_DEP"]))
            earliest = max(data["T_DEP"], dp_h) + tau_h
            ok = earliest <= min(ch, data["T_MAX"]) + 5.0 / 60.0
            print(f"    day {DAY_NAMES.get(t,t)} slot {sn}: DP={dp_h:.2f}h "
                  f"earliest={earliest:.2f}h close={ch:.2f}h -> "
                  f"{'reachable' if ok else 'RELAXED (window too tight)'}")
    return data
 
 
print("Coverage guards loaded: assert_load_coverage, assert_node_coverage, "
      "assert_solution_coverage, assert_export_coverage, preflight_hospital.")
 



load_data defined (v28: trips-sourced demand set, strict coverage, override-aware coords, corrected verification scoping).
Coverage guards loaded: assert_load_coverage, assert_node_coverage, assert_solution_coverage, assert_export_coverage, preflight_hospital.


In [6]:
def haversine_km(la1,lo1,la2,lo2):
    R=6371.0; p1,p2=math.radians(la1),math.radians(la2)
    dp,dl=math.radians(la2-la1),math.radians(lo2-lo1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

def haversine_mi(la1,lo1,la2,lo2):
    return haversine_km(la1,lo1,la2,lo2)*KM_TO_MI

def road_km(la1,lo1,la2,lo2,rf=1.0): return rf*haversine_km(la1,lo1,la2,lo2)
def road_mi(la1,lo1,la2,lo2,rf=1.0): return haversine_mi(la1,lo1,la2,lo2)*rf
def travel_h(d_km): return d_km/SPEED_KMH_DEFAULT   # last-resort helper only; see 

def parse_time(s):
    s=str(s).strip()
    if s in ("","None","nan","NaT"): return None
    p=s.split(":")
    try: return int(p[0])+int(p[1])/60.0
    except: return None

def time_diff_min(a,b):
    fa,fb=parse_time(a),parse_time(b)
    if fa is None or fb is None: return None
    d=(fb-fa)*60.0
    return d+1440.0 if d<-60 else d

def uniquify_trip_names(df, shl_name):
    """Give every blank Trip Name a unique label so untitled single-drop
    rounds are not merged into one giant pseudo-route (the Southampton bug
    and a major source of understated baseline mileage)."""
    df=df.copy()
    tn=df["Trip Name"].astype(str).str.strip()
    blank=tn.isin(["","nan","None","NaN"])
    labels=tn.tolist(); n=0
    for idx in df.index[blank]:
        labels[list(df.index).index(idx)]=f"{shl_name}-SINGLE-{n:03d}"; n+=1
    df["Trip Name"]=labels
    return df

# ---  Master-Schedule compound-slot expander -------------------------------
# 3 Filton hospitals (Musgrove Park T158, Yeovil T162, Weston T164) and 1 Tooting
# hospital pack TWO day-groups into one slot with '/':
#  T1 CO=08_00/06_00  DP=10_00/08_00  AR=13_15/11_15  Days=M/TWHF
# meaning Monday uses 10:00 depart, Tue-Fri uses 08:00 depart. If the cleaner that
# builds tw_params.csv does NOT split these, Tue-Fri inherit Monday's later depart
# (or the row defaults to all-days with one mis-parsed time), corrupting the arrival
# window and dispatch group for exactly the long-haul Filton legs (Yeovil/Weston/
# Musgrove). Use this in that upstream cleaner, one output row per returned dict.
_DAY_LETTER={'M':1,'T':2,'W':3,'H':4,'F':5}   # NHSBT: H = Thursday
def expand_days_token(tok):
    tok=str(tok).strip().upper()
    return [_DAY_LETTER[ch] for ch in tok if ch in _DAY_LETTER]
def split_compound_schedule(co, dp, ar, days):
    """Expand a slot that packs several day-groups into one cell with '/'. Non-compound
    cells pass straight through. Robust to only some fields being compound."""
    def parts(v):
        v=str(v).strip(); return v.split('/') if '/' in v else [v]
    dparts=parts(days); n=len(dparts)
    def field(v):
        p=parts(v)
        if len(p)==1: return p*n
        if len(p)==n: return p
        return (p+[p[-1]]*n)[:n]
    cos,dps,ars=field(co),field(dp),field(ar)
    return [dict(days=expand_days_token(dparts[i]),
                 co=cos[i].replace('_',':'), dp=dps[i].replace('_',':'),
                 ar=ars[i].replace('_',':')) for i in range(n)]

print("Utilities defined.")

Utilities defined.


In [7]:
def _fmt_h(h):
    if h is None: return "—"
    hh=int(h); mm=round((h-hh)*60)
    if mm==60: hh+=1; mm=0
    return f"{hh:02d}:{mm:02d}"

def _print_schedule_table(shl_name_label, trips, data, day_for_trip=None):
    """Print a route table. "Depart" is the single depot-dispatch time for the
    whole round; "Arrival" is each stop's solved arrival; a marker flags any
    arrival before the depot departure or the previous stop.

    v18 FIX: slot_name from baseline routes is now a string (matching
    co_times_slot keys). If an integer slips through as a legacy value,
    it is converted via slot_windows before the CO lookup."""
    W={"veh":12,"day":8,"seq":4,"pc":8,"name":42,"co":8,"dp":8,"ar":8,"dist":8}
    hdr=(f"{'Round':<{W['veh']}}  {'Day':<{W['day']}}  {'Seq':>{W['seq']}}  "
         f"{'Code':<{W['pc']}}  {'Hospital':<{W['name']}}  "
         f"{'CO':>{W['co']}}  {'Depart':>{W['dp']}}  {'Arrival':>{W['ar']}}  {'mi':>{W['dist']}}")
    sep="─"*len(hdr)
    print(f"\n  {shl_name_label}\n  {sep}\n  {hdr}\n  {sep}")
    hname_lu=data.get("hname",{}); co_times=data.get("co_times",{})
    co_times_large=data.get("co_times_large",{})
    co_times_slot=data.get("co_times_slot",{})
    co_times_large_slot=data.get("co_times_large_slot",{})
    slot_windows_lu=data.get("slot_windows",{})
    T_DEP_h=data.get("T_DEP")
    for trip in trips:
        t_day=trip.get("_day") or day_for_trip
        route_depart_h=trip.get("route_depart_h")
        dp_h=route_depart_h if route_depart_h is not None else T_DEP_h
        stops=trip.get("stops")
        if stops is None:
            stops=[dict(pc=pc, **(trip.get("arrivals",{}).get(pc) or {}))
                   for pc in trip.get("hospitals",[])]
        n_stops=len(stops)
        prev_arr_h=None
        for sn,stop in enumerate(stops,1):
            pc=stop.get("pc")
            arr_h=stop.get("arrival_h")
            name=hname_lu.get(pc,pc)[:W["name"]]
            slot_name=stop.get("slot")
            # v18 FIX: if slot_name is an integer (legacy baseline path),
            # convert it to the slot name string via slot_windows
            if isinstance(slot_name, int):
                sw_pc = slot_windows_lu.get(pc, {})
                for d_check in ([t_day] if t_day else []) + DAYS:
                    sw_d = sw_pc.get(d_check, [])
                    if slot_name < len(sw_d):
                        slot_name = sw_d[slot_name][0]
                        break
            co_h=co_times_slot.get((pc,t_day,slot_name))
            if co_h is None: co_h=co_times.get((pc,t_day))
            co_lh=co_times_large_slot.get((pc,t_day,slot_name))
            if co_lh is None: co_lh=co_times_large.get((pc,t_day))
            arr_h_display=arr_h
            try:
                _snap_tol_h=float(DISPLAY_SNAP_MIN)/60.0
            except Exception:
                _snap_tol_h=1.0/60.0
            if arr_h is not None and dp_h is not None and abs(arr_h-dp_h)<=_snap_tol_h:
                arr_h_display=dp_h
            warn=""
            if arr_h is not None and arr_h<dp_h-1e-6:
                warn=" ⚠ ARR<DEPOT-DEPART"
            elif arr_h is not None and prev_arr_h is not None and arr_h<prev_arr_h-1e-6:
                warn=" ⚠ ARR<PREV-STOP"
            vs=trip["vehicle"] if sn==1 else ""
            ds=("Weekday" if t_day==0 else DAY_NAMES.get(t_day,str(t_day))) if sn==1 else ""
            dm=f"{trip['km']:.1f}" if sn==1 else ""
            print(f"  {vs:<{W['veh']}}  {ds:<{W['day']}}  {sn:>{W['seq']}}  "
                  f"{pc:<{W['pc']}}  {name:<{W['name']}}  "
                  f"{(_fmt_h(co_h)+(' L:'+_fmt_h(co_lh) if co_lh else '')):>{W['co']+8 if co_lh else W['co']}}  "
                  f"{_fmt_h(dp_h):>{W['dp']}}  {_fmt_h(arr_h_display):>{W['ar']}}  {dm:>{W['dist']}}{warn}")
            if arr_h is not None: prev_arr_h=arr_h
        print(f"  {sep}")


def _export_routes_csv(shl_name, result, data, out_dir, month_label="pooled"):
    """Export the optimised route schedule to a CSV file matching the
    console table format produced by _print_schedule_table.

    Columns: Round, Day, Seq, Code, Hospital, CO, Depart, Arrival, mi
    """
    import pandas as _pd
    from pathlib import Path

    hname_lu = data.get("hname", {})
    co_times = data.get("co_times", {})
    co_times_slot = data.get("co_times_slot", {})
    co_times_large = data.get("co_times_large", {})
    co_times_large_slot = data.get("co_times_large_slot", {})
    slot_windows_lu = data.get("slot_windows", {})
    T_DEP_h = data.get("T_DEP")
    routes_by_day = result["routes_by_day"]

    rows = []
    route_idx = 0

    for t in DAYS:
        for route in routes_by_day.get(t, []):
            route_idx += 1
            vehicle_label = f"OPT-{route_idx:02d}"
            route_depart_h = route.get("route_depart_h")
            dp_h = route_depart_h if route_depart_h is not None else T_DEP_h

            stops = route.get("stops")
            if stops is None:
                stops = [dict(pc=pc) for pc in route.get("hospitals", [])]

            for sn, stop in enumerate(stops, 1):
                pc = stop.get("pc")
                arr_h = stop.get("raw_arrival_h", stop.get("arrival_h"))
                name = hname_lu.get(pc, pc)

                # Resolve slot name (handle legacy integer slot index)
                slot_name = stop.get("slot")
                if isinstance(slot_name, int):
                    sw_pc = slot_windows_lu.get(pc, {})
                    for d_check in ([t] if t else []) + DAYS:
                        sw_d = sw_pc.get(d_check, [])
                        if slot_name < len(sw_d):
                            slot_name = sw_d[slot_name][0]
                            break

                # Cut-off time lookup (same logic as _print_schedule_table)
                co_h = co_times_slot.get((pc, t, slot_name))
                if co_h is None:
                    co_h = co_times.get((pc, t))

                # Route duration: last stop arrival minus depot departure
                _last_arr = stops[-1].get("raw_arrival_h", stops[-1].get("arrival_h"))
                _dur_h = round(_last_arr - dp_h, 2) if (_last_arr and dp_h) else ""
                rows.append({
                    "Round":    vehicle_label,
                    "Day":      DAY_NAMES.get(t, str(t)),
                    "Seq":      sn,
                    "Code":     pc,
                    "Hospital": name,
                    "CO":       _fmt_h(co_h),
                    "Depart":   _fmt_h(dp_h),
                    "Arrival":  _fmt_h(stop.get("raw_arrival_h", arr_h)),
                    "mi":       round(route["km"], 1) if sn == 1 else "",
                    "dur_h":    _dur_h if sn == 1 else "",
                })

    if not rows:
        print("  (no routes to export)")
        return None

    df = _pd.DataFrame(rows)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    fname = f"routes_{shl_name.lower()}_{month_label}.csv"
    csv_path = out_path / fname
    df.to_csv(csv_path, index=False)
    print(f"  Routes CSV → {csv_path}  ({len(df)} rows, {route_idx} routes)")
    return csv_path

# =============================================================================
# SECTION K -- Full eleven-KPI export layer  (v28)
#
# Drop-in replacement for _export_kpi_csv (defined in the display-helpers cell).
# Run this cell AFTER the post-hoc cell (which defines container_shelf_life_hours,
# VEHICLE_VAQTEC_SIZE and _iter_routes_with_day) and BEFORE the run cell.
#
# Fixes two bugs found in the v27 output:
#   BUG 1  N_Early_Arrivals was identically zero on all 927 rounds because the
#          early test compared stop["arrival_h"] (the EFFECTIVE, post-wait
#          arrival, which is >= window open by construction) against the window
#          open. It must compare stop["raw_arrival_h"] (the unshifted arrival).
#          Waiting time = arrival_h - raw_arrival_h was likewise never exported.
#   BUG 2  Integrity_Margin_h and True_Margin_h were identical on all 895
#          populated rounds because BOTH were computed from PERISH_HOURS. The
#          flat operational ceiling now uses U; the true ceiling now uses the
#          DAT48/14 container spec via container_shelf_life_hours(), which is
#          what makes a "hidden breach" detectable at all.
# =============================================================================

# ── K.0  Policy constants (add to the config cell if you prefer) ─────────────
DRIVER_DUTY_CEILING_H = 9.0    # POLICY: max paid duty period per driver-day
CONTRACTED_SHIFT_H    = 8.0    # POLICY: contracted paid shift, for idle-pay floor
TIGHT_BUFFER_MIN      = 15.0   # a stop this close to window close is "fragile"
BLOOD_AGE_TARGET_MIN  = None   # optional clinical target; None = report only


def _safe(x, nd=2):
    return round(x, nd) if isinstance(x, (int, float)) else ""


def _round_products(hosp_list, day, demand_lu):
    """Products with positive demand anywhere on this round, on this day."""
    return sorted({g for pc in hosp_list for g in PRODUCTS
                   if demand_lu.get((pc, g, day), 0.0) > 0})


def _export_kpi_csv(shl_name, result, bsl_metrics, data, out_dir,
                    month_label="pooled", write_hospital_csv=True):
    """v28: per-round KPI export covering all eleven KPI families.

    Returns (round_csv_path, hospital_csv_path, shl_summary_dict).
    """
    import pandas as _pd
    from pathlib import Path
    from statistics import pstdev

    slot_windows_lu = data.get("slot_windows", {})
    demand_lu       = data.get("demand", {})
    bsms_floor      = data.get("bsms_min_visits", {}) or {}
    bsms_tier       = data.get("bsms_tier", {}) or {}
    T_DEP_h         = data.get("T_DEP", 6.0)
    routes_by_day   = result["routes_by_day"]

    # Per-SHL shift ceiling. solve() computes this internally but does not
    # return it -- add `U=U,` to solve()'s return dict and this picks it up.
    U_shl = result.get("U")
    if U_shl is None:
        raise RuntimeError(
            "solve() did not return U -- add `U=U,` to its return dict. Falling "
            "back to the module-level U silently mis-states op_ceiling on every "
            "round and makes Integrity_Margin_h uninterpretable.")

    relaxed = result.get("relaxed_nodes")
    if relaxed is None:
        raise RuntimeError(
            "solve() did not return relaxed_nodes -- see the v29 relaxed-window "
            "register. Without it, late arrivals cannot be separated from window "
            "relaxations and N_Late_Arrivals is uninterpretable.")

    # ── Baseline indices, for the two stability computations ────────────────
    bsl_round_sets, bsl_hosp_day = {}, {}
    if bsl_metrics and bsl_metrics.get("all_trips"):
        for bt in bsl_metrics["all_trips"]:
            d = bt.get("_day", 0)
            seq = bt.get("hospitals", [])
            bsl_round_sets[(bt.get("vehicle", "?"), d)] = set(seq)
            for pos, pc in enumerate(seq):
                bsl_hosp_day.setdefault((pc, d), (pos, set(seq) - {pc}))

    # ── PASS 1: build raw per-round records ─────────────────────────────────
    raw, visits_by_hosp, arrivals_by_hosp, hosp_stop_rows = [], {}, {}, []
    _age_outliers = []
    route_idx = 0

    for t in DAYS:
        for route in routes_by_day.get(t, []):
            route_idx += 1
            label   = f"OPT-{route_idx:02d}"
            vtype   = route.get("vtype", "Diesel Van")
            csize   = VEHICLE_VAQTEC_SIZES.get(vtype, "Large")
            km      = route.get("km", 0.0)
            dur_h   = route.get("dur_h")            # depot-to-depot, incl. return
            dp_h    = route.get("route_depart_h", T_DEP_h)
            hosp    = route.get("hospitals", [])
            stops   = route.get("stops") or [dict(pc=pc) for pc in hosp]

            # ---- KPI 3: container integrity -- TWO GENUINELY DISTINCT CEILINGS
            prods = _round_products(hosp, t, demand_lu) or ["Blood"]

            # (a) flat operational ceiling: what the MILP actually enforced
            op_ceiling = U_shl
            if USE_PRODUCT_PERISHABILITY:
                bio = [PERISH_HOURS[g] for g in prods if g in PERISH_HOURS]
                if bio:
                    op_ceiling = min(min(bio), U_shl)

            # (b) true DAT48/14 container ceiling: thermal spec for THIS container
            dat = {g: container_shelf_life_hours(vtype, g) for g in prods}
            incompat = sorted(g for g, h in dat.items() if h is None)
            numeric  = [h for h in dat.values() if h is not None]
            true_ceiling = min(numeric) if numeric else None

            op_margin   = (op_ceiling - dur_h) if dur_h is not None else None
            true_margin = (true_ceiling - dur_h) if (true_ceiling is not None
                                                     and dur_h is not None) else None
            hidden_breach = (true_margin is not None and op_margin is not None
                             and op_margin >= 0 and true_margin < 0)

            # ---- stop-level sweep: age, early, late, wait, buffer -----------
            ages, earlies, lates, waits, buffers = [], [], [], [], []
            widths, seq_pos = [], {}
            n_relaxed_stops = 0
            for pos, stop in enumerate(stops):
                pc  = stop.get("pc")
                eff = stop.get("arrival_h")                    # post-wait
                rawa = stop.get("raw_arrival_h", eff)          # unshifted
                slot = stop.get("slot")
                if pc is None:
                    continue
                seq_pos[pc] = pos
                visits_by_hosp[pc] = visits_by_hosp.get(pc, 0) + 1

                # KPI 4: blood age, measured to DELIVERY (not to depot return)
                if eff is not None and dp_h is not None:
                    age = (eff - dp_h) * 60.0
                    # v29: the old `if 0 <= age <= 600` guard silently dropped
                    # rounds from the round-level age statistics while the stop
                    # rows wrote Blood_Age_min unfiltered, so the two layers were
                    # computed on different samples -- and because the same guard
                    # gated arrivals_by_hosp, Arrival_Predictability_min inherited
                    # it. Nothing is dropped now; anomalies are reported.
                    ages.append(age)
                    arrivals_by_hosp.setdefault(pc, []).append(eff)
                    if not (0 <= age <= 600):
                        _age_outliers.append((shl_name, pc, DAY_NAMES.get(t, t),
                                              label, round(age, 1)))

                # KPI 7 (a): waiting time -- FIX, was never exported
                if eff is not None and rawa is not None:
                    waits.append(max(0.0, (eff - rawa) * 60.0))

                # window lookup -- also recovers the slot INDEX, which the
                # relaxed-window register is keyed on
                oh = ch = None; s_idx = None
                for _k, (sn_name, o, c_) in enumerate(slot_windows_lu.get(pc, {}).get(t, [])):
                    if sn_name == slot:
                        oh, ch, s_idx = o, c_, _k
                        break
                is_relaxed = (pc, t, s_idx) in relaxed if s_idx is not None else False
                if is_relaxed:
                    n_relaxed_stops += 1
                if oh is not None and ch is not None:
                    widths.append((ch - oh) * 60.0)
                    # KPI 7 (b): EARLY -- FIX: test the RAW arrival, not the
                    # effective one. This is the line that produced 927 zeros.
                    if rawa is not None and rawa < oh:
                        earlies.append((oh - rawa) * 60.0)
                    # KPI 6: LATE and buffer to close, on the effective arrival
                    if eff is not None:
                        if eff > ch:
                            lates.append((eff - ch) * 60.0)
                        buffers.append((ch - eff) * 60.0)

                if write_hospital_csv:
                    hosp_stop_rows.append(dict(
                        SHL=shl_name, Month=month_label, Day=DAY_NAMES.get(t, str(t)),
                        Round=label, Seq=pos + 1, Code=pc, Slot=slot,
                        # v29: 4dp (0.36s). At 2dp the hour columns had 0.6-min
                        # resolution while the minute columns were full precision,
                        # so Buffer_min could not be reproduced from them.
                        Raw_Arrival_h=_safe(rawa, 4), Eff_Arrival_h=_safe(eff, 4),
                        Window_Open_h=_safe(oh, 4), Window_Close_h=_safe(ch, 4),
                        Wait_min=_safe((eff - rawa) * 60.0, 1) if (eff is not None and rawa is not None) else "",
                        Early_min=_safe((oh - rawa) * 60.0, 1) if (oh is not None and rawa is not None and rawa < oh) else 0.0,
                        Late_min=_safe((eff - ch) * 60.0, 1) if (ch is not None and eff is not None and (eff - ch) * 60.0 > 0.5) else 0.0,
                        Buffer_min=_safe((ch - eff) * 60.0, 1) if (ch is not None and eff is not None) else "",
                        Blood_Age_min=_safe((eff - dp_h) * 60.0, 1) if (eff is not None and dp_h is not None) else "",
                        BSMS_Tier=bsms_tier.get(pc, ""),
                        BSMS_Floor=bsms_floor.get(pc, ""),
                        # v29: window-relaxation provenance for this stop
                        Window_Relaxed=is_relaxed,
                        Relax_Gap_min=_safe(relaxed[(pc, t, s_idx)]["gap_min"], 2)
                                      if is_relaxed else 0.0,
                        Window_Zero_Width=bool(relaxed[(pc, t, s_idx)]["zero_width"])
                                          if is_relaxed else False,
                    ))

            # ---- KPI 10: duty / idle-pay -----------------------------------
            wait_h  = sum(waits) / 60.0
            paid_h  = (dur_h + wait_h) if dur_h is not None else None
            duty_margin = (DRIVER_DUTY_CEILING_H - paid_h) if paid_h is not None else None
            idle_pay_h  = max(0.0, CONTRACTED_SHIFT_H - paid_h) if paid_h is not None else None

            # ---- delivery span vs depot-to-depot duration -------------------
            # dur_h includes the return leg; the perishability clock stops at
            # the LAST DELIVERY. Reporting both makes the difference auditable.
            last_arr = stops[-1].get("arrival_h") if stops else None
            span_h = (last_arr - dp_h) if (last_arr is not None and dp_h is not None) else None
            span_margin = (true_ceiling - span_h) if (true_ceiling is not None
                                                      and span_h is not None) else None

            # ---- KPI 4: blood-age MARGIN (not just age) --------------------
            age_ceiling_min = true_ceiling * 60.0 if true_ceiling is not None else None
            age_margin_min  = (age_ceiling_min - max(ages)) if (age_ceiling_min is not None and ages) else None

            # ---- KPI 8: stability, both computations -----------------------
            opt_set = set(hosp)
            best_j = 0.0
            for (bv, bd), bset in bsl_round_sets.items():
                if bd == t and bset:
                    u = len(opt_set | bset)
                    if u:
                        best_j = max(best_j, len(opt_set & bset) / u)
            mates_same = seq_same = n_cmp = 0
            for pc in hosp:
                if (pc, t) in bsl_hosp_day:
                    n_cmp += 1
                    bpos, bmates = bsl_hosp_day[(pc, t)]
                    if bmates == (opt_set - {pc}):
                        mates_same += 1
                    if bpos == seq_pos.get(pc):
                        seq_same += 1

            # ---- KPI 11: EV --------------------------------------------------
            ev_range = VEHICLE_TYPES.get("Electric Van", {}).get("max_range_mi")
            ev_feas  = (km <= ev_range) if ev_range else True

            raw.append(dict(
                _t=t, _hosp=hosp, _label=label,
                SHL=shl_name, Month=month_label, Day=DAY_NAMES.get(t, str(t)),
                Round=label, Vehicle_Type=vtype, Container_Size=csize,
                Stops=len(hosp), Hospitals="; ".join(hosp),
                # KPI 1 / 2
                Total_Miles=round(km, 1),
                CO2_kg=round(route.get("co2_kg", 0.0), 2),
                CO2_Factor_kg_per_mi=VEHICLE_TYPES[vtype]["co2_kg_per_mile"],
                # timing spine
                Depart_h=_safe(dp_h, 4), Last_Arrival_h=_safe(last_arr, 4),
                Duration_h=_safe(dur_h, 4),       # depot-to-depot, incl. return
                Delivery_Span_h=_safe(span_h, 4), # depart -> last delivery
                Return_Leg_h=_safe(dur_h - span_h) if (dur_h is not None and span_h is not None) else "",
                # KPI 9
                Boxes_Used=route.get("boxes_est", 0),
                Box_Capacity=route.get("boxes_capacity", VEHICLE_TYPES[vtype]["boxes_capacity"]),
                Box_Utilisation=_safe(route.get("box_utilisation"), 3),
                # KPI 3
                Products_On_Round="; ".join(prods),
                Operational_Ceiling_h=_safe(op_ceiling),
                Integrity_Margin_h=_safe(op_margin),
                True_Container_Ceiling_h=_safe(true_ceiling),
                True_Margin_h=_safe(true_margin),
                Delivery_Margin_h=_safe(span_margin),
                Hidden_Breach=bool(hidden_breach),
                Container_Incompatible="; ".join(incompat),
                # KPI 4
                Mean_Blood_Age_min=_safe(sum(ages) / len(ages), 1) if ages else "",
                Max_Blood_Age_min=_safe(max(ages), 1) if ages else "",
                Blood_Age_Ceiling_min=_safe(age_ceiling_min, 1),
                Min_Blood_Age_Margin_min=_safe(age_margin_min, 1),
                N_Age_Breaches=sum(1 for a in ages if age_ceiling_min is not None and a > age_ceiling_min),
                # KPI 6
                N_Late_Arrivals=len(lates),
                Mean_Late_min=_safe(sum(lates) / len(lates), 1) if lates else 0.0,
                Max_Late_min=_safe(max(lates), 1) if lates else 0.0,
                Mean_Buffer_min=_safe(sum(buffers) / len(buffers), 1) if buffers else "",
                Min_Buffer_min=_safe(min(buffers), 1) if buffers else "",
                N_Tight_Stops=sum(1 for b in buffers if b < TIGHT_BUFFER_MIN),
                # KPI 7
                N_Early_Arrivals=len(earlies),
                Mean_Early_min=_safe(sum(earlies) / len(earlies), 1) if earlies else 0.0,
                Max_Early_min=_safe(max(earlies), 1) if earlies else 0.0,
                Total_Wait_min=_safe(sum(waits), 1),
                Mean_Wait_min=_safe(sum(waits) / len(waits), 1) if waits else 0.0,
                Max_Wait_min=_safe(max(waits), 1) if waits else 0.0,
                Mean_Window_Width_min=_safe(sum(widths) / len(widths), 1) if widths else "",
                # KPI 8
                # v29: window-relaxation provenance. Late_Is_Relaxation True means
                # every late arrival on this round sits at a node whose stated
                # window was relaxed because it is unreachable on a direct depot
                # leg -- i.e. an input-data defect, not a routing outcome.
                N_Relaxed_Windows=n_relaxed_stops,
                Late_Is_Relaxation=bool(len(lates) == 0 or n_relaxed_stops >= len(lates)),
                N_Boundary_Stops=sum(1 for b in buffers if b <= 1.0),
                Route_Stability=round(best_j, 3),
                Stability_Valid=bool(bsl_round_sets),
                Route_Mates_Unchanged=_safe(mates_same / n_cmp, 3) if n_cmp else "",
                Seq_Position_Unchanged=_safe(seq_same / n_cmp, 3) if n_cmp else "",
                # KPI 10
                Wait_h=_safe(wait_h),
                Paid_Duration_h=_safe(paid_h),
                Duty_Ceiling_h=DRIVER_DUTY_CEILING_H,
                Duty_Margin_h=_safe(duty_margin),
                Duty_Breach=bool(duty_margin is not None and duty_margin < 0),
                Idle_Pay_h=_safe(idle_pay_h),
                # KPI 11
                EV_Feasible=ev_feas,
                EV_Range_mi=ev_range if ev_range else "",
                EV_Margin_mi=_safe(ev_range - km, 1) if ev_range else "",
            ))

    if not raw:
        print("  (no KPI rows to export)")
        return None, None, None

    # ── PASS 2: network-level derivations that need all rounds ──────────────
    # KPI 7 (c): arrival predictability -- spread of a hospital's arrival hour
    # across the planning week. Low spread = the hospital can staff to it.
    # v29: POPULATION standard deviation (pstdev), in minutes, of a hospital's
    # arrival times across the solved week. UNDEFINED (None, not 0.0) for
    # single-visit hospitals -- a 0.0 there means "one visit", not "perfectly
    # consistent", and averaging those in understates the network mean sharply.
    predict, predict_n = {}, {}
    for pc, hrs in arrivals_by_hosp.items():
        predict_n[pc] = len(hrs)
        predict[pc] = round(pstdev(hrs) * 60.0, 1) if len(hrs) > 1 else None

    # KPI 5: BSMS tier floor, evaluated on the whole solved week
    below = {pc: (bsms_tier.get(pc, "?"), floor, visits_by_hosp.get(pc, 0))
             for pc, floor in bsms_floor.items()
             if pc in data.get("ids", []) and visits_by_hosp.get(pc, 0) < floor}
    tier_rank = {"Very High": 5, "High": 4, "Moderate": 3, "Low": 2, "Very Low": 1}

    for r in raw:
        hosp = r.pop("_hosp"); r.pop("_t"); r.pop("_label")
        tiers = [bsms_tier.get(pc) for pc in hosp if bsms_tier.get(pc)]
        r["BSMS_Top_Tier"] = max(tiers, key=lambda x: tier_rank.get(x, 0)) if tiers else ""
        r["BSMS_N_Below_Floor"] = sum(1 for pc in hosp if pc in below)
        r["BSMS_Below_Floor_Codes"] = "; ".join(pc for pc in hosp if pc in below)
        sp = [predict[pc] for pc in hosp if predict.get(pc) is not None]
        r["Arrival_Predictability_min"] = round(sum(sp) / len(sp), 1) if sp else ""

    df = _pd.DataFrame(raw)
    # v30 PATCH 4: a hospital with ZERO visits appears in no round, so
    # BSMS_N_Below_Floor is 0 everywhere and the test is blind to the failure
    # mode it exists to catch. Carry the SHL-level verdict down.
    df = patch_bsms_round_layer(df, below, data)
    out_path = Path(out_dir); out_path.mkdir(parents=True, exist_ok=True)
    csv_path = out_path / f"kpi_{shl_name.lower()}_{month_label}.csv"
    df.to_csv(csv_path, index=False)
    print(f"  KPI CSV → {csv_path}  ({len(df)} rounds, {len(df.columns)} columns)")

    # ── hospital-level export (KPI 4, 5, 6, 7 at the level they belong) ─────
    hosp_path = None
    if write_hospital_csv and hosp_stop_rows:
        hdf = _pd.DataFrame(hosp_stop_rows)
        hdf["Visits_This_Week"] = hdf["Code"].map(visits_by_hosp)
        hdf["Arrival_Predictability_min"] = hdf["Code"].map(predict)   # NaN where undefined
        hdf["Predictability_N_Visits"] = hdf["Code"].map(predict_n)
        hdf["BSMS_Below_Floor"] = hdf["Code"].isin(below.keys())
        # v30 PATCH 4: complete the export over the REQUIRED set, not the served
        # set, so zero-visit hospitals appear with Served=False.
        hdf = patch_bsms_stop_layer(hdf, below, data, visits_by_hosp,
                                    bsms_tier, bsms_floor, shl_name, month_label)
        hosp_path = out_path / f"kpi_stops_{shl_name.lower()}_{month_label}.csv"
        hdf.to_csv(hosp_path, index=False)
        print(f"  Stop-level CSV → {hosp_path}  ({len(hdf)} stops)")

    # ── KPI 9: SHL-level fleet utilisation (cannot be a per-round column) ───
    fleet_comp = result.get("fleet_composition", {}) or {}
    Cd = sum(fleet_comp.values()) or result.get("Cd", 0)
    n_days = len(DAYS)
    slots = Cd * n_days
    box_cap_avail = sum(cnt * VEHICLE_TYPES[t]["boxes_capacity"]
                        for t, cnt in fleet_comp.items()) * n_days
    veh_h_avail = Cd * n_days * U_shl
    boxes_used = df["Boxes_Used"].sum()
    hours_used = _pd.to_numeric(df["Duration_h"], errors="coerce").sum()
    paid_used = _pd.to_numeric(df["Paid_Duration_h"], errors="coerce").sum()
    
    # ── FIX: KPI 11 EV Metrics safe calculation ──
    ev_feasible_avg = round(df["EV_Feasible"].mean(), 4) if not df.empty and "EV_Feasible" in df.columns else None
    ev_feas_25_pct = bool((df["EV_Feasible"].sum() / len(df)) >= 0.25) if not df.empty and "EV_Feasible" in df.columns else None
    ev_feas_50_pct = bool((df["EV_Feasible"].sum() / len(df)) >= 0.50) if not df.empty and "EV_Feasible" in df.columns else None
    ev_feas_75_pct = bool((df["EV_Feasible"].sum() / len(df)) >= 0.75) if not df.empty and "EV_Feasible" in df.columns else None
    ev_feas_100_pct = bool(df["EV_Feasible"].all()) if not df.empty and "EV_Feasible" in df.columns else None
    ev_worst_margin_mi = round(_pd.to_numeric(df["EV_Margin_mi"], errors="coerce").min(), 1) if not df.empty and "EV_Margin_mi" in df.columns else None

    summary = dict(
        SHL=shl_name, Month=month_label, n_rounds=len(df),
        # KPI 1 / 2
        total_miles=round(df["Total_Miles"].sum(), 1),
        total_co2_kg=round(df["CO2_kg"].sum(), 1),
        blended_co2_factor=round(df["CO2_kg"].sum() / max(df["Total_Miles"].sum(), 1e-9), 6),
        all_diesel_co2_kg=round(VEHICLE_TYPES["Diesel Van"]["co2_kg_per_mile"] * df["Total_Miles"].sum(), 1),
        # KPI 9
        dispatch_slot_utilisation=round(len(df) / slots, 4) if slots else None,
        box_capacity_utilisation=round(boxes_used / box_cap_avail, 4) if box_cap_avail else None,
        vehicle_hours_utilisation=round(hours_used / veh_h_avail, 4) if veh_h_avail else None,
        paid_hours_utilisation=round(paid_used / veh_h_avail, 4) if veh_h_avail else None,
        # KPI 3
        n_hidden_breach=int(df["Hidden_Breach"].sum()),
        n_container_incompatible=int((df["Container_Incompatible"] != "").sum()),
        min_true_margin_h=_pd.to_numeric(df["True_Margin_h"], errors="coerce").min(),
        min_delivery_margin_h=_pd.to_numeric(df["Delivery_Margin_h"], errors="coerce").min(),
        # KPI 5
        bsms_checked=len([p for p in bsms_floor if p in data.get("ids", [])]),
        bsms_below_floor=len(below),
        bsms_below_detail="; ".join(f"{p}({t},{f}>{a})" for p, (t, f, a) in sorted(below.items())),
        # KPI 6 / 7  -- window relaxations (v29)
        n_relaxed_nodes=result.get("n_relaxed_nodes", 0),
        n_relaxed_beyond_slack=result.get("n_relaxed_beyond_slack", 0),
        n_relaxed_zero_width=result.get("n_relaxed_zero_width", 0),
        n_relaxed_stops=int(df["N_Relaxed_Windows"].sum()),
        late_fully_explained=bool(df["Late_Is_Relaxation"].all()),
        n_late_rounds=int((df["N_Late_Arrivals"] > 0).sum()),
        n_early_rounds=int((df["N_Early_Arrivals"] > 0).sum()),
        total_wait_h=round(_pd.to_numeric(df["Total_Wait_min"], errors="coerce").sum() / 60.0, 2),
        n_tight_stops=int(df["N_Tight_Stops"].sum()),          # buffer <= TIGHT_BUFFER_MIN
        n_boundary_stops=int(df["N_Boundary_Stops"].sum()),    # buffer <= 1 min
        # v29: computed over hospitals where the statistic is DEFINED (>=2 visits)
        mean_predictability_min=(round(sum(_v for _v in predict.values() if _v is not None)
                                       / max(sum(1 for _v in predict.values() if _v is not None), 1), 1)
                                 if any(_v is not None for _v in predict.values()) else None),
        n_predictability_defined=sum(1 for _v in predict.values() if _v is not None),
        n_predictability_single_visit=sum(1 for _v in predict.values() if _v is None),
        # KPI 8: all three stability axes, plus the round-count comparison that
        # rho is structurally unable to detect
        stability_valid=bool(bsl_round_sets),
        rho_mean=round(df["Route_Stability"].mean(), 3),
        rho_min=round(df["Route_Stability"].min(), 3),
        rho_perfect_share=round((df["Route_Stability"] >= 0.9995).mean(), 3),
        mates_unchanged_mean=round(_pd.to_numeric(df["Route_Mates_Unchanged"],
                                                  errors="coerce").mean(), 3),
        seq_position_unchanged_mean=round(_pd.to_numeric(df["Seq_Position_Unchanged"],
                                                         errors="coerce").mean(), 3),
        n_rounds_opt=len(df),
        n_rounds_bsl=len(bsl_round_sets),
        round_count_delta=len(df) - len(bsl_round_sets),
        round_count_ratio=round(len(df) / max(len(bsl_round_sets), 1), 3),
        # KPI 10
        n_duty_breach=int(df["Duty_Breach"].sum()),
        total_idle_pay_h=round(_pd.to_numeric(df["Idle_Pay_h"], errors="coerce").sum(), 1),
        # KPI 11
        ev_feasible_share=ev_feasible_avg,
        ev_feas_25_pct=ev_feas_25_pct,
        ev_feas_50_pct=ev_feas_50_pct,
        ev_feas_75_pct=ev_feas_75_pct,
        ev_feas_100_pct=ev_feas_100_pct,
        ev_worst_margin_mi=ev_worst_margin_mi,
    )
    # ── v29: relaxed-window audit list. THIS FILE IS THE FIX LIST. ──────────
    if relaxed:
        rrows = [dict(SHL=shl_name, Month=month_label, Code=_pc,
                      Day=DAY_NAMES.get(_t, _t), Slot_Index=_s, **_v)
                 for (_pc, _t, _s), _v in relaxed.items()]
        rdf = _pd.DataFrame(rrows).sort_values("gap_min", ascending=False)
        rpath = out_path / "relaxed_windows_audit.csv"
        if rpath.exists():
            _old = _pd.read_csv(rpath)
            _old = _old[~((_old["SHL"] == shl_name) & (_old["Month"] == month_label))]
            rdf = _pd.concat([_old, rdf], ignore_index=True)
        rdf.to_csv(rpath, index=False)
        print(f"  Relaxed-window audit → {rpath}  ({len(rrows)} node-day(s) this SHL)")

    spath = out_path / "kpi_shl_summary.csv"
    sdf = _pd.DataFrame([summary])
    if spath.exists():
        old = _pd.read_csv(spath)
        old = old[~((old["SHL"] == shl_name) & (old["Month"] == month_label))]
        sdf = _pd.concat([old, sdf], ignore_index=True)
    sdf.to_csv(spath, index=False)
    print(f"  SHL summary → {spath}")

    # ── console guard rails: these should never fire silently ───────────────
    if summary["n_early_rounds"] == 0 and summary["total_wait_h"] == 0:
        print("  ⚠ No early arrivals AND no waiting recorded. If MAX_WAIT_H > 0 this "
              "means stops carry no raw_arrival_h — check solve() emitted it.")
    if summary["n_hidden_breach"]:
        print(f"  ⚠ {summary['n_hidden_breach']} hidden breach(es): pass the flat "
              f"U={U_shl}h ceiling but fail the DAT48/14 container ceiling.")
    if summary["n_duty_breach"]:
        print(f"  ⚠ {summary['n_duty_breach']} round(s) exceed the "
              f"{DRIVER_DUTY_CEILING_H}h paid-duty ceiling once waiting is included.")
    if summary["bsms_below_floor"]:
        print(f"  ⚠ {summary['bsms_below_floor']} hospital(s) below their BSMS tier floor.")
    if _age_outliers:
        print(f"  ⚠ {len(_age_outliers)} stop(s) with product age outside [0, 600] min "
              f"(previously dropped silently): {_age_outliers[:8]}")
    if summary["n_relaxed_nodes"]:
        print(f"  ⚠ {summary['n_relaxed_nodes']} node-day window(s) relaxed "
              f"({summary['n_relaxed_zero_width']} zero-width, "
              f"{summary['n_relaxed_beyond_slack']} beyond slack). "
              f"Late arrivals fully explained by relaxation: "
              f"{summary['late_fully_explained']}.")
    if not summary["stability_valid"]:
        print("  ⚠ No baseline rounds available: every Route_Stability value is a "
              "hard zero from initialisation, NOT a measured re-draw. Suppress "
              "all stability figures for this SHL.")
    if summary["round_count_delta"]:
        print(f"  ℹ Round count changed vs baseline: {summary['n_rounds_bsl']} -> "
              f"{summary['n_rounds_opt']} ({summary['round_count_delta']:+d}). "
              f"rho is structurally blind to this.")

    return csv_path, hosp_path, summary


print("KPI export v28 defined — all eleven KPI families, early-arrival and "
      "dual-ceiling bugs fixed.")


KPI export v28 defined — all eleven KPI families, early-arrival and dual-ceiling bugs fixed.


In [8]:
def build_arcs(data):
    """Build distance (c) and travel-time (tau) over LOGICAL nodes (one node per
    hospital delivery slot). Priority: observed value > given schedule field
    > calibrated estimate, applied separately to distance and time."""
    ids=data["ids"]; n_required_visits=data["n_required_visits"]
    dlat,dlon=data["dlat"],data["dlon"]
    sched_tau_raw=data.get("sched_tau",{}); sched_tau_h2h=data.get("sched_tau_h2h",{})
    emer_tau=data.get("emer_tau",{})
    sched_miles=data.get("sched_miles",{})
    road_factor_sl=data.get("road_factor_shl",ROAD_FACTOR_DEFAULT)
    calib_mph=data.get("calib_mph",FALLBACK_MPH_DEFAULT)
    hlat,hlon=data["hlat"],data["hlon"]

    max_slots={pc:max(n_required_visits.get((pc,t),0) for t in DAYS) for pc in ids}
    lnodes=[]; lnode_idx={}
    for pc in ids:
        for s in range(max(1, max_slots[pc])):
            lnode_idx[(pc,s)]=len(lnodes)+1
            lnodes.append((pc,s))
    N=len(lnodes)

    old_pc_to_node={ids[i]:i+1 for i in range(len(ids))}

    def lat(n):
        if n==0: return dlat
        return hlat[lnodes[n-1][0]]
    def lon(n):
        if n==0: return dlon
        return hlon[lnodes[n-1][0]]

    all_logical=list(range(N+1)); c={}; tau={}
    n_sched_tau=0; n_sched_mi=0; n_emer_tau=0

    for i in all_logical:
        for j in all_logical:
            if i==j: continue
            pc_i=lnodes[i-1][0] if i>0 else None
            pc_j=lnodes[j-1][0] if j>0 else None

            if i==0 and pc_j in sched_miles:
                c[(i,j)]=sched_miles[pc_j]; n_sched_mi+=1
            elif j==0 and pc_i in sched_miles:
                c[(i,j)]=sched_miles[pc_i]
            else:
                c[(i,j)]=haversine_mi(lat(i),lon(i),lat(j),lon(j))*road_factor_sl

            old_i=old_pc_to_node.get(pc_i) if pc_i else 0
            old_j=old_pc_to_node.get(pc_j) if pc_j else 0

            # v21 depot legs: both DATA sources considered, tightest wins.
            # emer_tau now carries the Master-Schedule AR-DP interval per
            # hospital (see load_data v21); observed first-stop elapsed can
            # include pre-departure dwell, so where the schedule interval is
            # tighter it is the better transit estimate. Both are data.
            if i==0 and (old_j in sched_tau_raw or pc_j in emer_tau):
                cands=[]
                if old_j in sched_tau_raw: cands.append(sched_tau_raw[old_j])
                if pc_j in emer_tau: cands.append(emer_tau[pc_j])
                tau[(i,j)]=min(cands)
                if old_j in sched_tau_raw and sched_tau_raw[old_j]<=tau[(i,j)]+1e-9:
                    n_sched_tau+=1
                else:
                    n_emer_tau+=1
            elif j==0 and pc_i in emer_tau:
                tau[(i,j)]=emer_tau[pc_i]; n_emer_tau+=1   # return legs: schedule interval
            elif i>0 and j>0 and pc_i!=pc_j and (old_i,old_j) in sched_tau_h2h:
                tau[(i,j)]=sched_tau_h2h[(old_i,old_j)]
            else:
                # last resort: real Miles / speed calibrated from observed legs
                tau[(i,j)]=c[(i,j)]/calib_mph  # no artificial floor; depot-adjacent hospitals get tau≈0

    n_h2h_sched=sum(1 for ii in range(1,N+1) for jj in range(1,N+1) if ii!=jj
                    and lnodes[ii-1][0]!=lnodes[jj-1][0]
                    and (old_pc_to_node.get(lnodes[ii-1][0]),old_pc_to_node.get(lnodes[jj-1][0])) in sched_tau_h2h)
    n_fallback=N*N - n_sched_tau - n_h2h_sched - n_emer_tau
    print(f"  Logical nodes: {N}  ({len(ids)} physical hospitals, max_slots={max(max_slots.values(),default=1)})")
    print(f"  Arc costs: {n_sched_mi} depot arcs from real Master Schedule Miles; "
          f"remaining arcs use Haversine x road_factor_shl={road_factor_sl:.3f} (calibrated from this SHL's own real Miles)")
    print(f"  Arc tau  : {n_sched_tau} depot legs + {n_h2h_sched} h2h legs from real Trips-and-Sequences observations, "
          f"{n_emer_tau} depot legs from given Ad Hoc/Emer Travel Time; "
          f"{n_fallback} arcs use calib_mph={calib_mph:.1f}mph (this SHL's own calibrated speed, not an assumed constant)")
    return c, tau, lnode_idx, lnodes, N

def make_prob(name, shl_name=None):
    """v19: presolve ON, feasibility-emphasis heuristics, tighter node-selection."""
    p = xp.problem(name)
    p.controls.outputlog = 1 if LOG else 0
    p.controls.timelimit = (PER_SHL_TIME_LIMIT.get(shl_name, TIME_LIMIT)
                            if shl_name else TIME_LIMIT)
    # v25: early termination when gap is acceptable (5%)
    try:
        p.controls.miprelstop = 0.001
    except Exception:
        pass
    # Only disable presolve when explicitly debugging infeasibility
    if globals().get('IIS_DEBUG', False):
        p.controls.presolve = 0
    # v19: solver tuning for feasibility
    # Control names vary across Xpress versions; silently skip any that don't exist.
    for ctrl, val in [("heurstrategy", 1), ("heursearcheffort", 2)]:
        try:
            setattr(p.controls, ctrl, val)
        except Exception:
            pass
    return p

print("build_arcs and make_prob defined (v19: feasibility-emphasis solver controls).")


build_arcs and make_prob defined (v19: feasibility-emphasis solver controls).


In [9]:
def baseline_routes(data, c, lnode_idx, lnodes):
    shl_name=data.get("shl_name","")
    trips_df=data["trips_df"]; hlat=data["hlat"]; hlon=data["hlon"]
    hname_lu=data["hname"]; dlat=data["dlat"]; dlon=data["dlon"]; ids=data["ids"]

    shl_trips=trips_df[trips_df["SHL"]==shl_name].copy()
    if len(shl_trips)==0:
        print(f"  ⚠ No trips_clean rows for {shl_name}"); return None,None

  # (Southampton/Oxford ffill merge): the cleaner forward-filled the Trip Name
  # over blank-named rounds, so grouping by "Trip Name" fused operationally separate
  # trips into one bogus mega-round (Southampton: 8 real rounds -> 1 loop of 10 stops,
  # understating baseline mileage and driving the -166% artefact). The surviving,
  # trustworthy signal is Delivery Sequence, which RESETS TO 1 at the start of every
  # real round. We rebuild round_id from sequence resets IN FILE ORDER (before any
  # Pulse-Code filtering or sorting, so a filtered lead-stop cannot merge two rounds),
  # then group on that instead of the corrupted Trip Name. Validated to recover the
  # exact per-SHL round counts from the raw Hospital_Trips_and_Sequences workbook.
    # v19.3 ROBUST round identity (matches the v13 data-processing engine):
    # contiguous (Trip Name) blocks -> subgroup by (depart_min, cutoff_min) ->
    # stable-sort by Delivery Sequence -> split on sequence resets. Survives
    # out-of-order rows AND same-name day-group variant rounds (FIL-WT08
    # M + TWHF blocks; P603 15.0 MW/TH/F singleton variants) that plain
    # seq-reset-in-file-order fuses or shreds.
    shl_trips = shl_trips.reset_index(drop=True)
    _blk=[]; _cur=0; _prev=None
    for _tn in shl_trips["Trip Name"].tolist():
        if _tn != _prev: _cur += 1; _prev = _tn
        _blk.append(_cur)
    shl_trips = shl_trips.assign(_blk=_blk)
    _dep_c = pd.to_numeric(shl_trips.get("depart_min"), errors="coerce")
    _co_c  = pd.to_numeric(shl_trips.get("cutoff_min"), errors="coerce")
    shl_trips = shl_trips.assign(_depm=_dep_c, _com=_co_c)
    _rid_of={}; _r=0
    for _key, _g in shl_trips.groupby(["_blk","_depm","_com"], dropna=False, sort=False):
        _g = _g.assign(_seqn=pd.to_numeric(_g["Delivery Sequence"], errors="coerce"))
        _g = _g.sort_values("_seqn", kind="stable")
        _prev_s=None
        for _ix, _row in _g.iterrows():
            _s=_row["_seqn"]
            if _prev_s is None or (pd.notna(_s) and pd.notna(_prev_s) and _s<=_prev_s):
                _r+=1
            _rid_of[_ix]=_r; _prev_s=_s
    shl_trips = shl_trips.assign(round_id=shl_trips.index.map(_rid_of),
                                 _orig_trip_name=shl_trips["Trip Name"]
                                 ).drop(columns=["_blk","_depm","_com"])

    shl_trips=shl_trips[shl_trips["Pulse Code"].astype(str).isin(ids)].copy()
    shl_trips["Pulse Code"]=shl_trips["Pulse Code"].astype(str)
    shl_trips=shl_trips.sort_values(["round_id","Delivery Sequence"])

    _n_rounds_rebuilt=shl_trips["round_id"].nunique()
    _n_tripnames=shl_trips["_orig_trip_name"].astype(str).replace("nan",pd.NA).dropna().nunique()
    if _n_rounds_rebuilt > _n_tripnames + shl_trips["_orig_trip_name"].isna().sum():
        print(f"  v10: reconstructed {_n_rounds_rebuilt} rounds from Delivery-Sequence resets "
              f"(the ffill-merged Trip Names would have given fewer — baseline mileage corrected).")

    routes_by_day={t:[] for t in DAYS}; all_trips=[]

    def _iter_rounds(_trips):
  #  one round per reconstructed round_id; label it with its original Trip Name
  # when it had one, else a synthesized single-drop label.
        for _rid_val,_g in _trips.groupby("round_id"):
            _nm=_g["_orig_trip_name"].dropna().astype(str)
            _nm=_nm.iloc[0] if len(_nm) and _nm.iloc[0] not in ("","nan","None") else f"{shl_name}-R{int(_rid_val):03d}"
            yield _nm,_g

    for round_name,grp in _iter_rounds(shl_trips):
        seq=[pc for pc in grp["Pulse Code"].tolist() if pc in hlat]
        if not seq: continue
        nodes=[None]+seq+[None]; mi=0.0
        for a,b in zip(nodes[:-1],nodes[1:]):
            ni=0 if a is None else lnode_idx.get((a,0))
            nj=0 if b is None else lnode_idx.get((b,0))
            if ni is not None and nj is not None and (ni,nj) in c:
                mi+=c[(ni,nj)]
            else:
                la1=dlat if a is None else hlat.get(a,dlat); lo1=dlon if a is None else hlon.get(a,dlon)
                la2=dlat if b is None else hlat.get(b,dlat); lo2=dlon if b is None else hlon.get(b,dlon)
                mi+=road_mi(la1,lo1,la2,lo2)
        dep_min=grp["depart_min"].dropna().min(); last_arr=grp["arrival_min"].dropna().max()
        dur_h=(last_arr-dep_min)/60.0 if (pd.notna(dep_min) and pd.notna(last_arr)) else None
        route_depart_h=round(dep_min/60.0,2) if pd.notna(dep_min) else None
        dp_times_slot=data.get("dp_times_slot",{})
        stops=[]
        for _,row in grp.iterrows():
            pc=str(row["Pulse Code"])
            if pc not in hlat: continue
            arr_min=row.get("arrival_min")
            slot_name=None
            if route_depart_h is not None:
                best=None; best_diff=999.0
                for (pc_q,day_q,sl_q),dp_q in dp_times_slot.items():
                    if pc_q!=pc: continue
                    diff=abs(dp_q-route_depart_h)
                    if diff<best_diff: best_diff=diff; best=sl_q
                if best is not None and best_diff<0.5:  # tolerance: 30 min
                    # v18 FIX: convert integer slot index to the slot name string
                    # so it matches co_times_slot / co_times_large_slot keys
                    _sw_pc = data.get("slot_windows", {}).get(pc, {})
                    _converted = False
                    for _d_chk in DAYS:
                        _sw_d = _sw_pc.get(_d_chk, [])
                        if best < len(_sw_d):
                            slot_name = _sw_d[best][0]   # (slot_name_str, oh, ch)
                            _converted = True
                            break
                    if not _converted:
                        slot_name = best  # fallback — keep integer (display will handle)
            stops.append(dict(pc=pc, slot=slot_name,
                              arrival_h=round(arr_min/60.0,2) if pd.notna(arr_min) else None,
                              depart_h=None))
        baseline_vtype="Diesel Van"
        route=dict(vehicle=round_name,vtype=baseline_vtype,hospitals=seq,km=round(mi,1),
                   dur_h=round(dur_h,2) if dur_h else None,stops=stops,
                   route_depart_h=route_depart_h,
                   co2_kg=round(mi*VEHICLE_TYPES[baseline_vtype]["co2_kg_per_mile"],2))
        # ── v19: day replication driven by DERIVED slot-days ──
        # `derived_days` (round level) and `stop_days` (per stop) come from the
        # v11 data-processing derivation: each stop was matched to its master-
        # schedule slot and inherits that slot's days. The round runs on the
        # UNION of its stops' days; on each day it includes only the stops
        # whose slot exists that day (mileage recomputed per day variant).
        _LETTER2DAY = {"M":1,"T":2,"W":3,"H":4,"F":5}
        def _code_to_days(code):
            ds = sorted({_LETTER2DAY[ch] for ch in str(code).strip().upper()
                         if ch in _LETTER2DAY})
            return ds if ds else list(DAYS)

        if "derived_days" in grp.columns and grp["derived_days"].notna().any():
            round_days = _code_to_days(grp["derived_days"].dropna().iloc[0])
            stop_days_of = {}
            for _, _r in grp.iterrows():
                _pc = str(_r["Pulse Code"])
                _sd = _r.get("stop_days")
                stop_days_of[_pc] = set(_code_to_days(_sd)) if pd.notna(_sd) else set(DAYS)
        else:
            # Fallback: legacy Delivery Day column (pre-v11 trips_clean)
            _raw_day = grp["Delivery Day"].dropna() if "Delivery Day" in grp.columns else []
            _dds = str(list(_raw_day)[0]).strip().lower() if len(_raw_day) > 0 else "weekday"
            _single = {"mon":1,"tue":2,"wed":3,"thu":4,"fri":5}.get(_dds[:3])
            round_days = [_single] if _single else list(DAYS)
            stop_days_of = {str(_r["Pulse Code"]): set(DAYS) for _, _r in grp.iterrows()}

        for d in round_days:
            # Stops present on day d
            seq_d = [pc for pc in seq if d in stop_days_of.get(pc, set(DAYS))]
            if not seq_d:
                continue
            stops_d = [st for st in stops if d in stop_days_of.get(st["pc"], set(DAYS))]
            if seq_d == seq:
                mi_d = mi                       # unchanged composition
            else:
                # Recompute mileage for the reduced stop set
                nodes_d = [None] + seq_d + [None]; mi_d = 0.0
                for a, b in zip(nodes_d[:-1], nodes_d[1:]):
                    ni = 0 if a is None else lnode_idx.get((a, 0))
                    nj = 0 if b is None else lnode_idx.get((b, 0))
                    if ni is not None and nj is not None and (ni, nj) in c:
                        mi_d += c[(ni, nj)]
                    else:
                        la1 = dlat if a is None else hlat.get(a, dlat)
                        lo1 = dlon if a is None else hlon.get(a, dlon)
                        la2 = dlat if b is None else hlat.get(b, dlat)
                        lo2 = dlon if b is None else hlon.get(b, dlon)
                        mi_d += road_mi(la1, lo1, la2, lo2)
            route_copy = dict(route)
            route_copy["_day"] = d
            route_copy["hospitals"] = seq_d
            route_copy["stops"] = stops_d
            route_copy["km"] = round(mi_d, 1)
            route_copy["co2_kg"] = round(
                mi_d * VEHICLE_TYPES[baseline_vtype]["co2_kg_per_mile"], 2)
            all_trips.append(route_copy)
            routes_by_day[d].append(route_copy)
    
    n_routes=len(all_trips); n_multi=sum(1 for r in all_trips if len(r["hospitals"])>1)
    total_mi=round(sum(r["km"] for r in all_trips),1)
    mean_stops=round(sum(len(r["hospitals"]) for r in all_trips)/max(n_routes,1),2)
    covered=set(pc for r in all_trips for pc in r["hospitals"])
    print(f"\n  Baseline ({shl_name}): {n_routes} rounds  {total_mi} mi  "
          f"mean {mean_stops} stops  coverage {len(covered)}/{len(ids)}")

    print(f"\n  {'Round':<12}  {'Day':<8}  {'Stops':>5}  {'Miles':>7}  {'Hospitals'}")
    print(f"  {'─'*70}")
    for r in all_trips:
        day_str="Weekday" if r["_day"]==0 else DAY_NAMES.get(r["_day"],"?")
        hosp_str=", ".join(r["hospitals"])
        print(f"  {r['vehicle']:<12}  {day_str:<8}  {len(r['hospitals']):>5}  {r['km']:>7.1f}  {hosp_str}")
    print(f"  {'─'*70}")
    print(f"  {'TOTAL':<12}  {'':8}  {'':>5}  {total_mi:>7.1f}")

    _print_schedule_table(f"{shl_name} — Baseline schedule", all_trips, data)
    for i,route in enumerate(all_trips): route["_color_idx"]=i
    metrics=dict(n_routes=n_routes,n_multi_stop=n_multi,mean_stops=mean_stops,
                 total_km=total_mi,covered=covered,all_trips=all_trips)
    return routes_by_day, metrics


def impute_baseline_vehicle(route_hospitals, t_day, data, prefer="smallest"):
    """Assign the historical round the vehicle the optimiser would have chosen:
    the smallest-capacity type that can legally carry the load. Mirrors the
    `initial_pool` rule in solve(), so baseline and optimised are costed on the
    same convention rather than on an all-Diesel-Van assumption."""
    demand = data.get("demand", {})
    order = sorted(VEHICLE_TYPES,
                   key=lambda v: VEHICLE_TYPES[v]["boxes_capacity"],
                   reverse=(prefer != "smallest"))
    prods = {g for pc in route_hospitals for g in PRODUCTS
             if demand.get((pc, g, t_day), 0.0) > 0}
    for vt in order:
        spec = VEHICLE_TYPES[vt]
        ok = True
        total = 0.0
        for g in PRODUCTS:
            gd = sum(demand.get((pc, g, t_day), 0.0) for pc in route_hospitals)
            if gd > BOX_CAPACITY_PER_PRODUCT[g] * spec["boxes_capacity"]:
                ok = False
                break
            total += gd
        if not ok or total > BOX_CAPACITY_AGGREGATE * spec["boxes_capacity"]:
            continue
        if container_ceiling_for_products(vt, prods) is None:
            continue
        return vt
    return None
 
 
def replay_baseline_like_for_like(shl_name, data, tau, c, lnodes, all_trips,
                                  fixed_vtype="Diesel Van", verbose=True):
    """Score every historical round against the optimiser's own constraint set.
 
    Returns a DataFrame with one row per baseline round: imputed vehicle,
    container ceiling, realised duration, and which constraints it violates.
    A baseline that fails R10 under the corrected ceiling is not a valid
    comparator, and the Delta% against it is not a saving.
    """
    lnode_idx = {(pc, s): i for i, (pc, s) in enumerate(lnodes, start=1)}
    demand = data.get("demand", {})
    rows = []
 
    for r in all_trips:
        t = r.get("_day")
        hosp = r.get("hospitals", [])
        if not hosp:
            continue
        vt = fixed_vtype or impute_baseline_vehicle(hosp, t, data)
        prods = {g for pc in hosp for g in PRODUCTS
                 if demand.get((pc, g, t), 0.0) > 0}
        ceil_h = container_ceiling_for_products(vt, prods) if vt else None
 
        # realised duration from the recorded stops, plus the return leg
        stops = r.get("stops", [])
        dep = r.get("route_depart_h")
        if dep is None and stops:
            dep = min(s.get("depart_h", s.get("arrival_h", 0.0)) for s in stops)
        last_arr = max((s.get("arrival_h", 0.0) for s in stops), default=None)
        ret = 0.0
        if hosp:
            ni = lnode_idx.get((hosp[-1], 0))
            if ni is not None:
                ret = tau.get((ni, 0), 0.0)
        dur = (last_arr + ret - dep) if (dep is not None and last_arr is not None) else None
 
        rows.append(dict(
            SHL=shl_name, Day=t, Round=r.get("label", r.get("Round")),
            n_stops=len(hosp), miles=r.get("km"),
            imputed_vehicle=vt,
            container=VEHICLE_VAQTEC_SIZES.get(vt) if vt else None,
            products=";".join(sorted(prods)),
            container_ceiling_h=ceil_h,
            duration_h=None if dur is None else round(dur, 2),
            R10_violation=(None if (dur is None or ceil_h is None)
                           else bool(dur > ceil_h + 1e-9)),
            R11_incompatible=bool(vt is None or ceil_h is None),
            true_margin_h=(None if (dur is None or ceil_h is None)
                           else round(ceil_h - dur, 2)),
            co2_kg=(None if r.get("km") is None or vt is None else
                    round(r["km"] * VEHICLE_TYPES[vt]["co2_kg_per_mile"], 2)),
        ))
 
    df = pd.DataFrame(rows)
    if verbose and len(df):
        n = len(df)
        nviol = int(df["R10_violation"].fillna(False).sum())
        nincomp = int(df["R11_incompatible"].sum())
        print(f"\n  BASELINE REPLAY — {shl_name}")
        print(f"    rounds {n} | R10 violations {nviol} ({100*nviol/n:.1f}%) "
              f"| R11 incompatible {nincomp}")
        print(f"    worst true margin {df['true_margin_h'].min()} h")
        print(f"    imputed fleet: "
              f"{df['imputed_vehicle'].value_counts().to_dict()}")
        print(f"    baseline CO2 on imputed fleet: {df['co2_kg'].sum():.1f} kg "
              f"(all-Diesel-Van: "
              f"{df['miles'].sum() * VEHICLE_TYPES['Diesel Van']['co2_kg_per_mile']:.1f} kg)")
        if nviol:
            print(f"    >>> the historical schedule is INFEASIBLE on {nviol} rounds "
                  f"under the corrected ceiling. Delta% against it is not a saving.")
    return df
 

## Section A — Module-level constants

In [10]:
# SECTION A -- Module-level constants (add to your config cell)
# =============================================================================

# ── §6 Improved Clarke-Wright (Pichpibul & Kawtummachai 2012) ────────────────
ICW_MAX_ITER        = 10000     # outer iteration cap (paper uses 10000; groups small)
ICW_STALL_LIMIT     = 200      # early termination after this many no-improvement iters
ICW_TOURNAMENT_MAX  = 9        # T_max in tournament + roulette selection
ICW_PENALTY         = 999_999.0  # infeasibility penalty (fleet overflow)
ICW_RANDOM_SEED     = 42       # reproducibility

# ── §7 Or-opt segment relocation ─────────────────────────────────────────────
OROPT_MAX_SEG_LEN   = 3        # move segments of length 2, 3
LS_OUTER_ITER       = 30       # LS 2-opt + relocate + Or-opt outer loop cap

# ── §9 Column generation with DP pricing (Choi & Tcha 2007) ──────────────────
CG_MAX_ITER         = 50       # column-generation outer loop cap
CG_MULTI_COL_MAX    = 15       # max columns added per pricing round per vehicle type
CG_REDUCED_COST_EPS = -1e-4    # tolerance for "negative" reduced cost
CG_2CYCLE_ELIM      = True     # enable 2-cycle elimination (§9.5)
CG_MAX_ROUTE_LEN    = 12       # depth cap in DP (dispatch-group routes rarely exceed 6)
CG_GAP_TOLERANCE    = 1.00      # % — accept restricted-master int solution below this gap

# ── §5 Vehicle-type preprocessing (Choi & Tcha §5.1) ─────────────────────────
PP_ENABLE           = True     # toggle preprocessing on/off (debug switch)


## Section B — Replacement `solve()` function (Round 4c: flush=True + entry markers)

Two robustness changes vs Round 4b:

1. **All diagnostic prints use `print(..., flush=True)`** instead of `sys.stdout.flush()`. This works even if `import sys` hasn't been re-run in the kernel, and eliminates any chance of a silent `NameError` on `sys`.

2. **Two unconditional entry markers** at the very top of `solve()` and immediately before the main-loop header:
   - `── SOLVE ENTRY (SHL=…, month=…) ──` — fires as the first statement of `solve()`. If you don't see this, the kernel is still running an older `solve()` definition and you need to re-run cell 12.
   - `── MAIN LOOP ENTRY ──` — fires just before `═══ v27 METHODOLOGY-ALIGNED SOLVE ═══`. If you see this but not the `═══` line, one of the constants (`PP_ENABLE`, `ICW_MAX_ITER`, `ICW_STALL_LIMIT`, `CG_MAX_ITER`, `GROUP_MILP_THRESHOLD`) is undefined, and cell 10 (Section A constants) needs to be re-run.

Everything else — Stage 5 parallel v27/v26 warm-start MILP, Fixes 1/2/3 from Round 3 — is unchanged from Round 4b.

All formulation constraints (R1–R11, T1–T6, D1–D4) preserved exactly.



In [11]:
# =============================================================================
# SECTION B -- Replacement solve() function (with K-Pool & Duplicate Fixes)
# =============================================================================

import sys
import random  # local import for ICW; deterministic via ICW_RANDOM_SEED
_SIMULATE_TOL = 1.0 / 3600.0 

_log = open('/tmp/solve_log.txt', 'w')
_orig_stdout = sys.stdout
sys.stdout = _log
try:
    def solve(data, c, tau, lnode_idx, lnodes, N):
        """v27.3: methodology-aligned decomposed solve with dynamic k-pool and CG duplicate cleaning.
        """
        print(f"  ── SOLVE ENTRY (SHL={data.get('shl_name','?')}, "
              f"month={data.get('month','?')}) ──", flush=True)
        # ═════════════════════════════════════════════════════════════════════════
        # PART 1 — Setup, fleet, node index
        # ═════════════════════════════════════════════════════════════════════════
        ids = data["ids"]; slot_windows = data["slot_windows"]
        n_required_visits = data["n_required_visits"]; visit_days = data["visit_days"]
        T_DEP = data["T_DEP"]; T_MAX = data["T_MAX"]
        BIG_M_TIME = T_MAX + (max(tau.values()) if tau else 0.0)
        dp_times_slot_lu = data.get("dp_times_slot", {}); dp_times_lu = data.get("dp_times", {})
        demand = data.get("demand", {})
        real_boxes_by_hospital = data.get("real_boxes_by_hospital", {})
    
        C_baseline = data["C_baseline"]
        busiest_day_slots = max(sum(n_required_visits.get((pc, t), 0) for pc in ids) for t in DAYS)
    
        # ── Fleet sizing ─────────────────────────────────────────
        dp_times_slot_lu0 = data.get("dp_times_slot", {}); dp_times_lu0 = data.get("dp_times", {})
        def _dp_of(pc, s, t):
            return dp_times_slot_lu0.get((pc, t, s),
                   dp_times_lu0.get((pc, t), data["T_DEP"]))
        groups_by_day = []
        for t in DAYS:
            dps = set()
            for pc in ids:
                for s in range(n_required_visits.get((pc, t), 0)):
                    dps.add(round(_dp_of(pc, s, t), 4))
            groups_by_day.append(len(dps))
        n_groups_busiest = max(groups_by_day) if groups_by_day else 1
        _shl_for_size = data.get("shl_name", "")
        _phys = FLEET_FROM_DATA.get(_shl_for_size, C_baseline)
        _fleet_capacity = _phys * max(n_groups_busiest, 1)
        _auto = max(C_baseline, n_groups_busiest)
        Cd = FLEET_SIZE_OVERRIDE.get(_shl_for_size) or min(_fleet_capacity, busiest_day_slots)
        Cd = max(Cd, _auto)
        K = list(range(Cd))
        print(f"  Fleet sizing: physical_fleet={_phys}, capacity={_fleet_capacity}, "
              f"dispatch-groups={n_groups_busiest}, C_baseline={C_baseline}, "
              f"busiest-day visits={busiest_day_slots} -> Cd={Cd}"
              f"{' (OVERRIDE)' if FLEET_SIZE_OVERRIDE.get(_shl_for_size) else ''}")
    
        shl_name = data.get("shl_name", "")
        fleet_comp = dict(FLEET_COMPOSITION_OVERRIDE.get(shl_name) or default_fleet_composition(Cd))
        fc_total = sum(fleet_comp.values())
        if fc_total < Cd:
            _fill_type = next(iter(VEHICLE_TYPES))  # first available type
            fleet_comp[_fill_type] = fleet_comp.get(_fill_type, 0) + (Cd - fc_total)
        elif fc_total > Cd:
            excess = fc_total - Cd
            for tname in list(fleet_comp.keys()):
                take = min(excess, fleet_comp.get(tname, 0))
                fleet_comp[tname] = fleet_comp.get(tname, 0) - take
                excess -= take
                if excess <= 0: break
    
        vtype_of = {}; _k = 0
        for tname, cnt in fleet_comp.items():
            for _ in range(cnt):
                if _k >= Cd: break
                vtype_of[_k] = tname; _k += 1
        for kk in K:
            if kk not in vtype_of: vtype_of[kk] = next(iter(VEHICLE_TYPES))
    
        print(f"\n  Fleet composition (Cd={Cd}):")
        for tname, cnt in fleet_comp.items():
            spec = VEHICLE_TYPES[tname]
            rng = f", range<={spec['max_range_mi']:.0f}mi" if spec.get("max_range_mi") else ""
            print(f"    {tname:<22s}: {cnt:>2d} available  "
                  f"(boxes_capacity={spec['boxes_capacity']}, "
                  f"CO2={spec['co2_kg_per_mile']:.3f} kg/mi{rng})")
    
        all_nodes = list(range(N + 1))
    
        # Required-node table
        required_node = {}
        for i in range(1, N + 1):
            pc, s = lnodes[i-1]
            for t in DAYS:
                nslots = n_required_visits.get((pc, t), 0)
                required_node[(i, t)] = (t in visit_days[pc]) and (s < nslots)
        req_nodes_day = {t: [i for i in range(1, N+1) if required_node.get((i, t), False)] for t in DAYS}
        req_set_day = {t: set(rn) for t, rn in req_nodes_day.items()}
    
        _i_of = {(lnodes[i-1][0], lnodes[i-1][1]): i for i in range(1, N+1)}

        tw = {}; dp_node = {}
        for i in range(1, N+1):
            pc, s = lnodes[i-1]
            for t in DAYS:
                slots = slot_windows[pc].get(t, [])
                tw[(i, t)] = (slots[s][1], slots[s][2]) if s < len(slots) else None
                dp_node[(i, t)] = dp_times_slot_lu.get((pc, t, s),
                                                        dp_times_lu.get((pc, t), T_DEP))
    
        s_node = {}; boxes_est_node = {}
        for i in range(1, N+1):
            pc, _s = lnodes[i-1]
            for t in DAYS:
                total_units = sum(demand.get((pc, g, t), 0.0) for g in PRODUCTS)
                boxes_est = max(1, math.ceil(total_units / BOX_CAPACITY_AGGREGATE)) if total_units > 0 else 1
                boxes_est_node[(i, t)] = boxes_est
                s_node[(i, t)] = (SERVICE_BASE_MIN + SERVICE_PER_BOX_MIN * boxes_est) / 60.0
    
        ch_adj = {}; infeasible_tw = []; minor_relaxed_tw = []; drop_set = set()
        for i in range(1, N+1):
            pc, s = lnodes[i-1]
            for t in DAYS:
                if not required_node[(i, t)]: ch_adj[(i, t)] = None; continue
                oh_ch = tw[(i, t)]
                if oh_ch is None: ch_adj[(i, t)] = None; continue
                oh, ch = oh_ch; dp_h = dp_node[(i, t)]
                earliest = max(T_DEP, dp_h) + tau[(0, i)]
                ch_eff = min(ch, T_MAX)
                _feas_tol = 5.0 / 60.0
                if earliest > ch_eff + _feas_tol:
                    infeasible_tw.append((pc, t, s, ch_eff, earliest, (earliest - ch_eff) * 60))
                    if DROP_UNREACHABLE:
                        ch_adj[(i, t)] = None
                        drop_set.add((i, t))
                    else:
                        ch_adj[(i, t)] = earliest + ARRIVAL_WINDOW_SLACK_H
                elif earliest > ch_eff:
                    minor_relaxed_tw.append((pc, t, s, ch_eff, earliest,
                                             (earliest - ch_eff) * 60))
                    ch_adj[(i, t)] = round(earliest + 0.01, 4)
                else:
                    ch_adj[(i, t)] = ch_eff

        relaxed_nodes = {}
        for _pc, _t, _s, _ch_eff, _ea, _gap_min in (infeasible_tw + minor_relaxed_tw):
            _i = _i_of.get((_pc, _s))
            _tw = tw.get((_i, _t)) if _i is not None else None
            relaxed_nodes[(_pc, _t, _s)] = dict(
                stated_open_h  = round(_tw[0], 4) if _tw else None,
                stated_close_h = round(_ch_eff, 4),
                earliest_h     = round(_ea, 4),
                gap_min        = round(_gap_min, 2),
                beyond_slack   = bool(_gap_min > ARRIVAL_WINDOW_SLACK_H * 60.0),
                zero_width     = bool(_tw is not None and abs(_tw[1] - _tw[0]) < 1e-9),
            )
        if relaxed_nodes:
            _nzw = sum(1 for v in relaxed_nodes.values() if v["zero_width"])
            _nbs = sum(1 for v in relaxed_nodes.values() if v["beyond_slack"])
            print(f"  Relaxed-window register: {len(relaxed_nodes)} node-day(s) "
                  f"({_nzw} zero-width, {_nbs} beyond the "
                  f"{ARRIVAL_WINDOW_SLACK_H*60:.0f}-min slack).")

        if drop_set:
            print(f"  ⚠ DROP_UNREACHABLE: {len(drop_set)} node-day(s) unreachable within "
                  f"the shift horizon are DROPPED from coverage:")
            for (i, t) in sorted(drop_set):
                pc, s = lnodes[i-1]
                print(f"      {pc} s={s} day={DAY_NAMES.get(t, t)}")
            for (i, t) in drop_set: required_node[(i, t)] = False
            req_nodes_day = {t: [i for i in range(1, N+1) if required_node.get((i, t), False)] for t in DAYS}
            req_set_day = {t: set(rn) for t, rn in req_nodes_day.items()}
    
        _dp_vals_all = sorted(set(
            round(dp_node.get((i, t), T_DEP), 4)
            for t in DAYS for i in range(1, N+1) if required_node.get((i, t), False)
        ))
        _dp_groups = []
        for dv in _dp_vals_all:
            placed = False
            for grp in _dp_groups:
                if abs(dv - grp[-1]) <= 0.5:
                    grp.append(dv); placed = True; break
            if not placed:
                _dp_groups.append([dv])
        _max_intra_group_gap = max((max(g) - min(g) for g in _dp_groups if len(g) > 1), default=0.0)
        MAX_WAIT_H = max(MAX_WAIT_H_DEFAULT, _max_intra_group_gap + 1.0)
        MAX_WAIT_H = round(min(MAX_WAIT_H, 4.0), 2)
        if abs(MAX_WAIT_H - MAX_WAIT_H_DEFAULT) > 0.01:
            print(f"  v19 MAX_WAIT_H: {MAX_WAIT_H_DEFAULT:.1f}h (default) -> {MAX_WAIT_H:.2f}h "
                  f"(SHL-adaptive; max intra-group gap={_max_intra_group_gap:.2f}h)")
    
        _demanded_products = set()
        for (pc_d, g_d, t_d), qty in demand.items():
            if qty > 0: _demanded_products.add(g_d)
        U = min((PERISH_HOURS[g] for g in _demanded_products if g in PERISH_HOURS),
                default=U_DEFAULT)
        if abs(U - U_DEFAULT) > 0.01:
            print(f"  v19 shift ceiling U: {U_DEFAULT:.1f}h (default) -> {U:.1f}h "
                  f"(tightest perishability for demanded products: {sorted(_demanded_products)})")
    
        print(f"\n  Fleet: C_baseline={C_baseline}, C_used={Cd}  "
              f"(busiest day={busiest_day_slots} slot-visits)")
        print(f"  T_DEP={T_DEP:.3f}h  T_MAX={T_MAX:.3f}h  BIG_M={BIG_M_TIME:.3f}h  "
              f"MAX_WAIT_H={MAX_WAIT_H:.2f}h")
        if infeasible_tw:
            _slack_m = ARRIVAL_WINDOW_SLACK_H * 60
            _big = [r for r in infeasible_tw if r[5] > _slack_m]
            print(f"  ⚠ {len(infeasible_tw)} node(s) arrive after observed window; ceiling relaxed. "
                  f"{len(_big)} exceed the {_slack_m:.0f}-min slack:")
            for pc, t, s, ch, ea, gm in infeasible_tw:
                tag = "   <-- beyond slack" if gm > _slack_m else ""
                print(f"    {pc} s={s} day={t}  AR={ch:.2f}h  earliest={ea:.2f}h  gap={gm:.1f}m{tag}")
        else:
            print("  ✓ All logical nodes reachable within model travel times.")
    
        _original_dps_by_day = {}
        for t in DAYS:
            vals = set()
            for i in range(1, N+1):
                if not required_node.get((i, t), False): continue
                pc, s = lnodes[i-1]
                orig_dp = dp_times_slot_lu.get((pc, t, s), dp_times_lu.get((pc, t), T_DEP))
                vals.add(round(orig_dp, 6))
            _original_dps_by_day[t] = sorted(vals) if vals else [T_DEP]
        _n_resnapped = 0
        for i in range(1, N+1):
            for t in DAYS:
                if not required_node.get((i, t), False): continue
                dp_cur = dp_node[(i, t)]
                best_g = min(_original_dps_by_day[t], key=lambda g: abs(g - dp_cur))
                if abs(best_g - dp_cur) > 1e-9 and abs(best_g - dp_cur) <= ABSORB_MAX_H + 1e-9:
                    dp_node[(i, t)] = best_g
                    _n_resnapped += 1
        if _n_resnapped:
            print(f"  dp re-snap: {_n_resnapped} absorbed node-day(s) snapped back to original group.")
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 2 — SHARED HELPERS
        # ═════════════════════════════════════════════════════════════════════════
    
        def _dp_of_node(nd, t_day):
            return dp_node.get((nd, t_day), T_DEP)
    
        def _dp_sorted(route, t_day):
            return sorted(route, key=lambda nd: (_dp_of_node(nd, t_day)))
        
        def _route_perishability(route, t_day, vtype_name=None):
            prods = products_on_route(route, t_day, demand, lnodes)
            if not prods:
                return max(PERISH_HOURS.values()) if USE_PRODUCT_PERISHABILITY else U
            if vtype_name is None:
                cands = [container_ceiling_for_products(vt, prods)
                         for vt in VEHICLE_VAQTEC_SIZES]
                cands = [x for x in cands if x is not None]
                return max(cands) if cands else min(PERISH_HOURS[g] for g in prods)
            ceil_h = container_ceiling_for_products(vtype_name, prods)
            return 0.0 if ceil_h is None else ceil_h
    
        def _simulate(route, t_day, vtype_name=None):
            if not route: return False, None
            first = route[0]
            D = max(T_DEP, max(_dp_of_node(nd, t_day) for nd in route))
            clock = D + tau.get((0, first), 0.0)
            details = []
            travel_svc_wait = tau.get((0, first), 0.0)
            for pos, nd in enumerate(route):
                A_raw = clock
                need = _dp_of_node(nd, t_day)
                tw_nd = tw.get((nd, t_day))
                if tw_nd is not None: need = max(need, tw_nd[0])
                wait = max(0.0, need - A_raw)
                if wait > MAX_WAIT_H + _SIMULATE_TOL: return False, None
                eff = A_raw + wait
                ch = ch_adj.get((nd, t_day))
                if ch is not None and eff > ch + _SIMULATE_TOL: return False, None
                svc = s_node.get((nd, t_day), 0.0)
                dep = eff + svc
                if dep > T_MAX + _SIMULATE_TOL: return False, None
                details.append((nd, A_raw, wait, dep))
                travel_svc_wait += wait + svc
                if pos < len(route) - 1:
                    leg = tau.get((nd, route[pos+1]), 0.0)
                    clock = dep + leg
                    travel_svc_wait += leg
            travel_svc_wait += tau.get((route[-1], 0), 0.0)
            if travel_svc_wait > _route_perishability(route, t_day, vtype_name) + 1e-9:
                return False, None
            return True, details
    
        def _route_miles(route):
            if not route: return 0.0
            m = c.get((0, route[0]), 0.0)
            for a, b in zip(route[:-1], route[1:]):
                m += c.get((a, b), 0.0)
            m += c.get((route[-1], 0), 0.0)
            return m
    
        def _fits_vehicle(route, t_day, k_cand):
            spec = VEHICLE_TYPES[vtype_of[k_cand]]
            Qagg_k = BOX_CAPACITY_AGGREGATE * spec["boxes_capacity"]
            total_d = 0.0
            for g in PRODUCTS:
                gd = sum(demand.get((lnodes[nd-1][0], g, t_day), 0.0) for nd in route)
                if gd > BOX_CAPACITY_PER_PRODUCT[g] * spec["boxes_capacity"]: return False
                total_d += gd
            if total_d > Qagg_k: return False
            rng = spec.get("max_range_mi")
            if rng is not None and _route_miles(route) > rng: return False
            
            vtype_name = vtype_of[k_cand]
            allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype_name, [])
            
            if allowed_sizes:
                for g in PRODUCTS:
                    gd = sum(demand.get((lnodes[nd-1][0], g, t_day), 0.0) for nd in route)
                    if gd > 0:
                        can_carry = any(VAQTEC_SPEC.get(sz, {}).get(g) is not None for sz in allowed_sizes)
                        if not can_carry: return False
                        
            if container_ceiling_for_products(vtype_name, products_on_route(route, t_day, demand, lnodes)) is None:
                return False
            _ok_t, _ = _simulate(route, t_day, vtype_name)
            if not _ok_t: return False
            return True
    
        def _fits_any_vehicle(route, t_day):
            return any(_fits_vehicle(route, t_day, kk) for kk in K)
    
        def _fits_any_of_types(route, t_day, vtype_names):
            for kk in K:
                if vtype_of[kk] in vtype_names and _fits_vehicle(route, t_day, kk):
                    return True
            return False
    
        def _representative_vehicle_of_type(vtype_name):
            for kk in K:
                if vtype_of[kk] == vtype_name: return kk
            return None
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 3 — DETERMINISTIC CW
        # ═════════════════════════════════════════════════════════════════════════
    
        def cw_warmstart_day(req_nodes, t_day, vtype_names=None):
            _fits_check = (_fits_any_vehicle
                           if vtype_names is None
                           else (lambda r, td: _fits_any_of_types(r, td, vtype_names)))
    
            routes = {i: [i] for i in req_nodes}
            merged_into = {i: i for i in req_nodes}
            pairs = sorted(
                [(c[(0, ni)] + c[(0, nj)] - c[(ni, nj)], ni, nj)
                 for a, ni in enumerate(req_nodes) for nj in req_nodes[a+1:]],
                reverse=True)
    
            for s_val, ni, nj in pairs:
                if s_val <= 0: break
                ri = merged_into[ni]; rj = merged_into[nj]
                if ri == rj: continue
                ri_r = routes[ri]; rj_r = routes[rj]
                if ri_r[-1] != ni or rj_r[0] != nj: continue
                merged = _dp_sorted(ri_r + rj_r, t_day)
                if not _fits_check(merged, t_day): continue
                ok, _ = _simulate(merged, t_day)
                if not ok: continue
                routes[ri] = merged; del routes[rj]
                for nd in rj_r: merged_into[nd] = ri
    
            result = sorted(routes.values(), key=len, reverse=True)
    
            if len(result) > len(K):
                guard = 0
                while len(result) > len(K) and guard < 200:
                    guard += 1
                    donor = result.pop()
                    unplaced = []
                    for nd in donor:
                        placed = False
                        for idx_r in range(len(result)):
                            host = result[idx_r]
                            cand = _dp_sorted(host + [nd], t_day)
                            if not _fits_check(cand, t_day): continue
                            ok, _ = _simulate(cand, t_day)
                            if ok:
                                result[idx_r] = cand; placed = True; break
                        if not placed: unplaced.append(nd)
                    if unplaced:
                        result.append(_dp_sorted(unplaced, t_day))
                        if len(unplaced) == len(donor): break
                    result.sort(key=len, reverse=True)
    
            covered_by_cw = set(nd for route in result for nd in route)
            missing_cw = set(req_nodes) - covered_by_cw
            if missing_cw:
                for nd in sorted(missing_cw):
                    pc_miss = lnodes[nd-1][0]
                    print(f"      ⚠ CW coverage fix: node {nd} ({pc_miss}) was uncovered — "
                          f"adding single-stop backup route")
                    result.append([nd])
            return result
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 4 — §5 VEHICLE-TYPE PREPROCESSING
        # ═════════════════════════════════════════════════════════════════════════
    
        def _mileage_lower_bound(g_nodes, t_day):
            if not g_nodes: return 0.0
            nodes = [0] + list(g_nodes)
            lb = 0.0
            for i in g_nodes:
                in_arcs = [c.get((j, i), float('inf')) for j in nodes if j != i]
                out_arcs = [c.get((i, k), float('inf')) for k in nodes if k != i]
                lb += 0.5 * (min(in_arcs) + min(out_arcs))
            return lb
    
        def _column_cost(route, vtype_name):
            spec_v = VEHICLE_TYPES[vtype_name]
            alpha_v = spec_v["cost_per_mile"]
            eps_v   = spec_v["co2_kg_per_mile"]
            return ROUTE_PENALTY + (alpha_v + CARBON_PRICE_PER_KG * eps_v) * _route_miles(route)
    
        def _preprocess_vehicle_types(g_nodes, t_day):
            V_g = set(vtype_of[kk] for kk in K)
            if not PP_ENABLE:
                return V_g
    
            V_after_r11 = set()
            demanded_g_in_group = set()
            for nd in g_nodes:
                pc_nd = lnodes[nd-1][0]
                for g in PRODUCTS:
                    if demand.get((pc_nd, g, t_day), 0.0) > 0:
                        demanded_g_in_group.add(g)
            
            # --- UPDATED ARRAY LOGIC ---
            for vt in V_g:
                allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vt, [])
                if not allowed_sizes:
                    V_after_r11.add(vt); continue
                
                # Check if EVERY demanded product can be carried by AT LEAST ONE of the allowed containers
                can_carry = True
                for g in demanded_g_in_group:
                    if not any(VAQTEC_SPEC.get(sz, {}).get(g) is not None for sz in allowed_sizes):
                        can_carry = False
                        break
                
                if can_carry: 
                    V_after_r11.add(vt)
            # ---------------------------
    
            V_after_cap = set()
            for vt in V_after_r11:
                spec = VEHICLE_TYPES[vt]
                boxes = spec["boxes_capacity"]
                fits_some = False
                for nd in g_nodes:
                    pc_nd = lnodes[nd-1][0]
                    node_ok = True
                    for g in PRODUCTS:
                        gd = demand.get((pc_nd, g, t_day), 0.0)
                        if gd > BOX_CAPACITY_PER_PRODUCT[g] * boxes:
                            node_ok = False; break
                    if node_ok:
                        fits_some = True; break
                if fits_some:
                    V_after_cap.add(vt)
    
            if len(V_after_cap) <= 1:
                return V_after_cap
    
            D_g = sum(demand.get((lnodes[nd-1][0], g, t_day), 0.0)
                      for nd in g_nodes for g in PRODUCTS)
            underline_c = _mileage_lower_bound(g_nodes, t_day)
    
            cw_baseline = cw_warmstart_day(g_nodes, t_day, vtype_names=V_after_cap)
            Zbar = 0.0
            for r in cw_baseline:
                best_c = float('inf')
                for vt in V_after_cap:
                    if _fits_any_of_types(r, t_day, {vt}):
                        cc = _column_cost(r, vt)
                        if cc < best_c: best_c = cc
                if best_c < float('inf'): Zbar += best_c
                else:                     
                    Zbar += _column_cost(r, next(iter(V_after_cap)))
    
            Q_of = {vt: BOX_CAPACITY_AGGREGATE * VEHICLE_TYPES[vt]["boxes_capacity"]
                    for vt in V_after_cap}
            cbar_of = {vt: ROUTE_PENALTY + underline_c * (VEHICLE_TYPES[vt]["cost_per_mile"] +
                                                         CARBON_PRICE_PER_KG *
                                                         VEHICLE_TYPES[vt]["co2_kg_per_mile"])
                       for vt in V_after_cap}
    
            Y_MAX = max(3, math.ceil(D_g / min(Q_of.values())) + 1) if D_g > 0 else 1
    
            V_g_prime = set(V_after_cap)
            types_list = sorted(V_after_cap)
    
            for v_star in list(V_after_cap):
                best_Z = float('inf')
                def _enum(idx, y_partial, cum_Q, cum_cost):
                    nonlocal best_Z
                    if cum_cost >= best_Z: return
                    if idx == len(types_list):
                        if cum_Q >= D_g and y_partial.get(v_star, 0) >= 1:
                            if cum_cost < best_Z: best_Z = cum_cost
                        return
                    vt = types_list[idx]
                    lo = 1 if vt == v_star else 0
                    for y in range(lo, Y_MAX + 1):
                        y_partial[vt] = y
                        _enum(idx + 1, y_partial, cum_Q + Q_of[vt] * y,
                              cum_cost + cbar_of[vt] * y)
                        y_partial[vt] = 0
                _enum(0, {}, 0.0, 0.0)
    
                if best_Z > Zbar + 1e-6:
                    V_g_prime.discard(v_star)
    
            if not V_g_prime:
                V_g_prime = V_after_cap
            return V_g_prime
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 5 — §6 IMPROVED CLARKE-WRIGHT
        # ═════════════════════════════════════════════════════════════════════════
    
        def _cost_of_route_set(routes):
            return sum(_route_miles(r) for r in routes) + ROUTE_PENALTY * len(routes)
    
        def _parallel_cw_given_savings(req_nodes, t_day, savings_list, vtype_names):
            _fits_check = lambda r, td: _fits_any_of_types(r, td, vtype_names)
            routes = {i: [i] for i in req_nodes}
            merged_into = {i: i for i in req_nodes}
    
            for s_val, ni, nj in savings_list:
                if s_val <= 0: break
                if ni not in merged_into or nj not in merged_into: continue
                ri = merged_into[ni]; rj = merged_into[nj]
                if ri == rj: continue
                ri_r = routes[ri]; rj_r = routes[rj]
                if ri_r[-1] != ni or rj_r[0] != nj: continue
                merged = _dp_sorted(ri_r + rj_r, t_day)
                if not _fits_check(merged, t_day): continue
                ok, _ = _simulate(merged, t_day)
                if not ok: continue
                routes[ri] = merged; del routes[rj]
                for nd in rj_r: merged_into[nd] = ri
            return sorted(routes.values(), key=len, reverse=True)
    
        def _reorder_icw(savings_list, rng):
            pos = [x for x in savings_list if x[0] > 0]
            neg = [x for x in savings_list if x[0] <= 0]
            L_prime = []
            pool = list(pos)
            while len(pool) > 1:
                T = rng.randint(2, min(ICW_TOURNAMENT_MAX, len(pool)))
                sample_idx = rng.sample(range(len(pool)), T)
                sample = [pool[i] for i in sample_idx]
                svals = [x[0] for x in sample]
                s_total = sum(svals)
                if s_total <= 0:
                    pick = 0
                else:
                    r = rng.random() * s_total
                    acc = 0.0; pick = 0
                    for n, sv in enumerate(svals):
                        acc += sv
                        if r <= acc:
                            pick = n; break
                chosen = sample[pick]
                L_prime.append(chosen)
                pool.remove(chosen)
            if pool: L_prime.append(pool[0])
            L_prime.extend(neg)
            return L_prime
    
        def _icw_construct(g_nodes, t_day, vtype_names):
            if not g_nodes:
                return [], set()
    
            raw_pairs = [(c[(0, ni)] + c[(0, nj)] - c[(ni, nj)], ni, nj)
                         for a, ni in enumerate(g_nodes) for nj in g_nodes[a+1:]]
            L0 = sorted(raw_pairs, reverse=True)
    
            best_routes = cw_warmstart_day(g_nodes, t_day, vtype_names=vtype_names)
            route_pool = set(tuple(r) for r in best_routes)
    
            best_cost = _cost_of_route_set(best_routes)
            if len(best_routes) > len(K):
                best_cost = ICW_PENALTY
    
            rng = random.Random(ICW_RANDOM_SEED + hash((tuple(g_nodes), t_day)) % (2**31))
            L_baseline = L0
    
            stall = 0
            for it in range(ICW_MAX_ITER):
                if stall >= ICW_STALL_LIMIT: break
                L_prime = _reorder_icw(L_baseline, rng)
                routes = _parallel_cw_given_savings(g_nodes, t_day, L_prime, vtype_names)
    
                for r in routes:
                    route_pool.add(tuple(r))
    
                cost = _cost_of_route_set(routes)
                if len(routes) > len(K):
                    cost = ICW_PENALTY
    
                if cost < best_cost - 1e-9:
                    best_routes = routes
                    best_cost = cost
                    L_baseline = L_prime
                    stall = 0
                else:
                    stall += 1
    
            covered = set(nd for r in best_routes for nd in r)
            missing = set(g_nodes) - covered
            if missing:
                for nd in sorted(missing):
                    pc_miss = lnodes[nd-1][0]
                    print(f"      ⚠ ICW coverage fix: node {nd} ({pc_miss}) uncovered — "
                          f"adding single-stop backup route")
                    best_routes.append([nd])
                    route_pool.add((nd,))
    
            return best_routes, route_pool
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 6 — §7 LOCAL SEARCH
        # ═════════════════════════════════════════════════════════════════════════
    
        def _two_opt(route, t_day):
            if len(route) <= 2: return route
            best = list(route)
            best_cost = _route_miles(best)
            improved = True
            max_iter = 50
            _iter = 0
            while improved and _iter < max_iter:
                improved = False
                _iter += 1
                for i in range(len(best) - 1):
                    for j in range(i + 1, len(best)):
                        candidate = best[:i] + best[i:j+1][::-1] + best[j+1:]
                        ok, _ = _simulate(candidate, t_day)
                        if not ok: continue
                        cost = _route_miles(candidate)
                        if cost < best_cost - 0.01:
                            best = candidate; best_cost = cost
                            improved = True; break
                    if improved: break
            return best
    
        def _or_opt(routes, t_day, vtype_names):
            _fits_check = lambda r, td: _fits_any_of_types(r, td, vtype_names)
            improved = True
            max_outer = 20; _outer = 0
            while improved and _outer < max_outer:
                improved = False; _outer += 1
                for lam in range(2, OROPT_MAX_SEG_LEN + 1):
                    for r1_idx in range(len(routes)):
                        r1 = routes[r1_idx]
                        if len(r1) < lam: continue
                        for start in range(len(r1) - lam + 1):
                            seg = r1[start:start+lam]
                            new_r1 = r1[:start] + r1[start+lam:]
                            if new_r1:
                                ok_r1, _ = _simulate(new_r1, t_day)
                                if not ok_r1: continue
                                if not _fits_check(new_r1, t_day): continue
    
                            cost_r1_old = _route_miles(r1)
                            cost_r1_new = _route_miles(new_r1) if new_r1 else 0.0
                            savings_r1 = cost_r1_old - cost_r1_new
    
                            best_net = -0.01
                            best_r2_idx = -1; best_r2_new = None
                            for r2_idx in range(len(routes)):
                                if r2_idx == r1_idx: continue
                                r2 = routes[r2_idx]
                                cand = _dp_sorted(r2 + seg, t_day)
                                if not _fits_check(cand, t_day): continue
                                ok, _ = _simulate(cand, t_day)
                                if not ok: continue
                                insert_cost = _route_miles(cand) - _route_miles(r2)
                                net = insert_cost - savings_r1
                                if net < best_net:
                                    best_net = net
                                    best_r2_idx = r2_idx
                                    best_r2_new = cand
                            if best_r2_idx >= 0 and best_r2_new is not None:
                                routes[r1_idx] = new_r1
                                routes[best_r2_idx] = best_r2_new
                                improved = True; break
                        if improved: break
                    if improved: break
                routes = [r for r in routes if r]
            return routes
    
        def _local_search(routes, t_day, vtype_names=None):
            if not routes: return routes
            if vtype_names is None: vtype_names = set(vtype_of[kk] for kk in K)
            _fits_check = lambda r, td: _fits_any_of_types(r, td, vtype_names)
    
            for r_idx in range(len(routes)):
                routes[r_idx] = _two_opt(routes[r_idx], t_day)
    
            improved = True
            max_outer = LS_OUTER_ITER; _outer = 0
            while improved and _outer < max_outer:
                improved = False; _outer += 1
                for r1_idx in range(len(routes)):
                    if not routes[r1_idx]: continue
                    for node_pos in range(len(routes[r1_idx])):
                        node = routes[r1_idx][node_pos]
                        old_r1 = routes[r1_idx]
                        new_r1 = [n for n in old_r1 if n != node]
                        old_r1_cost = _route_miles(old_r1)
                        new_r1_cost = _route_miles(new_r1) if new_r1 else 0.0
                        savings_r1 = old_r1_cost - new_r1_cost
    
                        best_insert_cost = 999999.0
                        best_r2_idx = -1; best_r2_new = None
                        for r2_idx in range(len(routes)):
                            if r2_idx == r1_idx: continue
                            old_r2 = routes[r2_idx]
                            cand = _dp_sorted(old_r2 + [node], t_day)
                            if not _fits_check(cand, t_day): continue
                            ok, _ = _simulate(cand, t_day)
                            if not ok: continue
                            insert_cost = _route_miles(cand) - _route_miles(old_r2)
                            net = insert_cost - savings_r1
                            if net < -0.01 and net < best_insert_cost:
                                best_insert_cost = net
                                best_r2_idx = r2_idx
                                best_r2_new = cand
    
                        if best_r2_idx >= 0 and best_r2_new is not None:
                            if new_r1:
                                ok_r1, _ = _simulate(new_r1, t_day)
                                if not ok_r1: continue
                            routes[r1_idx] = new_r1
                            routes[best_r2_idx] = best_r2_new
                            improved = True; break
                    if improved: break
                routes = [r for r in routes if r]
    
            routes = _or_opt(routes, t_day, vtype_names)
    
            for r_idx in range(len(routes)):
                routes[r_idx] = _two_opt(routes[r_idx], t_day)
    
            return routes
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 7 — §9 EXACT COLUMN GENERATION WITH DP PRICING
        # ═════════════════════════════════════════════════════════════════════════
    
        def _pricing_dp(g_nodes, t_day, vtype_name, pi_duals, mu_v):
            spec = VEHICLE_TYPES[vtype_name]
            boxes = spec["boxes_capacity"]
            Q_agg_max = BOX_CAPACITY_AGGREGATE * boxes
            Q_g_max = {g: BOX_CAPACITY_PER_PRODUCT[g] * boxes for g in PRODUCTS}
            range_max = spec.get("max_range_mi", None)
            co2_v = spec["co2_kg_per_mile"]
            unit_cost = 1.0 + CARBON_PRICE_PER_KG * co2_v
            
            allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype_name, [])
    
            servable = []
            for nd in g_nodes:
                pc_nd = lnodes[nd-1][0]
                ok = True
                for g in PRODUCTS:
                    gd = demand.get((pc_nd, g, t_day), 0.0)
                    if gd > 0:
                        if allowed_sizes:
                            can_carry = any(VAQTEC_SPEC.get(sz, {}).get(g) is not None for sz in allowed_sizes)
                            if not can_carry:
                                ok = False; break
                        if gd > Q_g_max[g]:
                            ok = False; break
                if ok: servable.append(nd)
            if not servable: return []
    
            k_rep = _representative_vehicle_of_type(vtype_name)
            if k_rep is None: return []
    
            columns_found = []
            seen_routes = set()
    
            def _try_close(seq, cost_so_far, load_agg):
                if not seq: return
                last = seq[-1]
                close_arc = c.get((last, 0), 0.0) * unit_cost
                reduced = ROUTE_PENALTY - mu_v + cost_so_far + close_arc
                if reduced >= CG_REDUCED_COST_EPS: return
                rt = tuple(seq)
                if rt in seen_routes: return
                ok, _ = _simulate(seq, t_day)
                if not ok: return
                if not _fits_vehicle(seq, t_day, k_rep): return
                seen_routes.add(rt)
                columns_found.append((list(seq), vtype_name, reduced))
    
            def _dfs(seq, cost_so_far, load_g, load_agg, miles, prev_prev):
                _try_close(seq, cost_so_far, load_agg)
                if len(seq) >= CG_MAX_ROUTE_LEN: return
    
                current = seq[-1] if seq else 0
                for j in servable:
                    if j in seq: continue
                    if CG_2CYCLE_ELIM and prev_prev == j: continue
    
                    pc_j = lnodes[j-1][0]
                    d_j_agg = sum(demand.get((pc_j, g, t_day), 0.0) for g in PRODUCTS)
                    new_load_agg = load_agg + d_j_agg
                    if new_load_agg > Q_agg_max: continue
                    new_load_g = list(load_g)
                    over = False
                    for gi, g in enumerate(PRODUCTS):
                        new_load_g[gi] += demand.get((pc_j, g, t_day), 0.0)
                        if new_load_g[gi] > Q_g_max[g]:
                            over = True; break
                    if over: continue
    
                    new_miles = miles + c.get((current, j), 0.0)
                    if range_max is not None and new_miles + c.get((j, 0), 0.0) > range_max:
                        continue
    
                    new_seq = seq + [j]
                    ok, _ = _simulate(new_seq, t_day)
                    if not ok: continue
    
                    arc_c = c.get((current, j), 0.0) * unit_cost
                    arc_reduced = arc_c - pi_duals.get(j, 0.0)
                    new_cost = cost_so_far + arc_reduced
    
                    _dfs(new_seq, new_cost, new_load_g, new_load_agg, new_miles, current)
    
            _dfs([], 0.0, [0.0]*len(PRODUCTS), 0.0, 0.0, -1)
            columns_found.sort(key=lambda x: x[2])
            return columns_found[:CG_MULTI_COL_MAX]
    
        def _solve_group_column_gen(g_nodes, t_day, vtype_names, initial_pool):
            if not g_nodes: return None
    
            pool = {}
            for r, vn in initial_pool:
                rt = tuple(r)
                if not rt: continue
                if vn not in vtype_names: continue
                k_rep = _representative_vehicle_of_type(vn)
                if k_rep is None: continue
                if not _fits_vehicle(list(rt), t_day, k_rep): continue
                ok, _ = _simulate(list(rt), t_day)
                if not ok: continue
                key = (rt, vn)
                if key not in pool:
                    pool[key] = _column_cost(list(rt), vn)
    
            if not pool:
                return None
    
            # Ensure every node is coverable by at least one column to prevent node-drops
            covered_check = set()
            for (rt, vn), cost in pool.items():
                for nd in rt: covered_check.add(nd)
            if set(g_nodes) - covered_check:
                for nd in set(g_nodes) - covered_check:
                    _added_singleton = False
                    for vn in vtype_names:
                        k_rep = _representative_vehicle_of_type(vn)
                        if k_rep is not None and _fits_vehicle([nd], t_day, k_rep):
                            ok, _ = _simulate([nd], t_day)
                            if ok:
                                pool[((nd,), vn)] = _column_cost([nd], vn)
                                _added_singleton = True
                                break
                    if not _added_singleton:
                        # Force column generation coverage fallback 
                        vn_fb = next(iter(vtype_names))
                        pool[((nd,), vn_fb)] = 999999.0
    
            n_v_avail = {vn: sum(1 for kk in K if vtype_of[kk] == vn) for vn in vtype_names}
            fleet_cap = sum(n_v_avail.values())
    
            lp_obj = None
            cg_iter = 0
            for cg_iter in range(CG_MAX_ITER):
                keys = list(pool.keys())
                costs = [pool[k] for k in keys]
    
                rmlp = make_prob(f"RMLP_d{t_day}_g{len(g_nodes)}_it{cg_iter}", shl_name=shl_name)
                lam = [xp.var(name=f"lam_{i}", vartype=xp.continuous, lb=0.0)
                       for i in range(len(keys))]
                y = {vn: xp.var(name=f"y_{vn.replace(' ', '_')}",
                                vartype=xp.continuous, lb=0.0)
                     for vn in vtype_names}
                rmlp.addVariable(lam + list(y.values()))
    
                cov_cons = {}
                for i in g_nodes:
                    terms = [lam[idx] for idx, k_ in enumerate(keys) if i in k_[0]]
                    if terms:
                        cov_cons[i] = xp.Sum(terms) >= 1
                        rmlp.addConstraint(cov_cons[i])
    
                y_cons = {}
                for vn in vtype_names:
                    terms = [lam[idx] for idx, k_ in enumerate(keys) if k_[1] == vn]
                    y_cons[vn] = xp.Sum(terms) - y[vn] == 0
                    rmlp.addConstraint(y_cons[vn])
    
                fleet_con = xp.Sum(y[vn] for vn in vtype_names) <= fleet_cap
                rmlp.addConstraint(fleet_con)
    
                avail_cons = {}
                for vn in vtype_names:
                    avail_cons[vn] = y[vn] <= n_v_avail[vn]
                    rmlp.addConstraint(avail_cons[vn])
    
                rmlp.setObjective(
                    xp.Sum(costs[idx] * lam[idx] for idx in range(len(keys))),
                    sense=xp.minimize)
    
                rmlp.controls.outputlog = 0
                try:
                    rmlp.solve()
                except Exception as e:
                    print(f"      ⚠ RMLP solve failed at iter {cg_iter}: {e}")
                    return None
    
                try:
                    lp_obj = rmlp.getObjVal()
                    if lp_obj is None or abs(lp_obj) > 1e19:
                        return None
                except Exception:
                    return None
    
                pi_duals = {}
                for i in g_nodes:
                    if i in cov_cons:
                        try:
                            pi_duals[i] = float(rmlp.getDual(cov_cons[i]))
                        except Exception:
                            pi_duals[i] = 0.0
                    else:
                        pi_duals[i] = 0.0
                mu_duals = {}
                for vn in vtype_names:
                    try:
                        mu_duals[vn] = float(rmlp.getDual(y_cons[vn]))
                    except Exception:
                        mu_duals[vn] = 0.0
    
                new_columns = []
                for vn in vtype_names:
                    cols = _pricing_dp(g_nodes, t_day, vn, pi_duals, mu_duals[vn])
                    new_columns.extend(cols)
    
                n_added = 0
                for route, vn, rc in new_columns:
                    key = (tuple(route), vn)
                    if key not in pool:
                        pool[key] = _column_cost(route, vn)
                        n_added += 1
    
                if n_added == 0:
                    break
    
            keys = list(pool.keys())
            costs = [pool[k] for k in keys]
    
            rmip = make_prob(f"RMIP_d{t_day}_g{len(g_nodes)}", shl_name=shl_name)
            lam_i = [xp.var(name=f"lami_{i}", vartype=xp.binary) for i in range(len(keys))]
            y_i = {vn: xp.var(name=f"yi_{vn.replace(' ', '_')}",
                              vartype=xp.integer, lb=0, ub=n_v_avail[vn])
                   for vn in vtype_names}
            rmip.addVariable(lam_i + list(y_i.values()))
    
            for i in g_nodes:
                terms = [lam_i[idx] for idx, k_ in enumerate(keys) if i in k_[0]]
                if terms:
                    rmip.addConstraint(xp.Sum(terms) >= 1)
            for vn in vtype_names:
                terms = [lam_i[idx] for idx, k_ in enumerate(keys) if k_[1] == vn]
                rmip.addConstraint(xp.Sum(terms) - y_i[vn] == 0)
            rmip.addConstraint(xp.Sum(y_i[vn] for vn in vtype_names) <= fleet_cap)
    
            rmip.setObjective(
                xp.Sum(costs[idx] * lam_i[idx] for idx in range(len(keys))),
                sense=xp.minimize)
    
            rmip.controls.outputlog = 0
            rmip.controls.miprelstop = CG_GAP_TOLERANCE / 100.0
            try:
                rmip.solve()
                Z_int = rmip.getObjVal()
            except Exception:
                return None
            if Z_int is None or abs(Z_int) > 1e19:
                return None
    
            chosen = []
            for idx, (rt, vn) in enumerate(keys):
                try:
                    if rmip.getSolution(lam_i[idx]) > 0.5:
                        chosen.append((list(rt), vn))
                except Exception:
                    pass
    
            # Safely get the MIP gap, manually calculating it if Xpress rejects the attribute
            try:
                gap = rmip.attributes.miprelgap * 100.0
            except Exception:
                if lp_obj is not None and Z_int is not None:
                    gap = max(0.0, 100.0 * (Z_int - lp_obj) / max(1e-9, abs(lp_obj)))
                else:
                    gap = 0.0

            return dict(
                routes=chosen,
                lp_bound=lp_obj,
                int_cost=Z_int,
                gap_pct=gap,
                columns=list(pool.items()),
                cg_iters=cg_iter + 1,
                n_cols=len(pool),
            )
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 8 — DISPATCH-GROUP IDENTIFICATION
        # ═════════════════════════════════════════════════════════════════════════
    
        DP_GROUP_TOLERANCE = 0.5
    
        def _identify_dispatch_groups(req_nodes, t_day):
            dp_vals = {nd: round(_dp_of_node(nd, t_day), 4) for nd in req_nodes}
            unique_dps = sorted(set(dp_vals.values()))
            dp_group_map = {}; groups_repr = []
            for dp in unique_dps:
                placed = False
                for g_dp in groups_repr:
                    if abs(dp - g_dp) <= DP_GROUP_TOLERANCE:
                        dp_group_map[dp] = g_dp; placed = True; break
                if not placed:
                    groups_repr.append(dp); dp_group_map[dp] = dp
            result = {}
            for nd in req_nodes:
                g = dp_group_map[dp_vals[nd]]
                result.setdefault(g, []).append(nd)
            return result
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 9 — COMPACT MILP
        # ═════════════════════════════════════════════════════════════════════════
    
        GROUP_MILP_THRESHOLD = 50
        MILP_POLISH_TIME_BUDGET = 7200.0
    
        def _solve_group_milp(group_nodes, t_day, heuristic_routes, k_pool):
            if not group_nodes:
                return []
    
            n_g = len(group_nodes)
            K_g = list(range(len(k_pool)))  # local vehicle indices
            k_map = {local: glob for local, glob in enumerate(k_pool)}  # local -> global
    
            arcs_g = set()
            for i in group_nodes:
                arcs_g.add((0, i))
                arcs_g.add((i, 0))
            for i in group_nodes:
                for j in group_nodes:
                    if i != j:
                        arcs_g.add((i, j))
            arcs_g = sorted(arcs_g)
    
            _bigM = {(i, j): T_MAX + tau.get((i, j), 0.0) for (i, j) in arcs_g}
            prob = make_prob(f"VRPTW_d{t_day}_g{n_g}", shl_name=shl_name)
    
            x = {}
            for (i,j) in arcs_g:
                for k in K_g:
                    x[(i,j,k)] = xp.var(name=f"x_{i}_{j}_{k}", vartype=xp.binary)
    
            yk = {k: xp.var(name=f"yk_{k}", vartype=xp.binary) for k in K_g}
    
            v = {}; A = {}; W = {}; E = {}
            for i in group_nodes:
                for k in K_g:
                    v[(i,k)] = xp.var(name=f"v_{i}_{k}", vartype=xp.binary)
                    A[(i,k)] = xp.var(name=f"A_{i}_{k}", vartype=xp.continuous, lb=0.0)
                    W[(i,k)] = xp.var(name=f"W_{i}_{k}", vartype=xp.continuous, lb=0.0)
                    E[(i,k)] = xp.var(name=f"E_{i}_{k}", vartype=xp.continuous, lb=0.0)
    
            Ucap = {}
            if USE_PRODUCT_PERISHABILITY:
                _ucap_ub = MAX_CONTAINER_CEILING_H
                Ucap = {k: xp.var(name=f"Ucap_{k}", vartype=xp.continuous, lb=0.0, ub=_ucap_ub)
                        for k in K_g}
    
            Ddep = {k: xp.var(name=f"Ddep_{k}", vartype=xp.continuous, lb=0.0) for k in K_g}
    
            all_vars = (list(x.values()) + list(yk.values()) +
                        list(v.values()) + list(A.values()) + list(W.values()) + list(E.values()) +
                        list(Ucap.values()) + list(Ddep.values()))
            prob.addVariable(all_vars)
    
            for i in group_nodes:
                prob.addConstraint(xp.Sum(v[(i,k)] for k in K_g) == 1)
    
            for i in group_nodes:
                in_nodes  = [j for j in [0] + group_nodes if j != i and (j,i,K_g[0]) in x]
                out_nodes = [j for j in [0] + group_nodes if j != i and (i,j,K_g[0]) in x]
                for k in K_g:
                    prob.addConstraint(
                        xp.Sum(x[(j,i,k)] for j in in_nodes) ==
                        xp.Sum(x[(i,j,k)] for j in out_nodes))
    
            for k in K_g:
                out_j = [j for j in group_nodes if (0,j,k) in x]
                in_j  = [j for j in group_nodes if (j,0,k) in x]
                prob.addConstraint(xp.Sum(x[(0,j,k)] for j in out_j) == yk[k])
                prob.addConstraint(xp.Sum(x[(j,0,k)] for j in in_j)  == yk[k])
    
            if SYMMETRY_BREAK:
                for k_idx in range(len(K_g)-1):
                    k1 = K_g[k_idx]; k2 = K_g[k_idx+1]
                    if vtype_of[k_map[k1]] == vtype_of[k_map[k2]]:
                        prob.addConstraint(yk[k1] >= yk[k2])
    
            for i in group_nodes:
                in_nodes = [j for j in [0] + group_nodes if j != i and (j,i,K_g[0]) in x]
                for k in K_g:
                    prob.addConstraint(v[(i,k)] == xp.Sum(x[(j,i,k)] for j in in_nodes))
    
            for k in K_g:
                glob_k = k_map[k]
                Qg_type={g: BOX_CAPACITY_PER_PRODUCT[g]*VEHICLE_TYPES[vtype_of[glob_k]]["boxes_capacity"]
                         for g in PRODUCTS}
                Qagg_type=BOX_CAPACITY_AGGREGATE*VEHICLE_TYPES[vtype_of[glob_k]]["boxes_capacity"]
                for g in PRODUCTS:
                    terms=[demand.get((lnodes[i-1][0],g,t_day),0.0)*v[(i,k)] for i in group_nodes]
                    prob.addConstraint(xp.Sum(terms)<=Qg_type[g])
                agg_terms=[demand.get((lnodes[i-1][0],g,t_day),0.0)*v[(i,k)]
                           for i in group_nodes for g in PRODUCTS]
                prob.addConstraint(xp.Sum(agg_terms)<=Qagg_type)
    
            for k in K_g:
                glob_k = k_map[k]
                rng=VEHICLE_TYPES[vtype_of[glob_k]].get("max_range_mi")
                if rng is None: continue
                prob.addConstraint(
                    xp.Sum(c[(i,j)]*x[(i,j,k)] for (i,j) in arcs_g) <= rng)
    
            for k in K_g:
                glob_k = k_map[k]
                vtype_name = vtype_of[glob_k]
                allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype_name, [])
                if not allowed_sizes:
                    continue
                
                for g in PRODUCTS:
                    can_carry = any(VAQTEC_SPEC.get(sz, {}).get(g) is not None for sz in allowed_sizes)
                    if can_carry:
                        continue
                        
                    nodes_needing_g = [i for i in group_nodes
                                       if demand.get((lnodes[i-1][0], g, t_day), 0.0) > 0]
                    for i in nodes_needing_g:
                        prob.addConstraint(v[(i,k)] == 0)
    
            if USE_PRODUCT_PERISHABILITY:
                _M_perish = max(
                    [container_ceiling_for_products(vt, set(PRODUCTS)) or 0.0
                     for vt in VEHICLE_VAQTEC_SIZES] + [max(PERISH_HOURS.values())])
                for k in K_g:
                    vt_k = vtype_of[k_map[k]]
                    prob.addConstraint(
                        xp.Sum(tau[(i, j)] * x[(i, j, k)] for (i, j) in arcs_g)
                        + xp.Sum(s_node[(i, t_day)] * v[(i, k)] for i in group_nodes)
                        + xp.Sum(W[(i, k)] for i in group_nodes) <= Ucap[k])
                    for i in group_nodes:
                        pc = lnodes[i - 1][0]
                        prods_i = {g for g in PRODUCTS
                                   if demand.get((pc, g, t_day), 0.0) > 0}
                        perish_i = container_ceiling_for_products(vt_k, prods_i)
                        if perish_i is None:
                            prob.addConstraint(v[(i, k)] == 0)   
                            continue
                        prob.addConstraint(
                            Ucap[k] <= perish_i + _M_perish * (1 - v[(i, k)]))

            for i in group_nodes:
                for k in K_g:
                    prob.addConstraint(E[(i,k)]<=T_MAX)
                    dp_h=dp_node[(i,t_day)]
                    prob.addConstraint(A[(i,k)]+W[(i,k)]>=dp_h*v[(i,k)])
    
            for k in K_g:
                prob.addConstraint(Ddep[k] >= T_DEP * yk[k])
                prob.addConstraint(Ddep[k] <= T_MAX * yk[k])
                for i in group_nodes:
                    prob.addConstraint(Ddep[k] >= dp_node[(i,t_day)] * v[(i,k)])
                    if (0,i,k) in x:
                        prob.addConstraint(A[(i,k)] >= Ddep[k] + tau[(0,i)] - _bigM[(0,i)] * (1 - x[(0,i,k)]))
    
            for i in group_nodes:
                for j in group_nodes:
                    if i == j: continue
                    if (i,j,K_g[0]) not in x: continue
                    for k in K_g:
                        prob.addConstraint(
                            A[(j,k)] >= E[(i,k)] + tau[(i,j)] - _bigM[(i,j)]*(1 - x[(i,j,k)]))
    
            for i in group_nodes:
                for k in K_g:
                    prob.addConstraint(E[(i,k)]==A[(i,k)]+W[(i,k)]+s_node[(i,t_day)]*v[(i,k)])
                    prob.addConstraint(A[(i,k)]<=T_MAX*v[(i,k)])
                    prob.addConstraint(E[(i,k)]<=T_MAX*v[(i,k)])
                    prob.addConstraint(W[(i,k)]<=MAX_WAIT_H*v[(i,k)])
    
            for i in group_nodes:
                ch_eff=ch_adj.get((i,t_day))
                if ch_eff is None:
                    continue
                for k in K_g:
                    prob.addConstraint(A[(i,k)]+W[(i,k)]<=ch_eff+T_MAX*(1-v[(i,k)]))
    
            cost_term = xp.Sum(c[(i,j)] * VEHICLE_TYPES[vtype_of[k_map[k]]]["cost_per_mile"] * x[(i,j,k)]
                               for (i,j) in arcs_g for k in K_g)
            consolidation_term = ROUTE_PENALTY * xp.Sum(yk[k] for k in K_g)
            emissions_term = xp.Sum(c[(i,j)] * VEHICLE_TYPES[vtype_of[k_map[k]]]["co2_kg_per_mile"] * x[(i,j,k)]
                                    for (i,j) in arcs_g for k in K_g)
            prob.setObjective(cost_term + consolidation_term + CARBON_PRICE_PER_KG * emissions_term,
                               sense=xp.minimize)
    
            sol_dict = {}
            for var_obj in x.values():     sol_dict[var_obj] = 0.0
            for var_obj in yk.values():    sol_dict[var_obj] = 0.0
            for var_obj in v.values():     sol_dict[var_obj] = 0.0
            for var_obj in A.values():     sol_dict[var_obj] = 0.0
            for var_obj in W.values():     sol_dict[var_obj] = 0.0
            for var_obj in E.values():     sol_dict[var_obj] = 0.0
            for var_obj in Ucap.values():  sol_dict[var_obj] = 0.0
            for var_obj in Ddep.values():  sol_dict[var_obj] = 0.0
    
            _ws_ok = True
            _ws_covered = set()
            k_order = sorted(K_g, key=lambda kk: (
                -VEHICLE_TYPES[vtype_of[k_map[kk]]]["boxes_capacity"], kk))
            free = list(k_order)
    
            h_routes_sorted = sorted(heuristic_routes,
                key=lambda r: sum(demand.get((lnodes[nd-1][0], g, t_day), 0.0)
                                  for nd in r for g in PRODUCTS), reverse=True)
    
            for route in h_routes_sorted:
                k_use = None
                for kk in free:
                    if _fits_vehicle(route, t_day, k_map[kk]):
                        k_use = kk
                        break
                if k_use is None:
                    _ws_ok = False
                    continue
                free.remove(k_use)
    
                ok, timing = _simulate(route, t_day)
                if not ok:
                    _ws_ok = False
                    continue
    
                nodes_seq = [0] + list(route) + [0]
                for a, b in zip(nodes_seq[:-1], nodes_seq[1:]):
                    key = (a, b, k_use)
                    if key in x:
                        sol_dict[x[key]] = 1.0
                for nd in route:
                    if (nd, k_use) in v:
                        sol_dict[v[(nd, k_use)]] = 1.0
                        _ws_covered.add(nd)
                sol_dict[yk[k_use]] = 1.0
    
                for (nd, A_raw, wait, dep) in timing:
                    if (nd, k_use) in A:
                        sol_dict[A[(nd, k_use)]] = A_raw
                        sol_dict[W[(nd, k_use)]] = wait
                        sol_dict[E[(nd, k_use)]] = dep
                if k_use in Ucap:
                    sol_dict[Ucap[k_use]] = _route_perishability(
                        route, t_day,
                        vtype_of[k_map[k_use]] if k_use in k_map else None)
                sol_dict[Ddep[k_use]] = max(T_DEP, max(_dp_of_node(nd, t_day) for nd in route))
    
            _missing_ws = set(group_nodes) - _ws_covered
            if not _missing_ws:
                sol_vars = list(sol_dict.keys())
                sol_vals = list(sol_dict.values())
                try:
                    prob.addmipsol(sol_vals, sol_vars, "heuristic_ws")
                except Exception as e:
                    pass
            else:
                print(f"      §10 warm start NOT applied to compact MILP: "
                      f"{len(_missing_ws)}/{len(group_nodes)} nodes uncovered "
                      f"(routes fed={len(heuristic_routes)}, "
                      f"|k_pool|={len(k_pool)})", flush=True)
    
            MAX_SUBTOUR_ITER = 25
            total_cuts = 0
    
            import time as _polish_time
            _polish_start = _polish_time.time()
    
            for subtour_iter in range(MAX_SUBTOUR_ITER):
                _polish_remaining = MILP_POLISH_TIME_BUDGET - (_polish_time.time() - _polish_start)
                if _polish_remaining <= 5.0:
                    print(f"      §10 MILP polish budget exhausted "
                          f"({MILP_POLISH_TIME_BUDGET:.0f}s) at DFJ iter {subtour_iter}", flush=True)
                    break
                try:
                    prob.controls.timelimit = _polish_remaining
                except Exception:
                    pass
                prob.solve()
    
                has_incumbent = False
                try:
                    obj = prob.getObjVal()
                    has_incumbent = (obj is not None) and (abs(obj) < 1e19)
                except Exception:
                    has_incumbent = False
    
                if not has_incumbent:
                    break
    
                subtours = []
                for k in K_g:
                    used = [(i,j) for (i,j) in arcs_g if (i,j,k) in x
                             and prob.getSolution(x[(i,j,k)]) > 0.5]
                    if not used:
                        continue
                    adj = {}
                    nodes_in_route = set()
                    for (i,j) in used:
                        adj[i] = j
                        nodes_in_route.add(i)
                        nodes_in_route.add(j)
                    if 0 not in nodes_in_route:
                        customer_nodes = nodes_in_route - {0}
                        if customer_nodes:
                            subtours.append((customer_nodes, k))
                        continue
                    visited = {0}
                    cur = adj.get(0)
                    while cur is not None and cur != 0 and cur not in visited:
                        visited.add(cur)
                        cur = adj.get(cur)
                    visited.add(0)
                    subtour_nodes = nodes_in_route - visited - {0}
                    if subtour_nodes:
                        remaining = set(subtour_nodes)
                        while remaining:
                            start = next(iter(remaining))
                            comp = set()
                            stack = [start]
                            while stack:
                                n = stack.pop()
                                if n in comp: continue
                                comp.add(n)
                                nxt = adj.get(n)
                                if nxt is not None and nxt in remaining:
                                    stack.append(nxt)
                            subtours.append((comp, k))
                            remaining -= comp
    
                if not subtours:
                    break
    
                n_cuts = 0
                for (S, k) in subtours:
                    complement = (set(group_nodes) | {0}) - S
                    leaving_arcs = [(i,j) for i in S for j in complement if (i,j,k) in x]
                    if leaving_arcs:
                        prob.addConstraint(xp.Sum(x[(i,j,k)] for (i,j) in leaving_arcs) >= 1)
                        n_cuts += 1
                total_cuts += n_cuts
    
            has_incumbent = False
            try:
                obj = prob.getObjVal()
                has_incumbent = (obj is not None) and (abs(obj) < 1e19)
            except Exception:
                has_incumbent = False
    
            if not has_incumbent:
                return None
    
            milp_routes = []
            for k in K_g:
                al = [(i,j) for (i,j) in arcs_g if (i,j,k) in x
                      and prob.getSolution(x[(i,j,k)]) > 0.5]
                if not al:
                    continue
                adj = {i: j for (i,j) in al}
                if 0 not in adj:
                    continue
                seq = []
                cur = adj[0]
                while cur != 0 and cur in adj:
                    seq.append(cur)
                    cur = adj.get(cur, 0)
                if seq:
                    milp_routes.append((seq, k_map[k]))
            
            # --- STRICT ANTI-DROP CHECK ---
            _mr_flat = [nd for r, _ in milp_routes for nd in r]
            if set(_mr_flat) != set(group_nodes) or len(_mr_flat) != len(group_nodes):
                return None
    
            return milp_routes
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 10 — §10 MAIN LOOP  (MODIFIED to implement Stages 1-5 + Defences)
        # ═════════════════════════════════════════════════════════════════════════
    
        print(f"  ── MAIN LOOP ENTRY ──", flush=True)
    
        print(f"\n  ═══ v27.3 METHODOLOGY-ALIGNED SOLVE ═══", flush=True)
        print(f"  Preprocessing: {'ON' if PP_ENABLE else 'OFF'}  "
              f"ICW iters: {ICW_MAX_ITER} (stall {ICW_STALL_LIMIT})  "
              f"CG iters: {CG_MAX_ITER}  MILP threshold: ≤{GROUP_MILP_THRESHOLD} nodes", flush=True)
    
        routes_by_day = {t: [] for t in DAYS}
        vehicles_used = set()
        total_dfj_cuts = 0
        total_lp_bound = 0.0     
        total_int_cost = 0.0     
        _global_route_idx = 0
        _day_timings = {}
        per_group_gaps = []
    
        import time 
        for t in DAYS:
            t0_day = time.time()
            req_today = req_nodes_day[t]
            if not req_today:
                print(f"\n  Day {DAY_NAMES.get(t, t)}: no required visits.", flush=True)
                _day_timings[t] = 0.0; continue
    
            dp_groups = _identify_dispatch_groups(req_today, t)
            n_groups = len(dp_groups)
            print(f"\n  Day {DAY_NAMES.get(t, t)}: {len(req_today)} nodes → {n_groups} "
                  f"dispatch group(s): "
                  + ", ".join(f"DP={g:.2f}h({len(ns)})" for g, ns in sorted(dp_groups.items())), flush=True)
    
            day_routes_with_k = []
    
            for g_dp, g_nodes in sorted(dp_groups.items()):
                g_label = f"d{t}_g{g_dp:.2f}"
    
                V_g_prime = _preprocess_vehicle_types(g_nodes, t)
                V_g_all = set(vtype_of[kk] for kk in K)
                if V_g_prime != V_g_all:
                    print(f"    Group DP={g_dp:.2f}h §5 preprocessing: "
                          f"V_g={sorted(V_g_all)} → V_g'={sorted(V_g_prime)}", flush=True)
    
                icw_best, icw_pool = _icw_construct(g_nodes, t, V_g_prime)
    
                ls_routes = _local_search([list(r) for r in icw_best], t, V_g_prime)
    
                _ls_flat = [nd for r in ls_routes for nd in r]
                _ls_set = set(_ls_flat)
                if _ls_set != set(g_nodes) or len(_ls_flat) != len(_ls_set):
                    _missing = sorted(set(g_nodes) - _ls_set)
                    _dupes = sorted({nd for nd in _ls_flat if _ls_flat.count(nd) > 1})
                    print(f"    ⚠ LS coverage violation in group {g_label} "
                          f"(day={t}): missing={_missing}, duplicated={_dupes} "
                          f"— reverting to ICW routes")
                    ls_routes = [list(r) for r in icw_best]
    
                icw_mi = sum(_route_miles(r) for r in icw_best)
                ls_mi = sum(_route_miles(r) for r in ls_routes)
                ls_gain = icw_mi - ls_mi
    
                for r in ls_routes: icw_pool.add(tuple(r))
    
                initial_pool = []
                for rt in icw_pool:
                    r_list = list(rt)
                    for vn in sorted(V_g_prime,
                                     key=lambda x: VEHICLE_TYPES[x]["boxes_capacity"]):
                        k_rep = _representative_vehicle_of_type(vn)
                        if k_rep is not None and _fits_vehicle(r_list, t, k_rep):
                            initial_pool.append((r_list, vn))
                            break
    
                print(f"    Group DP={g_dp:.2f}h: {len(g_nodes)} nodes, "
                      f"ICW→{len(icw_best)} routes ({icw_mi:.1f}mi), "
                      f"LS→{len(ls_routes)} routes ({ls_mi:.1f}mi"
                      f"{f', saved {ls_gain:.1f}mi' if ls_gain > 0.5 else ''})  "
                      f"pool={len(icw_pool)}", flush=True)
    
                final_routes = ls_routes
                cg_route_asgn = None
                cg_used = False
    
                cg_result = None
                try:
                    cg_result = _solve_group_column_gen(g_nodes, t, V_g_prime, initial_pool)
                except Exception as e:
                    print(f"      ⚠ CG raised exception: {e}; falling back to LS", flush=True)
    
                if cg_result is None:
                    print(f"      ⚠ CG returned None; falling back to LS + compact MILP", flush=True)
                else:
                    _cg_flat = [nd for r, _vn in cg_result["routes"] for nd in r]
                    
                    # FIX 1: Set Covering is allowed to duplicate nodes to maintain routes. 
                    # We just need to verify that it didn't MISS any nodes.
                    if set(_cg_flat) != set(g_nodes):
                        print(f"      ⚠ CG coverage violation (missing nodes). Rejecting CG.")
                        cg_result = None
                    else:
                        # Clean Set-Covering duplicates to make it a strict mathematical Partition
                        seen = set()
                        cleaned_cg = []
                        for r, _vn in cg_result["routes"]:
                            new_r = [nd for nd in r if nd not in seen and not seen.add(nd)]
                            if new_r:
                                cleaned_cg.append((new_r, _vn))
                        cg_route_asgn = cleaned_cg
                        
                        cg_used = True
                        Z_LP  = cg_result["lp_bound"]
                        Z_INT = cg_result["int_cost"]
                        gap   = cg_result["gap_pct"]
                        total_lp_bound += Z_LP
                        total_int_cost += Z_INT
                        per_group_gaps.append(gap)
                        print(f"      §9 CG: {cg_result['cg_iters']} iters, |pool|={cg_result['n_cols']}, "
                              f"Z_LP={Z_LP:.2f}, Z_INT={Z_INT:.2f}, gap={gap:.2f}%", flush=True)
                    
                        ls_cost = sum(_route_miles(r) for r in ls_routes)
                        cg_cost = sum(_route_miles(r) for (r, _vn) in cg_route_asgn)
                        if cg_cost + 0.1 < ls_cost:
                            final_routes = [r for (r, _vn) in cg_route_asgn]
    
                milp_used = False
                _cg_closed = (cg_used
                              and cg_result is not None
                              and cg_result["gap_pct"] <= CG_GAP_TOLERANCE + 1e-6
                              and cg_cost < ls_cost - 0.1)
    
                if _cg_closed:
                    print(f"      §10 skipping compact MILP polish: "
                          f"CG closed gap ({cg_result['gap_pct']:.2f}% ≤ "
                          f"{CG_GAP_TOLERANCE}%)", flush=True)
    
                if (not _cg_closed) and len(g_nodes) <= GROUP_MILP_THRESHOLD and len(g_nodes) > 0:
                    ls_cost  = sum(_route_miles(r) for r in ls_routes)
                    cg_cost  = (sum(_route_miles(r) for (r, _vn) in cg_route_asgn)
                                if cg_used else float('inf'))
                    ws_v27 = (ls_routes if ls_cost <= cg_cost
                              else [r for (r, _vn) in cg_route_asgn])
    
                    try:
                        cw_v26 = cw_warmstart_day(g_nodes, t, vtype_names=None)
                        ls_v26 = _local_search([list(r) for r in cw_v26], t, None)
                    except Exception as _e_v26_pre:
                        print(f"      §10 v26-pre exception ({_e_v26_pre}); "
                              f"falling back to v27 warm-start only", flush=True)
                        ls_v26 = list(ws_v27)
                    v26_cost = sum(_route_miles(r) for r in ls_v26)
    
                    # FIX 2: Build kpool dynamically to explicitly include capable vehicles.
                    # This prevents the "Problem is integer infeasible" Xpress crash.
                    kpool_set = set()
                    free_k = list(K)
                    for ws_route in ws_v27:
                        for kk in free_k:
                            if _fits_vehicle(ws_route, t, kk):
                                kpool_set.add(kk)
                                free_k.remove(kk)
                                break
                    # Give the MILP 3 slack vehicles to search across
                    for kk in free_k[:3]:
                        kpool_set.add(kk)
                    kpool = list(kpool_set)
    
                    candidates = []   
                    if kpool:
                        for label, ws in [("v27-warm", ws_v27)]:
                            _ws_cost = sum(_route_miles(r) for r in ws)
                            try:
                                _mr = _solve_group_milp(g_nodes, t, ws, kpool)
                            except Exception as _e_milp:
                                print(f"      §10 MILP({label}) exception: {_e_milp}", flush=True)
                                continue
                            if _mr is None:
                                print(f"      §10 MILP({label}) returned None "
                                      f"(ws={len(ws)}r@{_ws_cost:.1f}mi, "
                                      f"|kpool|={len(kpool)})", flush=True)
                                continue
                            _mi = sum(_route_miles(r) for r, _ in _mr)
                            candidates.append((_mi, label, _mr))
                            print(f"      §10 MILP({label}): ws={len(ws)}r@{_ws_cost:.1f}mi "
                                  f"→ {len(_mr)}r@{_mi:.1f}mi", flush=True)
    
                    if candidates:
                        candidates.sort(key=lambda x: x[0])
                        best_mi, best_label, best_mr = candidates[0]
                        heur_best = min(ls_cost, cg_cost, v26_cost)
                        if best_mi <= heur_best + 0.1:
                            day_routes_with_k.extend(best_mr)
                            milp_used = True
                            print(f"      §10 SELECTED: {best_label} "
                                  f"({best_mi:.1f}mi vs heur best {heur_best:.1f}mi) "
                                  f"[LS={ls_cost:.1f} CG={cg_cost if cg_used else float('nan'):.1f} "
                                  f"v26={v26_cost:.1f}]", flush=True)
                        else:
                            print(f"      §10 REJECTED all MILP results: "
                                  f"best={best_mi:.1f}mi > heur {heur_best:.1f}mi", flush=True)
                    else:
                        print(f"      §10 no MILP candidates produced "
                              f"(kpool empty or all exceptions)", flush=True)
    
                # ─────────────────────────────────────────────────────────────
                # STAGE 6  Vehicle assignment
                # ─────────────────────────────────────────────────────────────
                if not milp_used and cg_used and \
                   cg_result["gap_pct"] <= CG_GAP_TOLERANCE + 1e-6:
                    free_by_type = {vn: sorted([kk for kk in K if vtype_of[kk] == vn])
                                    for vn in V_g_prime}
                    for (route, vn) in cg_route_asgn:
                        if not free_by_type.get(vn):
                            assigned = False
                            for kk in K:
                                if _fits_vehicle(route, t, kk):
                                    day_routes_with_k.append((route, kk))
                                    assigned = True; break
                            if not assigned:
                                print(f"      ⚠ No vehicle for CG-assigned route "
                                      f"({vn} exhausted). Forcing K[0] to preserve coverage.")
                                day_routes_with_k.append((route, K[0]))
                            continue
                        k_use = free_by_type[vn].pop(0)
                        day_routes_with_k.append((route, k_use))
    
                elif not milp_used:
                    k_order = sorted(K,
                        key=lambda kk: (-VEHICLE_TYPES[vtype_of[kk]]["boxes_capacity"], kk))
                    free = list(k_order)
                    h_sorted = sorted(final_routes,
                        key=lambda r: sum(demand.get((lnodes[nd-1][0], g, t), 0.0)
                                          for nd in r for g in PRODUCTS), reverse=True)
                    for route in h_sorted:
                        k_use = None
                        for kk in free:
                            if _fits_vehicle(route, t, kk):
                                k_use = kk; break
                        if k_use is None:
                            if free: 
                                k_use = free[0]
                            else:
                                print(f"      ⚠ Fleet capacity exhausted in group {g_label}. "
                                      f"Forcing vehicle K[0] to preserve coverage.")
                                k_use = K[0]
                        if k_use in free:
                            free.remove(k_use)
                        day_routes_with_k.append((route, k_use))
    
            # ─────────────────────────────────────────────────────────────────────
            # Build route dicts for this day 
            # ─────────────────────────────────────────────────────────────────────
            for (route, k) in day_routes_with_k:
                ok, timing = _simulate(route, t)
                if not ok or timing is None:
                    pcs = [lnodes[nd-1][0] for nd in route]
                    print(f"    ⚠ Simulation failed for route on day {t}, vehicle {k}. "
                          f"Route hospitals: {pcs}. FORCING synthetic timing to prevent node loss.")
                    timing = []
                    clock = max(T_DEP, max(_dp_of_node(nd, t) for nd in route))
                    clock += tau.get((0, route[0]), 0.0)
                    for pos, nd in enumerate(route):
                        svc = s_node.get((nd, t), 0.0)
                        timing.append((nd, clock, 0.0, clock + svc))
                        if pos < len(route) - 1:
                            clock += svc + tau.get((nd, route[pos+1]), 0.0)
                            
                vehicles_used.add(k)
                km = _route_miles(route)
                dur = sum(tau.get((a, b), 0.0)
                          for a, b in zip([0] + list(route), list(route) + [0]))
                route_depart_h = round(max(T_DEP, max(_dp_of_node(nd, t) for nd in route)), 2)
    
                stops = []; boxes_used = 0
                for (nd, A_raw, wait, dep) in timing:
                    pc, s = lnodes[nd-1]
                    slot_name = (slot_windows[pc][t][s][0]
                                 if s < len(slot_windows[pc].get(t, [])) else f"s{s}")
                    stops.append(dict(pc=pc, slot=slot_name,
                                      arrival_h=round(A_raw + wait, 2),
                                      raw_arrival_h=round(A_raw, 2),
                                      depart_h=round(dep, 2)))
                    boxes_used += boxes_est_node.get((nd, t), 1)
    
                hosp_seq = [s["pc"] for s in stops]
                vtype = vtype_of[k]
                route_co2 = km * VEHICLE_TYPES[vtype]["co2_kg_per_mile"]
                box_cap = VEHICLE_TYPES[vtype]["boxes_capacity"]
    
                _global_route_idx += 1
                routes_by_day[t].append(dict(
                    vehicle=k, vtype=vtype, hospitals=hosp_seq,
                    logical_nodes=list(route),
                    km=round(km, 1), dur_h=round(dur, 2), stops=stops,
                    route_depart_h=route_depart_h,
                    co2_kg=round(route_co2, 2), boxes_est=boxes_used,
                    boxes_capacity=box_cap,
                    box_utilisation=round(boxes_used / box_cap, 3) if box_cap else None))
    
            t1_day = time.time()
            _day_timings[t] = t1_day - t0_day
            day_mi = sum(r["km"] for r in routes_by_day[t])
            print(f"  Day {DAY_NAMES.get(t, t)} complete: {len(routes_by_day[t])} routes, "
                  f"{day_mi:.1f}mi, {_day_timings[t]:.1f}s", flush=True)
    
        # ═════════════════════════════════════════════════════════════════════════
        # PART 11 — Summary 
        # ═════════════════════════════════════════════════════════════════════════
        all_r = [r for t in DAYS for r in routes_by_day[t]]
        total_km = sum(r["km"] for r in all_r)
        total_co2 = sum(r["co2_kg"] for r in all_r)
        covered_pc = set()
        for r in all_r:
            for pc in r["hospitals"]: covered_pc.add(pc)
    
        n_vehicles = len(vehicles_used)
        used_by_type = {}
        for k in vehicles_used:
            used_by_type[vtype_of[k]] = used_by_type.get(vtype_of[k], 0) + 1
    
        route_idx = 0; tagged = []
        for t in DAYS:
            for r in routes_by_day[t]:
                route_idx += 1
                rc = dict(r); rc["vehicle"] = f"OPT-{route_idx:02d}"; rc["_day"] = t
                tagged.append(rc)
    
        _print_schedule_table(f"{data.get('shl_name', SHL_NAME)} — Optimised routes (v27.3 methodology)",
                              tagged, data)
        print(f"\n  Total distance : {total_km:.1f} mi")
        print(f"  Total CO2      : {total_co2:.1f} kg CO2e/wk  (by type: " +
              ", ".join(f"{tn}={sum(r['co2_kg'] for r in all_r if r['vtype']==tn):.1f}kg"
                        for tn in VEHICLE_TYPES) + ")")
        print(f"  Vehicles used  : {n_vehicles}/{Cd} available (baseline={C_baseline})  "
              f"by type: {used_by_type}")
        print(f"  Coverage       : {len(covered_pc)}/{len(ids)} hospitals")
    
        if per_group_gaps:
            avg_gap = sum(per_group_gaps) / len(per_group_gaps)
            max_gap = max(per_group_gaps)
         #  certified_gap = 100.0 * (total_int_cost - total_lp_bound) / max(abs(total_lp_bound), 1e-9)
            print(f"  §9 CG bounds   : Z_LP={total_lp_bound:.2f}  Z_INT={total_int_cost:.2f}  "
                  f"worst-group CG gap={max_gap:.2f}%  "
                  f"(per-group avg {avg_gap:.2f}%)", flush=True)
    
        total_time = sum(_day_timings.values())
        print(f"  Total solve time: {total_time:.1f}s  "
              f"(by day: {', '.join(f'{DAY_NAMES.get(t,t)}={_day_timings.get(t,0):.1f}s' for t in DAYS)})")
    
        if real_boxes_by_hospital:
            model_avg = sum(boxes_est_node.values()) / max(len(boxes_est_node), 1)
            real_avg = sum(real_boxes_by_hospital.values()) / max(len(real_boxes_by_hospital), 1)
            print(f"  Boxes cross-check: model avg {model_avg:.2f} vs real avg {real_avg:.2f} — "
                  f"{'✓ consistent' if abs(model_avg - real_avg) < 1.5 else '⚠ check calibration'}")
    
        print("solve completed (v27.3: methodology-aligned; dynamic kpool).")
    
        return dict(
            routes_by_day=routes_by_day,
            U=U,
            MAX_WAIT_H=MAX_WAIT_H,
            relaxed_nodes=relaxed_nodes,
            n_relaxed_nodes=len(relaxed_nodes),
            n_relaxed_beyond_slack=sum(1 for _v in relaxed_nodes.values()
                                       if _v["beyond_slack"]),
            n_relaxed_zero_width=sum(1 for _v in relaxed_nodes.values()
                                     if _v["zero_width"]),
            total_km=total_km,
            total_co2_kg=round(total_co2, 1),
            n_routes=len(all_r),
            n_vehicles=n_vehicles,
            used_by_type=used_by_type,
            fleet_composition=fleet_comp,
            vtype_of=vtype_of,
            C_baseline=C_baseline,
            Cd=Cd,
            covered=covered_pc,
            mip_gap_pct=round(max(per_group_gaps), 2) if per_group_gaps else 0.0,
            lp_bound_total=total_lp_bound,
            int_cost_total=total_int_cost,
            certified_gap_pct=round(max(per_group_gaps), 2) if per_group_gaps else None,
            per_group_gaps=per_group_gaps,
            x={}, v={}, yk={}, A={}, W={}, E={}, Ddep={},
            c=c, tau=tau, lnode_idx=lnode_idx, lnodes=lnodes, arcs=[],
            dfj_cuts_added=total_dfj_cuts,
        )
finally:
    sys.stdout = _orig_stdout
    _log.close()
print(open('/tmp/solve_log.txt').read())

In [12]:
def make_map(shl_name, result, data, stage_label, subtitle):
    ids=data["ids"]; hlat=data["hlat"]; hlon=data["hlon"]
    hname=data["hname"]; dlat=data["dlat"]; dlon=data["dlon"]
    routes_by_day=result["routes_by_day"]
    fig=go.Figure(); all_lats=[]; all_lons=[]; ci=0
    for t in DAYS:
        for ri,route in enumerate(routes_by_day.get(t,[]),1):
            color=COLORS[ci%len(COLORS)]; ci+=1
            stops=route.get("stops") or [dict(pc=pc) for pc in route["hospitals"]]
            n_stops=len(stops)
            label=f"{DAY_NAMES[t]} · R{ri} · {n_stops} stop{'s' if n_stops!=1 else ''}"
            lats=[]; lons=[]; hovers=[]
            for sn,stop in enumerate(stops,1):
                pc=stop.get("pc")
                if pc not in hlat: continue
                lats.append(hlat[pc]); lons.append(hlon[pc])
                all_lats.append(hlat[pc]); all_lons.append(hlon[pc])
                ah=stop.get("arrival_h"); dh=stop.get("depart_h")
                arr_info=""
                if ah: arr_info=f"<br>Arrival: {ah:.2f}h"
                if dh: arr_info+=f" → depart {dh:.2f}h"
                vtype_str=f" | {route.get('vtype','')}" if route.get('vtype') else ""
                co2_str=f" | {route['co2_kg']:.1f}kg CO2" if route.get('co2_kg') is not None else ""
                hovers.append(f"<b>{pc}</b> — {hname.get(pc,pc)}<br>"
                              f"Day: {DAY_NAMES[t]} | Stop {sn}/{n_stops}<br>"
                              f"{route['km']:.1f} mi | {route['dur_h']*60:.0f} min"
                              +arr_info+vtype_str+co2_str)
            if not lats: continue
            fig.add_trace(go.Scattermap(lat=[dlat]+lats+[dlat],lon=[dlon]+lons+[dlon],
                mode="lines",line=dict(width=2.5,color=color),name=DAY_NAMES[t],showlegend=False,hoverinfo="skip"))
            fig.add_trace(go.Scattermap(lat=lats,lon=lons,mode="markers+text",
                marker=dict(size=12,color=color,opacity=0.93),
                text=[str(s) for s in range(1,n_stops+1)],textfont=dict(size=8,color="white"),
                textposition="middle center",name=label,showlegend=True,hovertext=hovers,hoverinfo="text"))
    if not all_lats: return None
    fig.add_trace(go.Scattermap(lat=[dlat],lon=[dlon],mode="markers+text",
        marker=dict(size=20,color="#111",symbol="star"),text=[shl_name],textposition="top right",
        textfont=dict(size=12,color="#111",weight="bold"),name=f"{shl_name} depot",showlegend=True))
    clat=sum(all_lats)/len(all_lats); clon=sum(all_lons)/len(all_lons)
    fig.update_layout(map=dict(style="carto-positron",center=dict(lat=clat,lon=clon),zoom=7),
        title=dict(text=f"{stage_label} — {shl_name} SHL<br>"
                   f'<sup style="color:#555;font-size:13px">{subtitle}</sup>',
                   font=dict(size=17,color="#111"),x=0.02),height=780,
        legend=dict(font=dict(size=10),bgcolor="rgba(255,255,255,0.9)",bordercolor="#ccc",
                    borderwidth=1,x=0.01,y=0.99,xanchor="left",yanchor="top"),
        margin=dict(t=85,b=10,l=10,r=10),paper_bgcolor="white")
    return fig

def make_baseline_map(shl_name, all_trips, data, subtitle):
    hlat=data["hlat"]; hlon=data["hlon"]; hname=data["hname"]
    dlat=data["dlat"]; dlon=data["dlon"]
    fig=go.Figure(); all_lats=[]; all_lons=[]
    for ri,route in enumerate(all_trips):
        color=COLORS[ri%len(COLORS)]
        stops=route.get("stops") or [dict(pc=pc) for pc in route["hospitals"]]
        n_s=len(stops)
        label=f"{route['vehicle']} · {n_s} stop{'s' if n_s!=1 else ''}"
        lats=[]; lons=[]; hovers=[]
        for sn,stop in enumerate(stops,1):
            pc=stop.get("pc")
            if pc not in hlat: continue
            lats.append(hlat[pc]); lons.append(hlon[pc])
            all_lats.append(hlat[pc]); all_lons.append(hlon[pc])
            ah=stop.get("arrival_h")
            co2_str=f" | {route['co2_kg']:.1f}kg CO2 ({route.get('vtype','')})" if route.get('co2_kg') is not None else ""
            hovers.append(f"<b>{pc}</b> — {hname.get(pc,pc)}<br>Round: {route['vehicle']} | Stop {sn}/{n_s}<br>"
                          f"{route['km']:.1f} mi total"+(f"  Sched: {_fmt_h(ah)}" if ah else "")+co2_str)
        if not lats: continue
        fig.add_trace(go.Scattermap(lat=[dlat]+lats+[dlat],lon=[dlon]+lons+[dlon],
            mode="lines",line=dict(width=2.5,color=color),name=label,showlegend=False,hoverinfo="skip"))
        fig.add_trace(go.Scattermap(lat=lats,lon=lons,mode="markers+text",
            marker=dict(size=12,color=color,opacity=0.93),text=[str(s) for s in range(1,n_s+1)],
            textfont=dict(size=8,color="white"),textposition="middle center",
            name=label,showlegend=True,hovertext=hovers,hoverinfo="text"))
    if not all_lats: return None
    fig.add_trace(go.Scattermap(lat=[dlat],lon=[dlon],mode="markers+text",
        marker=dict(size=20,color="#111",symbol="star"),text=[shl_name],textposition="top right",
        textfont=dict(size=12,color="#111",weight="bold"),name=f"{shl_name} depot",showlegend=True))
    clat=sum(all_lats)/len(all_lats); clon=sum(all_lons)/len(all_lons)
    fig.update_layout(map=dict(style="carto-positron",center=dict(lat=clat,lon=clon),zoom=7),
        title=dict(text=f"Baseline — {shl_name} SHL<br><sup style='color:#555'>{subtitle}</sup>",
                   font=dict(size=17,color="#111"),x=0.02),height=780,
        legend=dict(font=dict(size=10),bgcolor="rgba(255,255,255,0.9)",bordercolor="#ccc",
                    borderwidth=1,x=0.01,y=0.99,xanchor="left",yanchor="top"),
        margin=dict(t=85,b=10,l=10,r=10),paper_bgcolor="white")
    return fig

print("Map functions defined.")

Map functions defined.


In [13]:
def compute_kpis(routes_list, data, label=""):
    """Compute KPIs (mileage, CO2 by vehicle type, box-capacity use, integrity
    margins, blood age, EV feasibility) for a set of routes."""
    n_routes=len(routes_list); total_mi=sum(r["km"] for r in routes_list)
    n_multi=sum(1 for r in routes_list if len(r["hospitals"])>1)
    mean_stops=sum(len(r["hospitals"]) for r in routes_list)/max(n_routes,1)
    total_co2=sum(r.get("co2_kg",r["km"]*VEHICLE_TYPES["Diesel Van"]["co2_kg_per_mile"])
                  for r in routes_list)
    co2_by_type={}
    for r in routes_list:
        tn=r.get("vtype","Diesel Van")
        co2_by_type[tn]=co2_by_type.get(tn,0.0)+r.get("co2_kg",0.0)
    used_by_type={}
    for r in routes_list:
        tn=r.get("vtype","Diesel Van")
        used_by_type[tn]=used_by_type.get(tn,0)+1
    box_utils=[r["box_utilisation"] for r in routes_list if r.get("box_utilisation") is not None]
    mean_box_util=sum(box_utils)/len(box_utils) if box_utils else None
    max_box_util=max(box_utils) if box_utils else None
    integrity_margins=[U-r["dur_h"] for r in routes_list if r.get("dur_h")]
    mean_int=sum(integrity_margins)/len(integrity_margins) if integrity_margins else None
    min_int=min(integrity_margins) if integrity_margins else None
    blood_ages=[]
    for r in routes_list:
        dep_h=r.get("depart_h") or data.get("T_DEP")
        if dep_h is None: continue
        stops=r.get("stops") or [dict(pc=pc) for pc in r["hospitals"]]
        for stop in stops:
            ah=stop.get("arrival_h")
            if ah:
                age=(ah-dep_h)*60
                if 0<=age<=480: blood_ages.append(age)
    mean_age=sum(blood_ages)/len(blood_ages) if blood_ages else None
    max_age=max(blood_ages) if blood_ages else None
    n_electric=used_by_type.get("Electric Van",0)
    ev_rate=n_electric/n_routes if n_routes else 0.0
    kpis=dict(n_routes=n_routes,n_multi_stop=n_multi,mean_stops=round(mean_stops,2),
              total_mi=round(total_mi,1),total_co2_kg=round(total_co2,1),
              co2_by_type={k:round(v,1) for k,v in co2_by_type.items()},
              used_by_type=used_by_type,
              mean_box_utilisation=round(mean_box_util,3) if mean_box_util is not None else None,
              max_box_utilisation=round(max_box_util,3) if max_box_util is not None else None,
              mean_integrity_h=round(mean_int,2) if mean_int else None,
              min_integrity_h=round(min_int,2) if min_int else None,
              mean_blood_age_min=round(mean_age,1) if mean_age else None,
              max_blood_age_min=round(max_age,1) if max_age else None,
              ev_feasibility_rate=round(ev_rate,3),n_ev_used=n_electric)
    W2=58
    print(f"\n  {'─'*W2}\n  KPI Report — {label}\n  {'─'*W2}")
    print(f"  {'Routes (total/multi)':<35} {n_routes:>5} / {n_multi}")
    print(f"  {'Mean stops':<35} {mean_stops:>6.2f}")
    print(f"  {'Total distance (mi)':<35} {total_mi:>7.1f}")
    print(f"  {'Total CO2 (kg CO2e/wk)':<35} {total_co2:>7.1f}")
    for tn,kg in co2_by_type.items():
        n_used=used_by_type.get(tn,0)
        print(f"    {'of which '+tn:<33} {kg:>7.1f}   ({n_used} route(s), "
              f"{VEHICLE_TYPES[tn]['co2_kg_per_mile']:.3f} kg/mi)")
    if mean_box_util is not None:
        print(f"  {'Mean box-capacity utilisation':<35} {mean_box_util:>6.1%}")
        print(f"  {'Max box-capacity utilisation':<35} {max_box_util:>6.1%}  "
              f"({'✓' if max_box_util<=1.0 else '⚠ EXCEEDS capacity — check solve() constraints'})")
    if mean_int: print(f"  {'Mean integrity margin':<35} {mean_int*60:>5.0f} min")
    if min_int:  print(f"  {'Min integrity margin':<35} {min_int*60:>5.0f} min  ({'✓' if min_int>1 else '⚠ <1h'})")
    if mean_age: print(f"  {'Mean blood age (min)':<35} {mean_age:>5.0f}")
    if max_age:  print(f"  {'Max blood age (min)':<35} {max_age:>5.0f}")
    print(f"  {'Electric Van usage':<35} {ev_rate:>6.1%}   ({n_electric}/{n_routes} routes)")
    print(f"  {'─'*W2}")
    return kpis

def compare_kpis(base_kpis, opt_kpis):
    W2=60
    print(f"\n  {'═'*W2}\n  KPI COMPARISON — Baseline vs Optimised\n  {'═'*W2}")
    print(f"  {'Metric':<30} {'Baseline':>10} {'Optimised':>10} {'Δ':>8}  Note")
    print(f"  {'─'*W2}")
    def row(label,bk,ok,fmt=".1f",lower_better=True):
        bv=base_kpis.get(bk); ov=opt_kpis.get(ok)
        if bv is None or ov is None:
            print(f"  {label:<30} {'n/a':>10} {'n/a':>10} {'—':>8}"); return
        delta=ov-bv; d_str=f"{delta:{fmt}}"
        flag="✓" if (delta<0)==lower_better else ("✗" if delta!=0 else "=")
        bv_str=format(bv,fmt); ov_str=format(ov,fmt); d_str=format(delta,fmt)
        print(f"  {label:<30} {bv_str:>10} {ov_str:>10} {d_str:>8}  {flag}")
    row("Total distance (mi)","total_mi","total_mi",lower_better=True)
    row("Routes","n_routes","n_routes","d",lower_better=True)
    row("Multi-stop routes","n_multi_stop","n_multi_stop","d",lower_better=False)
    row("Mean stops/route","mean_stops","mean_stops",".2f",lower_better=False)
    row("CO2 (kg/wk)","total_co2_kg","total_co2_kg",lower_better=True)
    row("Mean box utilisation","mean_box_utilisation","mean_box_utilisation",".1%",lower_better=False)
    row("Mean integrity (h)","mean_integrity_h","mean_integrity_h",".2f",lower_better=False)
    row("Min integrity (h)","min_integrity_h","min_integrity_h",".2f",lower_better=False)
    row("Mean blood age (min)","mean_blood_age_min","mean_blood_age_min",".0f",lower_better=True)
    row("Max blood age (min)","max_blood_age_min","max_blood_age_min",".0f",lower_better=True)
    row("Electric Van usage","ev_feasibility_rate","ev_feasibility_rate",".1%",lower_better=False)
    print(f"  {'═'*W2}")
    print(f"  NOTE: baseline CO2/vehicle-type figures assume every historical round used a\n"
          f"  'Diesel Van' (no historical vehicle-type record exists) — see\n"
          f"  baseline_routes, . Optimised figures reflect the model's actual\n"
          f"  per-route vehicle-type assignment.")

def save_results_row(shl_name, data, result, baseline_metrics, solve_time_s, out_dir):
    csv_path=out_dir/"all_shls_results.csv"; bm=baseline_metrics or {}
    row=dict(SHL=shl_name,n_hospitals=len(data["ids"]),
             opt_total_mi=result["total_km"],opt_total_co2_kg=result.get("total_co2_kg"),
             opt_n_routes=result["n_routes"],
             opt_n_vehicles=result["n_vehicles"],opt_used_by_type=str(result.get("used_by_type")),
             opt_fleet_composition=str(result.get("fleet_composition")),
             opt_C_baseline=result["C_baseline"],
             opt_Cd=result["Cd"],opt_mip_gap_pct=result["mip_gap_pct"],
             opt_solve_time_s=round(solve_time_s,1),
             
             # --- EV Feasibility Thresholds ---
             ev_feasible_share=result.get("ev_feasible_share", ""),
             ev_feas_25_pct=result.get("ev_feas_25_pct", ""),
             ev_feas_50_pct=result.get("ev_feas_50_pct", ""),
             ev_feas_75_pct=result.get("ev_feas_75_pct", ""),
             ev_feas_100_pct=result.get("ev_feas_100_pct", ""),
             
             base_total_mi=bm.get("total_km",""),base_n_routes=bm.get("n_routes",""),
             base_n_multi_stop=bm.get("n_multi_stop",""),base_mean_stops=bm.get("mean_stops",""),
             mi_saving=(round(bm.get("total_km",0)-result["total_km"],1) if bm.get("total_km") else ""),
             mi_saving_pct=(round((bm.get("total_km",0)-result["total_km"])/bm.get("total_km",1)*100,1)
                            if bm.get("total_km") else ""))
    df_new=pd.DataFrame([row])
    if csv_path.exists():
        df_ex=pd.read_csv(csv_path); df_ex=df_ex[df_ex["SHL"]!=shl_name]
        df_out=pd.concat([df_ex,df_new],ignore_index=True)
    else:
        df_out=df_new
    df_out.to_csv(csv_path,index=False)
    print(f"  Results saved -> {csv_path.name}")
    return row

print("KPI and results functions defined (v2: CO2 by vehicle type, box utilisation).")

KPI and results functions defined (v2: CO2 by vehicle type, box utilisation).


In [14]:
VAQTEC_SPEC = {
    "Large": {
        "Blood":    (12, 9.0),   
        "Platelet": (15, 8.0),   
        "Other":    (10, 8.0),   
        "Frozen":   None,        # RESTORED: Large boxes physically do not hold Frozen
    },
    "Medium": {
        "Blood":    (15, 3.0),   
        "Platelet": (5, 5.0),   
        "Frozen":   (10, 9.5),   
        "Other":    (10, 8.0),   
    },
    "Small": {
        "Blood":    (6, 5.5),    
        "Platelet": (7, 7.0),    
        "Frozen":   (4, 11.0),   
        "Other":    None,        
    },
}

# STRUCTURAL FIX: Vehicles now map to an array of permitted containers
VEHICLE_VAQTEC_SIZES = {
    "Diesel Van":   ["Large", "Medium", "Small"],
    "Car":          ["Medium", "Small"],
    "Electric Van": ["Large", "Medium", "Small"]
}

# Restored missing function needed by the KPI exporter, updated for array logic
def container_shelf_life_hours(vtype, product):
    """Real DAT48/14 max hours out of temperature-controlled storage for the
    vehicle type carrying product g, evaluating against an array of mixed containers."""
    allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype, [])
    if not allowed_sizes:
        return None
        
    best_h = -1
    for size in allowed_sizes:
        entry = VAQTEC_SPEC.get(size, {}).get(product)
        if entry is not None:
            best_h = max(best_h, entry[1])
            
    return best_h if best_h != -1 else None

VAQTEC_PACKING_SPECS = {
    "Small": {
        "Red Blood Cells": {"min": 1, "max": 6,  "config_max": "Laid flat, ports folded, 2 stacks x 3 units"},
        "Platelets":       {"min": 1, "max": 6,  "config_max": "Laid flat, 1 stack x 6 units"},
        "FFP":             {"min": 1, "max": 4,  "config_max": "Laid flat, 1 stack x 4 units"}
    },
    "Medium": {
        "Red Blood Cells": {"min": 1, "max": 15, "config_max": "Laid flat, ports folded, 3 stacks x 4 units + 1 stack x 3 units"},
        "Platelets":       {"min": 1, "max": 15, "config_max": "Laid flat, 1 stack x 15 units"},
        "FFP":             {"min": 1, "max": 10, "config_max": "Laid flat, 1 stack x 10 units"}
    },
    "Large": {
        "Red Blood Cells": {"min": 1, "max": 12, "config_max": "Laid flat, ports folded, 2 stacks x 6 units"},
        "Platelets":       {"min": 1, "max": 15, "config_max": "Laid flat, 1 stack x 15 units"},
        "FFP":             None  
    }
}

def get_packing_config(vtype_name, product):
    """Retrieve the DAT48/14 packing configuration for a given vehicle type and product,
    scanning through permitted containers."""
    allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype_name, [])
    if not allowed_sizes: 
        return None
        
    for size in allowed_sizes:
        specs = VAQTEC_PACKING_SPECS.get(size, {}).get(product)
        if specs is not None:
            return specs
            
    return None

DONATION_VAQTEC_SPEC = {
    "Large": {
        "CD Platelets":       (7, 8.0),  
        "Donated Plasma":     (7, 8.0),  
        "Whole Blood":        (4, 8.0),  
    },
    "Medium": {
        "CD Platelets":       (7, 5.0),
        "Donated Plasma":     (7, 5.0),  
        "Whole Blood":        (9, 3.5),  
    },
    "Small": {
        "CD Platelets":       (3, 7.0),
        "Donated Plasma":     (3, 7.0),  
        "Whole Blood":        (3, 5.0),  
    },
}

PRODUCT_STORAGE_SHELF_LIFE_DAYS = {   
    "Blood":    35,                    
    "Platelet": 7,
    "Frozen":   365,
    "Other":    14,
}
SAFETY_STOCK_DAYS        = 1.5    
DEMAND_CV                = 0.35   
AD_HOC_CALLOUT_COST_GBP  = 85.0   
AD_HOC_COST_PER_MILE_GBP = 1.20   
N_SIM_TRIALS              = 200    
RUN_POST_HOC_SIMULATION   = True   

print("Post-hoc constants loaded (v3: structural array containers, packing configs, restored shelf-life function).")

Post-hoc constants loaded (v3: structural array containers, packing configs, restored shelf-life function).


In [15]:
def container_ceiling_for_products(vtype_name, products):
    """DAT48/14 hours for `vtype_name` over the products actually carried,
    evaluating against an array of mixed containers."""
    allowed_sizes = VEHICLE_VAQTEC_SIZES.get(vtype_name, [])
    if not allowed_sizes:
        return min((PERISH_HOURS[g] for g in products if g in PERISH_HOURS),
                   default=max(PERISH_HOURS.values()))
    
    hours = []
    for g in products:
        best_h = -1
        # Find the best valid container for this specific product
        for size in allowed_sizes:
            entry = VAQTEC_SPEC.get(size, {}).get(g)
            if entry is not None:
                best_h = max(best_h, entry[1])
        
        if best_h == -1: 
            return None # No allowed container can carry this product
            
        hours.append(min(best_h, PERISH_HOURS.get(g, best_h)))
        
    return min(hours) if hours else max(PERISH_HOURS.values())
 
 
def loosest_container_ceiling(products):
    """Best case over all vehicle types — used while the vehicle is unbound
    (CW / ICW / local search), so those heuristics do not discard a route that
    a Car could legally run."""
    cands = [container_ceiling_for_products(vt, products)
             for vt in VEHICLE_VAQTEC_SIZES]  # FIX: Now points to the new SIZES array
    cands = [x for x in cands if x is not None]
    if cands:
        return max(cands)
    return min((PERISH_HOURS[g] for g in products if g in PERISH_HOURS),
               default=max(PERISH_HOURS.values()))


MAX_CONTAINER_CEILING_H = max(
    [loosest_container_ceiling({g}) for g in PERISH_HOURS]
    + [max(PERISH_HOURS.values())])


def products_on_route(route, t_day, demand, lnodes):
    """Products with strictly positive demand anywhere on the route."""
    out = set()
    for nd in route:
        pc = lnodes[nd - 1][0]
        for g in PRODUCTS:
            if demand.get((pc, g, t_day), 0.0) > 0:
                out.add(g)
    return out

def _iter_routes_with_day(routes_input):
    if isinstance(routes_input, dict):
        return [dict(r, _day=t) for t, rs in routes_input.items() for r in rs]
    return list(routes_input)

def _window_close(data, pc, day, slot_name):
    """Return the assigned slot's window-close (deadline) hour for one stop."""
    for sn, oh, ch in data.get("slot_windows", {}).get(pc, {}).get(day, []):
        if sn == slot_name:
            return ch
    return None

def container_integrity_check(routes_input, data, label=""):
    """Post-hoc check: recompute each delivery's integrity margin against the
    REAL DAT48/14 time-out-of-storage ceiling for its vehicle/product, and
    flag any delivery that breaches it. Reporting only, not a constraint."""
    rows = []
    for r in _iter_routes_with_day(routes_input):
        day = r.get("_day")
        dur_h = r.get("dur_h")
        if dur_h is None or day is None:
            continue
        vtype = r.get("vtype", "Diesel Van")
        products_on_route = sorted({
            g for pc in r["hospitals"] for g in PRODUCTS
            if data.get("demand", {}).get((pc, g, day), 0.0) > 0
        }) or ["Blood"]  # conservative fallback if no demand record exists

        ceilings = {g: container_shelf_life_hours(vtype, g) for g in products_on_route}
        incompatible = sorted(g for g, h in ceilings.items() if h is None)
        numeric_ceilings = [h for h in ceilings.values() if h is not None]
        true_ceiling = min(numeric_ceilings) if numeric_ceilings else None

        flat_margin_h = U - dur_h
        true_margin_h = (true_ceiling - dur_h) if true_ceiling is not None else None
        hidden_breach = (true_margin_h is not None and flat_margin_h >= 0 and true_margin_h < 0)

        rows.append(dict(
            vehicle=r.get("vehicle"), day=day, vtype=vtype, dur_h=dur_h,
            products=products_on_route, incompatible=incompatible,
            flat_margin_h=round(flat_margin_h, 2),
            true_ceiling_h=round(true_ceiling, 2) if true_ceiling is not None else None,
            true_margin_h=round(true_margin_h, 2) if true_margin_h is not None else None,
            hidden_breach=hidden_breach,
        ))

    n_breach = sum(1 for x in rows if x["hidden_breach"])
    n_incompat = sum(1 for x in rows if x["incompatible"])
    print(f"\n  Container-specific integrity check — {label} "
          f"(DAT48/14, real container/product shelf life)")
    print(f"    Routes checked: {len(rows)}")
    print(f"    Hidden breaches (pass flat U=8.0h, fail real DAT48/14 ceiling): {n_breach}")
    print(f"    Container/product incompatibilities (product not specified for "
          f"assigned container size): {n_incompat}")
    if n_breach:
        print(f"    {'Route':<10} {'Day':<5} {'Vehicle type':<20} {'dur_h':>6} "
              f"{'flat_margin':>12} {'true_ceiling':>13} {'true_margin':>12}")
        for x in rows:
            if x["hidden_breach"]:
                print(f"    {str(x['vehicle']):<10} {DAY_NAMES.get(x['day'],'?'):<5} "
                      f"{x['vtype']:<20} {x['dur_h']:>6.2f} {x['flat_margin_h']:>12.2f} "
                      f"{x['true_ceiling_h']:>13.2f} {x['true_margin_h']:>12.2f}")
    if n_incompat:
        for x in rows:
            if x["incompatible"]:
                print(f"    ⚠ {x['vehicle']} ({DAY_NAMES.get(x['day'],'?')}): "
                      f"{x['vtype']} not specified in DAT48/14 for {x['incompatible']}")
    return dict(rows=rows, n_hidden_breach=n_breach, n_incompatible=n_incompat)

def delivery_window_quality(routes_input, data, label=""):
    """Post-hoc only: report early/late arrivals vs each slot window and the
    slack distribution. Not part of the objective."""
    waits, buffers = [], []
    for r in _iter_routes_with_day(routes_input):
        day = r.get("_day")
        for stop in r.get("stops") or []:
            arr = stop.get("arrival_h")
            raw = stop.get("raw_arrival_h")
            if arr is not None and raw is not None:
                waits.append(max(0.0, (arr - raw) * 60.0))
            ch = _window_close(data, stop.get("pc"), day, stop.get("slot"))
            if ch is not None and arr is not None:
                buffers.append((ch - arr) * 60.0)

    def _stats(vals):
        if not vals: return None
        return dict(mean=round(sum(vals) / len(vals), 1), min=round(min(vals), 1),
                    max=round(max(vals), 1), n=len(vals))

    w_stats, b_stats = _stats(waits), _stats(buffers)
    print(f"\n  Delivery time window quality — {label}")
    if w_stats:
        print(f"    Waiting time (min): mean {w_stats['mean']}  max {w_stats['max']}  "
              f"(n={w_stats['n']} stops; solved A/E fields — optimised routes only)")
    else:
        print("    Waiting time: n/a (no raw_arrival_h on these routes — baseline has no "
              "solved wait variable)")
    if b_stats:
        n_tight = sum(1 for b in buffers if b < 15.0)
        print(f"    Buffer to window close (min): mean {b_stats['mean']}  "
              f"min {b_stats['min']}  (n={b_stats['n']} stops)")
        print(f"    Stops within 15 min of their window close: {n_tight}/{b_stats['n']}  "
              f"({n_tight/b_stats['n']:.1%}) — these are the ones a small upstream delay "
              f"would tip into a late arrival")
    else:
        print("    Buffer to window close: n/a (no matching slot_windows entry for these stops)")
    return dict(wait_stats=w_stats, buffer_stats=b_stats)

def route_stability_index(baseline_trips, optimised_routes_by_day, label=""):
    """Fraction of hospital-day assignments whose route composition is stable
    across the week (a simple robustness proxy)."""
    def _index_by_hosp_day(trips):
        idx = {}
        for r in trips:
            day = r.get("_day")
            hosp_seq = r.get("hospitals", [])
            for pos, pc in enumerate(hosp_seq):
                idx.setdefault((pc, day), (pos, hosp_seq))
        return idx

    base_idx = _index_by_hosp_day(baseline_trips)
    opt_idx = _index_by_hosp_day(_iter_routes_with_day(optimised_routes_by_day))
    common = sorted(set(base_idx) & set(opt_idx))

    unchanged_mates = jaccards = seq_matches = 0
    for key in common:
        pos_b, seq_b = base_idx[key]
        pos_o, seq_o = opt_idx[key]
        mates_b = set(seq_b) - {key[0]}
        mates_o = set(seq_o) - {key[0]}
        if mates_b == mates_o:
            unchanged_mates += 1
        union = mates_b | mates_o
        jaccards += (len(mates_b & mates_o) / len(union)) if union else 1.0
        if pos_b == pos_o:
            seq_matches += 1

    n = len(common)
    print(f"\n  Route stability index — {label}")
    if n == 0:
        print("    n/a — no overlapping hospital-day keys between baseline and optimised "
              "(check that both were computed for the same SHL/week)")
        return None
    stability = unchanged_mates / n
    mean_jaccard = jaccards / n
    seq_stability = seq_matches / n
    print(f"    Hospital-days compared: {n}")
    print(f"    Unchanged route-mates (strict):  {stability:.1%}")
    print(f"    Mean route-mate similarity (Jaccard): {mean_jaccard:.1%}")
    print(f"    Unchanged sequence position:     {seq_stability:.1%}")
    return dict(n=n, stability_index=round(stability, 4),
                mean_jaccard=round(mean_jaccard, 4), sequence_stability=round(seq_stability, 4))

def fleet_utilisation_rate(routes_input, fleet_comp, label=""):
    """Total vehicle-hours and box-capacity consumed across dispatched routes."""
    routes = _iter_routes_with_day(routes_input)
    n_days = len(DAYS)
    Cd = sum(fleet_comp.values())
    dispatch_slots_available = Cd * n_days
    n_dispatches_used = len(routes)

    box_capacity_available = sum(
        cnt * VEHICLE_TYPES[tname]["boxes_capacity"] for tname, cnt in fleet_comp.items()
    ) * n_days
    boxes_used = [r.get("boxes_est") for r in routes if r.get("boxes_est") is not None]
    box_capacity_used = sum(boxes_used) if boxes_used else None

    vehicle_hours_available = Cd * n_days * U
    hours_used = [r.get("dur_h") for r in routes if r.get("dur_h") is not None]
    vehicle_hours_used = sum(hours_used) if hours_used else None

    dispatch_util = n_dispatches_used / dispatch_slots_available if dispatch_slots_available else None
    box_util = (box_capacity_used / box_capacity_available) if (
        box_capacity_used is not None and box_capacity_available) else None
    hours_util = (vehicle_hours_used / vehicle_hours_available) if (
        vehicle_hours_used is not None and vehicle_hours_available) else None

    print(f"\n  Fleet utilisation rate — {label}")
    print(f"    Dispatch-slot utilisation:  {n_dispatches_used}/{dispatch_slots_available}"
          + (f"  ({dispatch_util:.1%})" if dispatch_util is not None else ""))
    if box_util is not None:
        print(f"    Box-capacity utilisation:   {box_capacity_used:.0f}/{box_capacity_available:.0f}"
              f"  ({box_util:.1%})")
    else:
        print("    Box-capacity utilisation:   n/a (route boxes_est not available on this side)")
    if hours_util is not None:
        print(f"    Vehicle-hours utilisation:  {vehicle_hours_used:.1f}/{vehicle_hours_available:.1f}h"
              f"  ({hours_util:.1%})")
    return dict(dispatch_utilisation=round(dispatch_util, 4) if dispatch_util is not None else None,
                box_capacity_utilisation=round(box_util, 4) if box_util is not None else None,
                vehicle_hours_utilisation=round(hours_util, 4) if hours_util is not None else None)

def simulate_service_level(data, n_trials=N_SIM_TRIALS, seed=42, label=""):
    """PLACEHOLDER Monte-Carlo service-level scaffold (not yet calibrated)."""
    rng = np.random.default_rng(seed)
    hospitals = data["ids"]; demand = data.get("demand", {})
    hlat, hlon, dlat, dlon = data["hlat"], data["hlon"], data["dlat"], data["dlon"]
    visit_days = data.get("visit_days", {})

    records = []
    for pc in hospitals:
        depot_mi = road_mi(dlat, dlon, hlat[pc], hlon[pc])
        v_days = sorted(visit_days.get(pc, []))
        if not v_days:
            continue
        for g in PRODUCTS:
            week_demand = {t: demand.get((pc, g, t), 0.0) for t in DAYS}
            total = sum(week_demand.values())
            if total <= 0:
                continue
            mean_daily = total / len(DAYS)
            shelf_days = PRODUCT_STORAGE_SHELF_LIFE_DAYS.get(g, 14)
            sigma = math.sqrt(math.log(1 + DEMAND_CV ** 2))
            mu = math.log(max(mean_daily, 1e-6)) - 0.5 * sigma ** 2

            stockout_days = total_days = ad_hoc_events = 0
            wasted_units = delivered_units = 0.0

            for _ in range(n_trials):
                stock = mean_daily * SAFETY_STOCK_DAYS
                buckets = []  # [units, age_days], FIFO
                trial_ad_hoc = False
                for t in DAYS:
                    if t in v_days:
                        future = [d for d in v_days if d > t]
                        gap = (min(future) - t) if future else (5 - t + min(v_days))
                        gap = max(1, gap)
                        target = mean_daily * (gap + SAFETY_STOCK_DAYS)
                        delivered = max(0.0, target - stock)
                        stock += delivered
                        delivered_units += delivered
                        if delivered > 0:
                            buckets.append([delivered, 0])
                    consumption = float(rng.lognormal(mean=mu, sigma=sigma))
                    total_days += 1
                    if consumption > stock:
                        stockout_days += 1
                        trial_ad_hoc = True
                        stock = mean_daily * SAFETY_STOCK_DAYS
                        buckets = [[stock, 0]] if stock > 0 else []
                    else:
                        stock -= consumption
                        remaining = consumption
                        kept = []
                        for units, age in buckets:
                            if remaining >= units:
                                remaining -= units
                                continue
                            kept.append([units - remaining, age])
                            remaining = 0.0
                        buckets = [[u, a + 1] for u, a in kept]
                        surviving = []
                        for units, age in buckets:
                            if age > shelf_days:
                                wasted_units += units
                            else:
                                surviving.append([units, age])
                        buckets = surviving
                if trial_ad_hoc:
                    ad_hoc_events += 1

            ad_hoc_rate = ad_hoc_events / max(n_trials, 1)
            records.append(dict(
                pc=pc, product=g, mean_daily_demand=round(mean_daily, 2),
                stockout_rate=round(stockout_days / max(total_days, 1), 4),
                ad_hoc_trigger_rate=round(ad_hoc_rate, 4),
                wastage_rate=round(wasted_units / max(delivered_units, 1e-6), 4),
                ad_hoc_cost_gbp=round(ad_hoc_rate * (AD_HOC_CALLOUT_COST_GBP +
                                       AD_HOC_COST_PER_MILE_GBP * 2 * depot_mi), 2),
                depot_mi=round(depot_mi, 1),
            ))

    if not records:
        print(f"\n  Service-level / wastage / Ad-Hoc simulation — {label}\n"
              "    n/a — no (hospital, product) combination had modelled demand")
        return dict(records=[], summary=None)

    mean_stockout = sum(r["stockout_rate"] for r in records) / len(records)
    mean_wastage_num = sum(r["wastage_rate"] * r["mean_daily_demand"] for r in records)
    mean_wastage_den = sum(r["mean_daily_demand"] for r in records)
    weighted_wastage = mean_wastage_num / mean_wastage_den if mean_wastage_den else 0.0
    total_ad_hoc_cost = sum(r["ad_hoc_cost_gbp"] for r in records)
    worst = sorted(records, key=lambda x: -x["stockout_rate"])[:5]

    print(f"\n  Service-level / wastage / Ad-Hoc simulation — {label}  "
          f"({n_trials} Monte Carlo trials/hospital-product, PLACEHOLDER calibration)")
    print(f"    Mean stockout rate (demand-days): {mean_stockout:.2%}")
    print(f"    Demand-weighted wastage rate:     {weighted_wastage:.2%}")
    print(f"    Estimated weekly Ad-Hoc cost displacement: £{total_ad_hoc_cost:,.0f}")
    print(f"    Highest stockout-risk hospital-products:")
    for r in worst:
        print(f"      {r['pc']:<10} {r['product']:<9} stockout={r['stockout_rate']:.1%}  "
              f"ad_hoc_rate={r['ad_hoc_trigger_rate']:.1%}  £{r['ad_hoc_cost_gbp']:.0f}/wk")

    return dict(records=records, summary=dict(
        mean_stockout_rate=round(mean_stockout, 4),
        weighted_wastage_rate=round(weighted_wastage, 4),
        total_ad_hoc_cost_gbp=round(total_ad_hoc_cost, 2),
    ))


def bsms_minimum_service_check(routes_input, data, label=""):
    """Post-hoc check on the BSMS minimum-service-level tier floor.

    For each hospital with a BSMS worst-case category (loaded from the pipeline output
    into `data['bsms_min_visits']`), count the number of weekdays on which it is served
    in the solved schedule and compare against the tier's implied floor. Reports the
    number of hospitals meeting / falling short of the floor, and the specific
    below-floor cases so they can be inspected. Left as a reporting flag rather than
    a hard constraint because a hospital whose schedule itself provides fewer weekday
    slots than its tier floor demands would otherwise force spurious infeasibility --
    when this happens it means the survey and the operational schedule disagree, and
    the routing model is not the right layer to reconcile them.
    """
    min_visits = data.get("bsms_min_visits", {}) or {}
    tier_lu    = data.get("bsms_tier", {}) or {}
    if not min_visits:
        return None
    visits_by_hosp = {}
    for r in _iter_routes_with_day(routes_input):
        for pc in r.get("hospitals", []):
            visits_by_hosp[pc] = visits_by_hosp.get(pc, 0) + 1
    checked = 0; below = []; met = 0
    for pc, floor in min_visits.items():
        if pc not in data.get("ids", []): continue
        checked += 1
        actual = visits_by_hosp.get(pc, 0)
        if actual < floor:
            below.append((pc, tier_lu.get(pc, "?"), floor, actual))
        else:
            met += 1
    print(f"\n  BSMS minimum-service-level check -- {label}")
    print(f"    Categorised hospitals: {checked}  |  meeting floor: {met}  |  below floor: {len(below)}")
    if below:
        for pc, tier, floor, actual in sorted(below, key=lambda r: (-r[2], r[0]))[:20]:
            print(f"    below-floor: {pc:<8} tier={tier:<10} floor={floor}/week  optimised={actual}/week")
        if len(below) > 20:
            print(f"    ... {len(below)-20} more below-floor cases (see solved schedule for detail)")
    return dict(checked=checked, meeting=met, below_floor=len(below),
                below_detail=[(pc, tier, floor, actual) for pc, tier, floor, actual in below])


def post_hoc_evaluation(shl_name, result, bsl_metrics, data, run_simulation=RUN_POST_HOC_SIMULATION,
                         n_sim_trials=N_SIM_TRIALS):
    """Run the full post-hoc evaluation layer on an already-solved result."""
    W = 66
    print(f"\n  {'='*W}\n  POST-HOC EVALUATION — {shl_name}\n  {'='*W}")

    opt_by_day = result["routes_by_day"]
    bsl_trips = (bsl_metrics or {}).get("all_trips", [])

    out = {}
    out["integrity_opt"] = container_integrity_check(opt_by_day, data, label=f"{shl_name} — Optimised")
    if bsl_trips:
        out["integrity_base"] = container_integrity_check(bsl_trips, data, label=f"{shl_name} — Baseline")

    out["window_opt"] = delivery_window_quality(opt_by_day, data, label=f"{shl_name} — Optimised")
    if bsl_trips:
        out["window_base"] = delivery_window_quality(bsl_trips, data, label=f"{shl_name} — Baseline")

    if bsl_trips:
        out["stability"] = route_stability_index(bsl_trips, opt_by_day, label=shl_name)

    out["bsms_min_service"] = bsms_minimum_service_check(
        opt_by_day, data, label=f"{shl_name} -- Optimised")
    out["fleet_util_opt"] = fleet_utilisation_rate(
        opt_by_day, result.get("fleet_composition", {}), label=f"{shl_name} — Optimised")
    if bsl_metrics:
        base_fleet_comp = {"Diesel Van": data["C_baseline"]}
        out["fleet_util_base"] = fleet_utilisation_rate(
            bsl_trips, base_fleet_comp, label=f"{shl_name} — Baseline")

    if run_simulation:
        out["simulation"] = simulate_service_level(data, n_trials=n_sim_trials, label=shl_name)
    else:
        print("\n  Service-level / wastage / Ad-Hoc simulation — skipped "
              "(RUN_POST_HOC_SIMULATION=False)")

    print(f"\n  {'='*W}")
    return out

print("Post-hoc evaluation functions defined (container integrity, window quality, "
      "stability index, fleet utilisation, service-level simulation).")

Post-hoc evaluation functions defined (container integrity, window quality, stability index, fleet utilisation, service-level simulation).


In [16]:
import glob
import math
import numpy as np
import pandas as pd

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================

PATTERN = "/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6/Analysis_output/vanilla_v26_complete_changes_copy_diesel_van/kpi_stops_*_pooled.csv"
DWELL_MIN = 15.0  # SERVICE_BASE_MIN; SERVICE_PER_BOX_MIN = 0
GRID = [0.0, 0.005, 0.025, 0.05, 0.10, 0.15, 0.25]
SHL_ORDER = [
    "Plymouth", "Lancaster", "Filton", "Southampton",
    "Newcastle", "Basildon", "Oxford", "Liverpool", "Cambridge",
    "Barnsley", "Birmingham", "Colindale", "Tooting", "Manchester"
]


# ==========================================
# DATA LOADING & PROCESSING
# ==========================================

def load_stops(pattern=PATTERN):
    """Load and concatenate all pooled KPI stops CSVs."""
    frames = [pd.read_csv(f) for f in sorted(glob.glob(pattern))]
    d = pd.concat(frames, ignore_index=True)
    return d.sort_values(["SHL", "Day", "Round", "Seq"]).reset_index(drop=True)


def add_travel_component(d, dwell_min=DWELL_MIN):
    """Calculate the travel component and validate non-negative values."""
    d = d.copy()
    d["travel_min"] = d["Blood_Age_min"] - dwell_min * (d["Seq"] - 1)
    bad = (d["travel_min"] < 0).sum()
    if bad:
        raise ValueError(
            f"{bad} stops have a negative travel component — check that "
            f"Blood_Age_min is elapsed-since-departure and that dwell is {dwell_min} min."
        )
    return d


# ==========================================
# ANALYSIS & SENSITIVITY
# ==========================================

def network_band(d, grid=GRID, ref_mph=21.7):
    """Evaluate network-level sensitivity across a grid of speed reductions."""
    rows = []
    for x in grid:
        delay = d["travel_min"] * (1.0 / (1.0 - x) - 1.0)
        late = delay > d["Buffer_min"] + 1e-9
        excess = (delay - d["Buffer_min"]).clip(lower=0)
        rows.append({
            "speed_reduction": f"{x:.1%}",
            "effective_mph": round(ref_mph * (1 - x), 1),
            "late_stops": int(late.sum()),
            "pct_stops": round(100 * late.mean(), 1),
            "mean_excess_min": round(excess[late].mean(), 1) if late.any() else 0.0,
            "worst_min": round(excess.max(), 1),
            "rounds_affected": int(d[late].groupby(["SHL", "Day", "Round"]).ngroups),
        })
    return pd.DataFrame(rows)


def per_shl_band(d, levels=(0.05, 0.10)):
    """Evaluate per-SHL sensitivity at specific speed reduction levels."""
    out = pd.DataFrame({
        "stops": d.groupby("SHL").size(),
        "zero_buffer_pct": d.groupby("SHL")["Buffer_min"]
                            .apply(lambda z: round(100 * (z <= 0).mean(), 1)),
    })
    for x in levels:
        delay = d["travel_min"] * (1.0 / (1.0 - x) - 1.0)
        late = delay > d["Buffer_min"] + 1e-9
        excess = (delay - d["Buffer_min"]).clip(lower=0)
        tmp = d.assign(_late=late, _exc=excess).groupby("SHL")
        out[f"late_pct_{int(x*100)}"] = tmp["_late"].apply(lambda z: round(100 * z.mean(), 1))
        out[f"worst_min_{int(x*100)}"] = tmp["_exc"].max().round(1)
        
    return out.reindex([s for s in SHL_ORDER if s in out.index])


def upside_check(d, x=0.05):
    """A speed INCREASE can only hurt by arriving before the window opens."""
    f = 1.0 / (1.0 + x)
    new_arrival = d["Raw_Arrival_h"] - (d["travel_min"] * (1 - f)) / 60.0
    n = int((new_arrival < d["Window_Open_h"] - 1e-9).sum())
    return n


def audit_ceiling_change(kpi_csv, label=""):
    """
    Before/after check on one SHL: how many rounds were only feasible under
    the loose ceiling. Run on the CURRENT export before re-solving to predict
    how much the instance will tighten.
    """
    df = pd.read_csv(kpi_csv)
    n = len(df)
    breach = int(df["Hidden_Breach"].sum())
    
    print(f"  {label or kpi_csv}: {n} rounds, {breach} ({100*breach/n:.1f}%) "
          f"infeasible under DAT48/14 but feasible under the operational ceiling.")
          
    worst_margin = df['True_Margin_h'].min()
    print(f"    worst true margin {worst_margin:.2f} h — a round that "
          f"long must be split into at least "
          f"{math.ceil(abs(worst_margin) / 3.0) + 1} under a 3.0 h limit.")
          
    return breach


# ==========================================
# BSMS PATCHING & COMPLIANCE
# ==========================================

def patch_bsms_round_layer(df, below, data):
    """
    Add SHL-scoped BSMS columns to the per-round frame.
    Insert immediately after `df = _pd.DataFrame(raw)` in compute_kpis.
    """
    unserved = sorted(pc for pc in below if below[pc][2] == 0)  # (tier, floor, actual)
    df["BSMS_SHL_N_Below_Floor"] = len(below)
    df["BSMS_SHL_Unserved_Codes"] = "; ".join(unserved)
    df["BSMS_SHL_Compliant"] = (len(below) == 0)
    return df


def patch_bsms_stop_layer(hdf, below, data, visits_by_hosp, bsms_tier, bsms_floor,
                          shl_name, month_label):
    """
    Complete the stop-level export over the REQUIRED hospital set.
    Unserved required hospitals are appended with Served=False and Seq=NA.
    """
    hdf = hdf.copy()
    if "Served" not in hdf.columns:
        hdf.insert(hdf.columns.get_loc("Code") + 1, "Served", True)

    served = set(hdf["Code"])
    required = [pc for pc in data.get("ids", []) if pc not in served]
    if not required:
        return hdf

    rows = []
    for pc in sorted(required):
        rows.append({
            "SHL": shl_name, "Month": month_label, "Day": pd.NA,
            "Round": pd.NA, "Seq": pd.NA, "Code": pc, "Served": False,
            "Slot": pd.NA,
            "Visits_This_Week": visits_by_hosp.get(pc, 0),
            "BSMS_Tier": bsms_tier.get(pc, ""),
            "BSMS_Floor": bsms_floor.get(pc, pd.NA),
            "BSMS_Below_Floor": pc in below,
            "Arrival_Predictability_min": pd.NA,
            "Predictability_N_Visits": 0,
        })
        
    out = pd.concat([hdf, pd.DataFrame(rows)], ignore_index=True)
    n_un = len(rows)
    print(f"  ⚠ BSMS: {n_un} required hospital(s) received ZERO visits and are "
          f"appended to the stop export with Served=False: "
          f"{sorted(required)}")
    return out


def assert_bsms_compliance(summary, strict=False):
    """Optional hard gate. Call after compute_kpis returns the summary dict."""
    n = summary.get("bsms_below_floor", 0)
    if n:
        msg = (f"BSMS FAILURE — {summary.get('SHL')}: {n} hospital(s) below "
               f"floor: {summary.get('bsms_below_detail')}")
        if strict:
            raise AssertionError(msg)
        print("  ⚠ " + msg)
        return False
    return True


# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    d = add_travel_component(load_stops())
    
    print(f"{len(d)} stops, {d.SHL.nunique()} SHLs, "
          f"{d.groupby(['SHL','Day','Round']).ngroups} rounds\n")

    print("=== NETWORK SENSITIVITY BAND ===")
    print(network_band(d).to_string(index=False))

    print("\n=== PER-SHL ===")
    print(per_shl_band(d).to_string())

    r = np.corrcoef(
        per_shl_band(d)["zero_buffer_pct"],
        per_shl_band(d)["late_pct_5"]
    )[0, 1]
    
    print(f"\nr(zero-buffer share, late share at 5%) = {r:.2f}")
    print(f"stops arriving before window open under a 5% SPEED INCREASE: "
          f"{upside_check(d, 0.05)}")

1948 stops, 14 SHLs, 921 rounds

=== NETWORK SENSITIVITY BAND ===
speed_reduction  effective_mph  late_stops  pct_stops  mean_excess_min  worst_min  rounds_affected
           0.0%           21.7         117        6.0              0.4        1.8               92
           0.5%           21.6        1149       59.0              0.4        1.8              664
           2.5%           21.2        1190       61.1              2.0        6.9              674
           5.0%           20.6        1225       62.9              4.1       14.2              689
          10.0%           19.5        1294       66.4              8.6       30.0              722
          15.0%           18.4        1406       72.2             13.2       47.6              761
          25.0%           16.3        1538       79.0             25.2       90.0              808

=== PER-SHL ===
             stops  zero_buffer_pct  late_pct_5  worst_min_5  late_pct_10  worst_min_10
SHL                                  

## Section M — Frequency Extension Post-hoc Diagnostics

Per-SHL diagnostics called from `post_hoc_evaluation()`. Uses `FREQ_` and `_freq_` prefixes throughout so nothing collides with the solver.


In [17]:
# =============================================================================
# SECTION M -- Frequency Extension Post-hoc Diagnostics (v29)
# =============================================================================
# Three per-SHL diagnostics called from post_hoc_evaluation():
#   frequency_cascade()        -- planned vs actual vs demand-implied
#   frequency_classification() -- over/under/appropriate bucketing
#   redesign_saving()          -- projected £/yr from demand-derived f_i
#
# All names in this cell are prefixed with FREQ_ or _freq to prevent
# collisions with the solver's own DAY_NAMES, load_data, etc.
# =============================================================================

import pandas as _freq_pd
import numpy as _freq_np
from collections import defaultdict as _freq_defaultdict

FREQ_SAFE_CAP = 15                # DAT48/14 Medium/Blood ceiling (boxes)
FREQ_COST_PER_MI = 0.45           # £/mi diesel-van tariff
FREQ_SOLO_FILL_THRESHOLD = 0.10   # box utilisation below this = "low fill"
FREQ_WEEKS_PER_YEAR = 52


def _freq_find_col(df, candidates, label=""):
    """Case-insensitive column-name lookup with clear error."""
    lower = {c.lower().replace(' ', '_'): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().replace(' ', '_')
        if key in lower:
            return lower[key]
    for cand in candidates:
        for c in df.columns:
            if cand.lower() in c.lower():
                return c
    raise KeyError(
        f"Cannot find {label} column. Tried {candidates}. "
        f"Available: {list(df.columns)}")


def frequency_cascade(shl_name, result, data, util_df, demand_df,
                      safe_cap=FREQ_SAFE_CAP):
    """Per-hospital planned/actual/demand-implied frequency for one SHL."""
    hosp_ids = data.get("ids", [])

    # PLANNED: count visits per hospital in the solved routes
    planned = _freq_defaultdict(int)
    routes = result.get("routes_by_day", {})
    for _t, day_routes in routes.items():
        for route in day_routes:
            for node in route:
                if isinstance(node, str) and node in hosp_ids:
                    planned[node] += 1
                elif isinstance(node, tuple):
                    pc = node[0] if len(node) >= 1 else None
                    if pc in hosp_ids:
                        planned[pc] += 1

    # ACTUAL: from 12-month util_clean.csv, if provided
    actual = {}
    if util_df is not None and len(util_df):
        code_col = _freq_find_col(util_df,
            ['Pulse Code', 'pulse_code', 'Code'], label='util code')
        u = util_df.copy()
        if 'record_type' in u.columns:
            u = u[u['record_type'] == 'genuine_delivery']
        if 'is_weekend' in u.columns:
            u = u[u['is_weekend'] == False]
        n_weeks = u['week_num'].nunique() if 'week_num' in u.columns else 52
        u = u[u[code_col].isin(hosp_ids)]
        if n_weeks:
            actual = (u.groupby(code_col).size() / n_weeks).to_dict()

    # DEMAND-IMPLIED: from box demand, if provided
    demand_min = {}
    if demand_df is not None and len(demand_df):
        code_col = _freq_find_col(demand_df,
            ['Pulse Code', 'pulse_code', 'Code'], label='demand code')
        # Peak-day column: max_boxes (boxes_by_hospital.csv) or peak_day_boxes
        if 'max_boxes' in demand_df.columns:
            peak_col, per_hosp = 'max_boxes', True
        elif 'peak_day_boxes' in demand_df.columns:
            peak_col, per_hosp = 'peak_day_boxes', True
        elif 'boxes_est' in demand_df.columns:
            peak_col, per_hosp = 'boxes_est', False
        else:
            peak_col = next((c for c in demand_df.columns if 'box' in c.lower()), None)
            per_hosp = True
        if peak_col is None:
            demand_min = {pc: 5 for pc in hosp_ids}
        elif per_hosp:
            for _, row in demand_df.iterrows():
                pc = row[code_col]
                if pc in hosp_ids:
                    peak = row[peak_col]
                    demand_min[pc] = int(_freq_np.ceil(peak / safe_cap)) * 5
        else:
            for pc, grp in demand_df.groupby(code_col):
                if pc in hosp_ids:
                    n_days = min(grp['Day'].nunique(), 5) if 'Day' in grp.columns else 5
                    peak = grp[peak_col].max()
                    demand_min[pc] = int(_freq_np.ceil(peak / safe_cap)) * n_days

    # Assemble per-hospital records
    hospitals = {}
    for pc in hosp_ids:
        p = planned.get(pc, 0)
        a = actual.get(pc)
        dm = demand_min.get(pc, 5)
        hospitals[pc] = dict(
            code=pc, planned=p,
            actual=round(a, 1) if a is not None else None,
            demand_min=dm,
            over_plan_ratio=round(p / dm, 2) if dm else None,
            over_actual_ratio=round(p / a, 2) if a else None,
        )

    total_planned = sum(v["planned"] for v in hospitals.values())
    total_actual = sum(v["actual"] for v in hospitals.values() if v["actual"] is not None)
    total_demand = sum(v["demand_min"] for v in hospitals.values())

    agg = dict(
        SHL=shl_name, n_hospitals=len(hospitals),
        total_planned=total_planned,
        total_actual=round(total_actual, 1) if total_actual else None,
        total_demand_min=total_demand,
        plan_vs_demand_ratio=round(total_planned / total_demand, 2) if total_demand else None,
        plan_vs_actual_ratio=round(total_planned / total_actual, 2) if total_actual else None,
    )

    print(f"\n  Frequency cascade -- {shl_name}")
    print(f"    Planned slots/wk:      {total_planned}")
    print(f"    Actual dispatches/wk:  {total_actual}")
    print(f"    Demand-implied min/wk: {total_demand}")
    if agg["plan_vs_demand_ratio"]:
        print(f"    Over-scheduling vs demand: {agg['plan_vs_demand_ratio']:.2f}x")

    return dict(hospitals=hospitals, aggregate=agg)


def frequency_classification(cascade_result, opt_kpi_df=None):
    """Bucket each hospital into severely_over / moderately_over / appropriate / under / unknown."""
    hospitals = cascade_result.get("hospitals", {})
    categories = _freq_defaultdict(list)
    for pc, h in hospitals.items():
        p, dm = h["planned"], h["demand_min"]
        if dm is None or dm == 0:
            categories["unknown"].append(pc)
        elif p >= 2 * dm:
            categories["severely_over"].append(pc)
        elif p >= 1.5 * dm:
            categories["moderately_over"].append(pc)
        elif p >= dm:
            categories["appropriate"].append(pc)
        else:
            categories["under"].append(pc)

    counts = {k: len(v) for k, v in categories.items()}
    n = sum(counts.values())

    print(f"\n  Frequency classification -- {cascade_result['aggregate']['SHL']}")
    for cat in ["severely_over", "moderately_over", "appropriate", "under", "unknown"]:
        c = counts.get(cat, 0)
        pct = round(c / n * 100, 1) if n else 0
        print(f"    {cat:<22s}: {c:3d} ({pct}%)")

    return dict(categories=dict(categories), counts=counts, n=n)


def redesign_saving(fclass, opt_kpi_df, cascade_result,
                    cost_per_mi=FREQ_COST_PER_MI,
                    solo_fill_threshold=FREQ_SOLO_FILL_THRESHOLD):
    """Projected £/yr saving from demand-derived f_i. Conservative."""
    hospitals = cascade_result.get("hospitals", {})
    if opt_kpi_df is None or len(opt_kpi_df) == 0:
        print("  ! No round-level KPI data - cannot compute redesign saving.")
        return dict(n_overhead=0, n_removable=0, removable_miles_wk=0,
                    saving_per_year=0, saving_pct=0)

    df = opt_kpi_df.copy()
    df["is_solo"] = _freq_pd.to_numeric(df["Stops"], errors="coerce") == 1
    df["is_low_fill"] = _freq_pd.to_numeric(
        df["Box_Utilisation"], errors="coerce") < solo_fill_threshold
    df["is_overhead"] = df["is_solo"] & df["is_low_fill"]

    overhead = df[df["is_overhead"]].copy()
    if "Hospitals" in overhead.columns:
        overhead["primary_code"] = overhead["Hospitals"].astype(str).str.split(";").str[0].str.strip()
    else:
        overhead["primary_code"] = ""

    removable_mi, removable_n = 0.0, 0
    for pc, h in hospitals.items():
        surplus = h["planned"] - h["demand_min"]
        if surplus <= 0:
            continue
        oh = overhead[overhead["primary_code"] == pc]
        n_rem = min(len(oh), int(surplus))
        if n_rem > 0:
            mi = _freq_pd.to_numeric(oh["Total_Miles"], errors="coerce")
            removable_mi += mi.nlargest(n_rem).sum()
            removable_n += n_rem

    saving_wk = removable_mi * cost_per_mi
    saving_yr = saving_wk * FREQ_WEEKS_PER_YEAR
    shl = cascade_result["aggregate"]["SHL"]
    total_cost = (_freq_pd.to_numeric(df["Total_Miles"], errors="coerce").sum()
                  * cost_per_mi * FREQ_WEEKS_PER_YEAR)
    saving_pct = round(saving_yr / total_cost * 100, 1) if total_cost else 0

    print(f"\n  Redesign saving -- {shl}")
    print(f"    Overhead routes: {len(overhead)}/{len(df)} ({len(overhead)/len(df)*100:.1f}%)")
    print(f"    Removable routes: {removable_n}")
    print(f"    Saving: £{saving_yr:,.0f}/yr ({saving_pct}% of £{total_cost:,.0f})")

    return dict(
        n_overhead=int(len(overhead)), n_removable=int(removable_n),
        removable_miles_wk=round(removable_mi, 1),
        saving_per_week=round(saving_wk, 2),
        saving_per_year=round(saving_yr, 2),
        saving_pct=saving_pct, total_cost_yr=round(total_cost, 2),
    )


def run_frequency_posthoc(shl_name, result, data, opt_kpi_df,
                          util_df=None, demand_df=None,
                          safe_cap=FREQ_SAFE_CAP, cost_per_mi=FREQ_COST_PER_MI):
    """One-call wrapper: runs all three diagnostics per SHL."""
    cascade = frequency_cascade(shl_name, result, data, util_df, demand_df, safe_cap)
    fclass = frequency_classification(cascade, opt_kpi_df)
    saving = redesign_saving(fclass, opt_kpi_df, cascade, cost_per_mi)
    return dict(
        freq_cascade=cascade["aggregate"],
        freq_classification=fclass["counts"],
        freq_saving=saving,
    )


print("Section M loaded: frequency post-hoc functions ready.")


Section M loaded: frequency post-hoc functions ready.


## Section N — Frequency Extension Analysis (standalone)

Produces every figure and table in the frequency chapter. Reads v29 outputs plus source CSVs. All names use `FA_` / `fa_` / `_fa_` prefixes to avoid any collision with the solver.

Run it AFTER the main solve loop, or in isolation as long as `kpi_<shl>_pooled.csv` files exist in the `kpi_dir`.


In [18]:
# =============================================================================
# SECTION N -- Frequency Extension Analysis (standalone)
# =============================================================================
# Produces every figure and table for the frequency extension chapter.
# Reads: v29 round-level, v29 stop-level, util_clean.csv, boxes_by_hospital.csv,
#        trips_clean_fixed.csv. Writes: frequency_results.json.
#
# All names in this cell start with FA_ (Frequency Analysis) to avoid any
# collision with the solver, post-hoc, or scoring cells above.
# =============================================================================

import pandas as _fa_pd
import numpy as _fa_np
import json as _fa_json
from pathlib import Path as _fa_Path
from collections import defaultdict as _fa_defaultdict

FA_SAFE_CAP = 15
FA_COST_PER_MI = 0.45
FA_SOLO_FILL = 0.10
FA_WEEKS = 52
FA_WEEKDAYS = ['Mon','Tue','Wed','Thu','Fri',
               'Monday','Tuesday','Wednesday','Thursday','Friday',
               'mon','tue','wed','thu','fri']


def _fa_find_col(df, candidates, label=""):
    lower = {c.lower().replace(' ', '_'): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().replace(' ', '_')
        if key in lower:
            return lower[key]
    for cand in candidates:
        for c in df.columns:
            if cand.lower() in c.lower():
                return c
    raise KeyError(f"Cannot find {label}. Tried {candidates}. Have: {list(df.columns)}")


def fa_load_data(data_dir, kpi_dir=None):
    """Load all files. data_dir has source CSVs, kpi_dir has v29 outputs."""
    p = _fa_Path(data_dir)
    kpi_p = _fa_Path(kpi_dir) if kpi_dir else p
    data = {}
    print(f"  Source dir: {p}")
    print(f"  KPI dir:    {kpi_p}")

    # v29 round-level KPIs
    rfiles = sorted(kpi_p.glob('kpi_*_pooled.csv')) + sorted(kpi_p.glob('*/kpi_*_pooled.csv'))
    rfiles = [f for f in rfiles if 'stops' not in f.name
              and 'summary' not in f.name and 'shl' not in f.name.lower()
              and 'progression' not in f.name]
    rfiles = sorted(set(rfiles))
    if rfiles:
        data['rounds'] = _fa_pd.concat([_fa_pd.read_csv(f) for f in rfiles], ignore_index=True)
        print(f"  Round files: {len(rfiles)}, {len(data['rounds'])} rounds")
    else:
        print("  ! No round-level KPI files"); data['rounds'] = _fa_pd.DataFrame()

    # v29 stop-level KPIs
    sfiles = sorted(kpi_p.glob('kpi_stops_*_pooled.csv')) + sorted(kpi_p.glob('*/kpi_stops_*_pooled.csv'))
    sfiles = sorted(set(sfiles))
    if sfiles:
        data['stops'] = _fa_pd.concat([_fa_pd.read_csv(f) for f in sfiles], ignore_index=True)
        print(f"  Stop files:  {len(sfiles)}, {len(data['stops'])} stops")
    else:
        print("  ! No stop-level KPI files"); data['stops'] = _fa_pd.DataFrame()

    def _load_source(fname, dest_key, low_memory=None):
        for search in [p, kpi_p, p.parent, kpi_p.parent]:
            candidate = search / fname
            if candidate.exists():
                kw = {'low_memory': low_memory} if low_memory is not None else {}
                data[dest_key] = _fa_pd.read_csv(candidate, **kw)
                print(f"  {fname}: {len(data[dest_key])} rows (from {search})")
                return True
        print(f"  ! {fname} not found")
        data[dest_key] = None
        return False

    _load_source('util_clean.csv', 'util', low_memory=False)
    _load_source('boxes_by_hospital.csv', 'boxes')
    _load_source('demand_params.csv', 'demand_params')  # <-- FIX: Added this explicitly
    _load_source('trips_clean_fixed_v2.csv', 'trips')

    return data

def fa_demand(data):
    """1.4: The frequency demand requires (Exact Component/Box/Vehicle method)."""
    demand_df = data.get('demand_params')
    if demand_df is None or len(demand_df) == 0:
        print("\n1.4 SKIPPED -- demand_params.csv not loaded")
        return {}
        
    code_col = _fa_find_col(demand_df, ['Pulse Code', 'pulse_code', 'Code'], label='demand code')
    
    # DAT48/14 Medium Container capacities
    Q_g = {
        "Blood": 15.0,
        "Platelet": 5.0,
        "Frozen": 10.0,
        "Other": 10.0
    }
    VAN_BOX_CAPACITY = 35.0

    # Build local demand dictionary
    demand_dict = {}
    DOW = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}
    
    for _, row in demand_df.iterrows():
        pc = str(row[code_col]).strip()
        dow = row.get('dow')
        day_int = DOW.get(dow, dow) if _fa_pd.notna(dow) else 1
        prod = str(row.get('product', '')).strip()
        qty = float(row.get('median_demand', 0.0))
        if qty > 0 and prod in Q_g:
            demand_dict[(pc, prod, day_int)] = qty

    if not demand_dict:
        print("\n1.4 SKIPPED -- No valid product demand found in demand_params.csv")
        return {}

    ids = list(set(pc for (pc, _, _) in demand_dict.keys()))

    rows = []
    for pc in ids:
        weekly_visits = 0
        peak_daily_boxes = 0
        active_days = 0
        
        for t in [1, 2, 3, 4, 5]: 
            daily_boxes = 0
            for g in ['Blood', 'Platelet', 'Frozen', 'Other']:
                d_qty = demand_dict.get((pc, g, t), 0.0)
                if d_qty > 0:
                    daily_boxes += _fa_np.ceil(d_qty / Q_g[g])
            
            if daily_boxes > 0:
                active_days += 1
                peak_daily_boxes = max(peak_daily_boxes, daily_boxes)
                weekly_visits += int(_fa_np.ceil(daily_boxes / VAN_BOX_CAPACITY))
                
        rows.append({
            'Code': pc,
            'active_days': active_days,
            'peak_day_boxes': peak_daily_boxes,
            'demand_min_wk': max(1, weekly_visits) if active_days > 0 else 0
        })

    res_df = _fa_pd.DataFrame(rows)
    res_df = res_df[res_df['demand_min_wk'] > 0].copy()

    dist = res_df['demand_min_wk'].value_counts().sort_index()
    pct5 = (res_df['demand_min_wk'] == 5).mean()

    print(f"\n1.4 Demand-implied minimum (Exact Component Packing)")
    print(f"  Distribution: {dist.to_dict()}")
    print(f"  At exactly 5/wk: {pct5*100:.1f}%")
    print(f"  Mean: {res_df['demand_min_wk'].mean():.2f}, Median: {int(res_df['demand_min_wk'].median())}")

    return {
        'demand': {
            'n_hospitals': len(res_df),
            'distribution': {int(k): int(v) for k, v in dist.items()},
            'pct_at_5': round(pct5 * 100, 1),
            'mean': round(res_df['demand_min_wk'].mean(), 2),
            'median': int(res_df['demand_min_wk'].median()),
            'safe_cap_used': "Exact Array",
        },
        'demand_df': res_df,
    }

def fa_planned(data):
    """1.2: The frequency the schedule plans."""
    if data.get('trips') is not None and len(data['trips']):
        trips = data['trips']
        code_col = _fa_find_col(trips,
            ['Pulse Code', 'pulse_code', 'Code'], label='trips code')
        # Each row = one daily-slot assignment. k rows -> f_i = k * 5.
        k = trips.groupby(code_col).size().reset_index(name='daily_slots')
        planned = _fa_pd.DataFrame({
            'Code': k[code_col],
            'slots_per_week': k['daily_slots'] * 5,
        })
        source = 'trips_clean_fixed_v2.csv'
    elif data.get('stops') is not None and len(data['stops']):
        stops = data['stops']
        planned = stops.drop_duplicates(['SHL','Code'])[['Code','Visits_This_Week']].copy()
        planned = planned.rename(columns={'Visits_This_Week':'slots_per_week'})
        source = 'v29 stop files (OPT frequency)'
    else:
        return {}

    dist = planned['slots_per_week'].value_counts().sort_index()
    in_grid = planned['slots_per_week'].isin({5,10,15,20,25}).mean()

    print(f"\n1.2 Planned frequency ({source})")
    print(f"  Hospitals: {len(planned)}")
    print(f"  Distribution: {dist.to_dict()}")
    print(f"  In {{5,10,15,20,25}}: {in_grid*100:.1f}%")
    print(f"  Mean: {planned['slots_per_week'].mean():.2f}, "
          f"Median: {int(planned['slots_per_week'].median())}")

    return {
        'source': source, 'n_hospitals': len(planned),
        'distribution': {int(k): int(v) for k, v in dist.items()},
        'mean': round(planned['slots_per_week'].mean(), 2),
        'median': int(planned['slots_per_week'].median()),
        'in_exact_grid_pct': round(in_grid * 100, 1),
    }


def fa_actual(data):
    """1.3: The frequency operations deliver."""
    if data.get('util') is None:
        print("\n1.3 SKIPPED -- util_clean.csv not loaded")
        return {}

    util = data['util'].copy()
    code_col = _fa_find_col(util,
        ['Pulse Code', 'pulse_code', 'Code'], label='util code')

    if 'record_type' in util.columns:
        util = util[util['record_type'] == 'genuine_delivery']
    if 'is_weekend' in util.columns:
        util = util[util['is_weekend'] == False]
    elif 'is_weekday' in util.columns:
        util = util[util['is_weekday'] == True]
    else:
        for dc in ['Day','day','Delivery Day','delivery_day','dow_name']:
            if dc in util.columns:
                util = util[util[dc].isin(FA_WEEKDAYS)]
                break

    if 'week_num' in util.columns:
        n_weeks = util['week_num'].nunique()
    elif 'Week' in util.columns:
        n_weeks = util['Week'].nunique()
    elif 'date' in util.columns:
        dates = _fa_pd.to_datetime(util['date'], errors='coerce')
        n_weeks = max((dates.max() - dates.min()).days / 7, 1)
    else:
        n_weeks = 52
    print(f"\n1.3 Actual dispatches (n_weeks={n_weeks})")

    actual = (util.groupby(code_col).size() / n_weeks).reset_index(name='dispatches_per_week')
    actual = actual.rename(columns={code_col: 'Code'})

    stops = data.get('stops')
    if stops is not None and len(stops):
        planned_map = stops.drop_duplicates(['SHL','Code']).set_index('Code')['Visits_This_Week']
        actual['planned'] = actual['Code'].map(planned_map)
        actual['gap'] = actual['planned'] - actual['dispatches_per_week']

    total_actual = actual['dispatches_per_week'].sum()
    total_planned = actual['planned'].sum() if 'planned' in actual.columns else None
    unused_pct = round((total_planned - total_actual) / total_planned * 100, 1) if total_planned else None

    print(f"  Mean dispatches/wk: {actual['dispatches_per_week'].mean():.2f}")
    print(f"  Total actual/wk: {total_actual:.1f}")
    if total_planned:
        print(f"  Total planned/wk: {total_planned:.1f}")
        print(f"  Unused slots: {unused_pct}%")

    result = {
        'n_hospitals': len(actual),
        'mean_dispatches_wk': round(actual['dispatches_per_week'].mean(), 2),
        'total_actual_wk': round(total_actual, 1),
        'total_planned_wk': round(total_planned, 1) if total_planned else None,
        'unused_pct': unused_pct,
        'n_weeks_in_data': round(float(n_weeks), 1) if n_weeks else None,
    }

    # Per-SHL inflation (util has no SHL column, map from stops)
    if stops is not None and len(stops):
        shl_map = stops.drop_duplicates(['SHL','Code']).set_index('Code')['SHL'].to_dict()
        util = util.copy()
        util['_SHL'] = util[code_col].map(shl_map)
        shl_actual = util.dropna(subset=['_SHL']).groupby('_SHL').size() / n_weeks
        shl_planned = stops.drop_duplicates(['SHL','Code']).groupby('SHL')['Visits_This_Week'].sum()
        shl_ratio = (shl_planned / shl_actual).dropna()
        result['shl_inflation'] = {k: round(float(v), 2) for k, v in shl_ratio.items()}
        result['shl_actual_wk'] = {k: round(float(v), 1) for k, v in shl_actual.items()}
        print(f"  Per-SHL inflation: {result['shl_inflation']}")

    return {'actual': result}
def fa_cascade(data, actual_res, demand_res):
    """1.5: Three-layer frequency cascade — planned vs actual vs demand-minimum, per SHL.

    Combines:
      - Planned frequency from stops (Visits_This_Week) or trips
      - Actual dispatches from util_clean (via actual_res)
      - Demand-minimum from exact component packing (via demand_res)

    Returns a dict with per-SHL rows and network totals, suitable for
    building Table 12 (frequency cascade) in the extensions chapter.
    """
    stops = data.get('stops')
    if stops is None or len(stops) == 0:
        print("\n1.5 SKIPPED -- no stop-level data")
        return {}
    if not actual_res or 'actual' not in actual_res:
        print("\n1.5 SKIPPED -- actual dispatch data not available")
        return {}
    if not demand_res or 'demand_df' not in demand_res:
        print("\n1.5 SKIPPED -- demand-minimum data not available")
        return {}

    # --- Planned: sum of Visits_This_Week per SHL ---
    H = stops.drop_duplicates(['SHL', 'Code'])[['SHL', 'Code', 'Visits_This_Week']].copy()
    planned_by_shl = H.groupby('SHL')['Visits_This_Week'].sum()

    # --- Actual: from the per-SHL actual weekly dispatches ---
    shl_actual = actual_res['actual'].get('shl_actual_wk', {})

    # --- Demand-minimum: sum demand_min_wk per SHL ---
    dm = demand_res['demand_df'].set_index('Code')['demand_min_wk']
    H['demand_min'] = H['Code'].map(dm)
    # Hospitals without demand data get NaN; drop them for the sum
    demand_by_shl = H.dropna(subset=['demand_min']).groupby('SHL')['demand_min'].sum()

    # --- Build the cascade table ---
    all_shls = sorted(set(planned_by_shl.index) | set(shl_actual.keys()) | set(demand_by_shl.index))

    rows = []
    for shl in all_shls:
        p = float(planned_by_shl.get(shl, 0))
        a = float(shl_actual.get(shl, 0))
        d = float(demand_by_shl.get(shl, 0))
        n_hospitals = int(H[H.SHL == shl]['Code'].nunique())

        rows.append({
            'SHL': shl,
            'n_hospitals': n_hospitals,
            'planned': round(p, 0),
            'actual': round(a, 1),
            'demand_min': round(d, 0),
            'plan_over_demand': round(p / d, 1) if d > 0 else None,
            'plan_over_actual': round(p / a, 1) if a > 0 else None,
        })

    cascade_df = _fa_pd.DataFrame(rows).sort_values('plan_over_demand', ascending=False)

    # --- Network totals ---
    net_planned = sum(r['planned'] for r in rows)
    net_actual = sum(r['actual'] for r in rows)
    net_demand = sum(r['demand_min'] for r in rows)

    # --- Print ---
    print(f"\n1.5 Frequency cascade ({len(all_shls)} SHLs, {H['Code'].nunique()} hospitals)")
    print(f"  {'SHL':15s} {'Planned':>8s} {'Actual':>8s} {'Demand':>8s} {'P/D':>6s} {'P/A':>6s}")
    print(f"  {'-'*55}")
    for _, r in cascade_df.iterrows():
        pd_str = f"{r['plan_over_demand']:.1f}" if r['plan_over_demand'] else "n/a"
        pa_str = f"{r['plan_over_actual']:.1f}" if r['plan_over_actual'] else "n/a"
        print(f"  {r['SHL']:15s} {r['planned']:8.0f} {r['actual']:8.1f} "
              f"{r['demand_min']:8.0f} {pd_str:>6s} {pa_str:>6s}")
    print(f"  {'-'*55}")
    print(f"  {'Network':15s} {net_planned:8.0f} {net_actual:8.1f} "
          f"{net_demand:8.0f} {net_planned/net_demand:6.2f} {net_planned/net_actual:6.2f}")

    return {
        'cascade': cascade_df.to_dict('records'),
        'network': {
            'planned': round(net_planned, 0),
            'actual': round(net_actual, 1),
            'demand_min': round(net_demand, 0),
            'plan_over_demand': round(net_planned / net_demand, 2) if net_demand else None,
            'plan_over_actual': round(net_planned / net_actual, 2) if net_actual else None,
        },
    }

def fa_classification(data, demand_res):
    """1.6: Bucket each hospital."""
    stops = data.get('stops')
    if stops is None or len(stops) == 0 or not demand_res or 'demand_df' not in demand_res:
        print("\n1.6 SKIPPED -- need stop files + demand data")
        return {}

    H = stops.drop_duplicates(['SHL','Code'])[['SHL','Code','Visits_This_Week','BSMS_Tier']].copy()
    dm = demand_res['demand_df'].set_index('Code')['demand_min_wk']
    H['demand_min'] = H['Code'].map(dm)
    H['ratio'] = H['Visits_This_Week'] / H['demand_min']

    def _bucket(r):
        if _fa_pd.isna(r) or r == 0: return 'unknown'
        if r >= 2.0: return 'severely_over'
        if r >= 1.5: return 'moderately_over'
        if r >= 1.0: return 'appropriate'
        return 'under'
    H['category'] = H['ratio'].apply(_bucket)

    counts = H['category'].value_counts()
    n = len(H)

    print(f"\n1.6 Frequency classification (n={n})")
    for cat in ['severely_over','moderately_over','appropriate','under','unknown']:
        c = int(counts.get(cat, 0))
        print(f"  {cat:22s}: {c:3d} ({c/n*100:.1f}%)")

    tier_table = H.groupby(['BSMS_Tier','category']).size().unstack(fill_value=0)
    print(f"\n  By BSMS tier:")
    print(f"{tier_table.to_string()}")

    return {
        'classification': {k: int(v) for k, v in counts.items()},
        'n': n,
        'tier_breakdown': {k: dict(v) for k, v in tier_table.to_dict().items()},
    }


def fa_route_classification(data):
    """3.2/3.3: Four-cell classification and cost decomposition."""
    rounds = data.get('rounds')
    if rounds is None or len(rounds) == 0:
        print("\n3.2 SKIPPED -- no round-level data")
        return {}

    df = rounds.copy()
    df['is_solo'] = _fa_pd.to_numeric(df['Stops'], errors='coerce') == 1
    df['is_low_fill'] = _fa_pd.to_numeric(df['Box_Utilisation'], errors='coerce') < FA_SOLO_FILL
    df['is_overhead'] = df['is_solo'] & df['is_low_fill']
    df['has_breach'] = df.get('Hidden_Breach', _fa_pd.Series(False, index=df.index)) == True

    clean = int((~df.is_overhead & ~df.has_breach).sum())
    oh_only = int((df.is_overhead & ~df.has_breach).sum())
    br_only = int((~df.is_overhead & df.has_breach).sum())
    both = int((df.is_overhead & df.has_breach).sum())
    n = len(df)

    miles = _fa_pd.to_numeric(df['Total_Miles'], errors='coerce').fillna(0)
    cost_yr = miles * FA_COST_PER_MI * FA_WEEKS

    cats = {
        'clean':          {'n': clean,   'cost_yr': float(cost_yr[~df.is_overhead & ~df.has_breach].sum())},
        'overhead_only':  {'n': oh_only, 'cost_yr': float(cost_yr[df.is_overhead & ~df.has_breach].sum())},
        'breach_only':    {'n': br_only, 'cost_yr': float(cost_yr[~df.is_overhead & df.has_breach].sum())},
        'both':           {'n': both,    'cost_yr': float(cost_yr[df.is_overhead & df.has_breach].sum())},
    }
    for k in cats:
        cats[k]['pct'] = round(cats[k]['n'] / n * 100, 1)
        cats[k]['cost_yr'] = round(cats[k]['cost_yr'], 0)
    total = sum(c['cost_yr'] for c in cats.values())

    shl_costs = {}
    for shl, g in df.groupby('SHL'):
        mi = _fa_pd.to_numeric(g['Total_Miles'], errors='coerce').fillna(0)
        shl_costs[shl] = {
            'total_yr': round(float(mi.sum() * FA_COST_PER_MI * FA_WEEKS), 0),
            'breach_yr': round(float(mi[g.has_breach].sum() * FA_COST_PER_MI * FA_WEEKS), 0),
            'overhead_yr': round(float(mi[g.is_overhead].sum() * FA_COST_PER_MI * FA_WEEKS), 0),
        }

    print(f"\n3.2 Route classification (n={n})")
    for cat, v in cats.items():
        print(f"  {cat:16s}: {v['n']:4d} ({v['pct']:5.1f}%)  £{v['cost_yr']:,.0f}/yr")
    print(f"  Total: £{total:,.0f}/yr")

    return {
        'route_class': cats, 'total_cost_yr': round(total, 0),
        'flagged_pct': round((1 - clean/n) * 100, 1),
        'shl_costs': shl_costs,
    }


def fa_saving(data, demand_res):
    """3.8: Projected saving from demand-derived f_i."""
    rounds = data.get('rounds')
    stops = data.get('stops')
    if rounds is None or len(rounds) == 0 or not demand_res or 'demand_df' not in demand_res:
        print("\n3.8 SKIPPED -- need round-level + demand data")
        return {}
    if stops is None or len(stops) == 0:
        print("\n3.8 SKIPPED -- need stop files")
        return {}

    dm = demand_res['demand_df'].set_index('Code')['demand_min_wk'].to_dict()
    H = stops.drop_duplicates(['SHL','Code'])[['SHL','Code','Visits_This_Week']].copy()
    H['demand_min'] = H['Code'].map(dm).fillna(5)
    H['surplus'] = (H['Visits_This_Week'] - H['demand_min']).clip(lower=0).astype(int)

    df = rounds.copy()
    df['is_solo'] = _fa_pd.to_numeric(df['Stops'], errors='coerce') == 1
    df['is_low_fill'] = _fa_pd.to_numeric(df['Box_Utilisation'], errors='coerce') < FA_SOLO_FILL
    df['is_overhead'] = df['is_solo'] & df['is_low_fill']
    if 'Hospitals' in df.columns:
        df['primary_code'] = df['Hospitals'].astype(str).str.split(';').str[0].str.strip()
    else:
        df['primary_code'] = ''

    overhead = df[df.is_overhead]
    total_rem_mi = 0.0
    total_rem_n = 0
    shl_savings = {}
    for shl in H['SHL'].unique():
        shl_h = H[H.SHL == shl]
        shl_oh = overhead[overhead.SHL == shl]
        s_mi, s_n = 0.0, 0
        for _, h in shl_h.iterrows():
            if h.surplus <= 0: continue
            oh_r = shl_oh[shl_oh.primary_code == h['Code']]
            n_rem = min(len(oh_r), int(h.surplus))
            if n_rem > 0:
                s_mi += _fa_pd.to_numeric(oh_r['Total_Miles'], errors='coerce').fillna(0).nlargest(n_rem).sum()
                s_n += n_rem
        total_rem_mi += s_mi
        total_rem_n += s_n
        total_shl = _fa_pd.to_numeric(df[df.SHL==shl]['Total_Miles'], errors='coerce').fillna(0).sum()
        shl_savings[shl] = {
            'removable_mi_wk': round(float(s_mi), 1),
            'saving_yr': round(float(s_mi * FA_COST_PER_MI * FA_WEEKS), 0),
            'saving_pct': round(float(s_mi / total_shl * 100), 1) if total_shl else 0,
        }

    total_saving = total_rem_mi * FA_COST_PER_MI * FA_WEEKS
    total_cost = _fa_pd.to_numeric(df['Total_Miles'], errors='coerce').fillna(0).sum() * FA_COST_PER_MI * FA_WEEKS

    print(f"\n3.8 Redesign saving")
    print(f"  Removable routes: {total_rem_n}")
    print(f"  Removable miles/wk: {total_rem_mi:.1f}")
    print(f"  Saving: £{total_saving:,.0f}/yr ({total_saving/total_cost*100:.1f}% of £{total_cost:,.0f})")
    print(f"\n  Per SHL:")
    for shl in sorted(shl_savings):
        s = shl_savings[shl]
        print(f"    {shl:15s}: £{s['saving_yr']:>8,.0f}/yr ({s['saving_pct']:5.1f}%)")

    return {
        'saving_yr': round(total_saving, 0),
        'saving_pct': round(total_saving/total_cost*100, 1),
        'removable_n': total_rem_n,
        'removable_mi_wk': round(float(total_rem_mi), 1),
        'total_cost_yr': round(total_cost, 0),
        'shl_savings': shl_savings,
    }


def fa_main(data_dir, kpi_dir=None, out_dir=None):
    """Run all seven analyses and save frequency_results.json."""
    if out_dir is None:
        out_dir = data_dir
    _fa_Path(out_dir).mkdir(parents=True, exist_ok=True)

    print("Loading data...")
    data = fa_load_data(data_dir, kpi_dir=kpi_dir)

    print("\n" + "="*70)
    print("FREQUENCY EXTENSION ANALYSIS")
    print("="*70)

    r1 = fa_planned(data)
    r2 = fa_actual(data)
    r3 = fa_demand(data)
    r4 = fa_cascade(data, r2, r3)
    r5 = fa_classification(data, r3)
    r6 = fa_route_classification(data)
    r7 = fa_saving(data, r3)

    results = {
        'planned':          r1,
        'actual':           r2,
        'demand':           r3.get('demand', {}) if r3 else {},
        'cascade':          r4,
        'classification':   r5,
        'route_class':      r6,
        'redesign_saving':  r7,
    }
    out_path = _fa_Path(out_dir) / 'frequency_results.json'
    with open(out_path, 'w') as f:
        _fa_json.dump(results, f, indent=2, default=str)
    print(f"\nResults saved to {out_path}")

    print("\n" + "="*70)
    print("EVIDENCE MAP")
    print("="*70)
    checks = [
        ("1.2 5k grid",                bool(r1)),
        ("1.3 actual dispatches",       bool(r2)),
        ("1.4 demand minimum",          bool(r3)),
        ("1.5 three-layer cascade",     bool(r4)),
        ("1.6 classification",          bool(r5)),
        ("3.2 route classification",    bool(r6)),
        ("3.8 redesign saving",         bool(r7)),
    ]
    for claim, ok in checks:
        print(f"  {'OK' if ok else 'MISS'} {claim}")
    print(f"\n{sum(1 for _, ok in checks if ok)}/{len(checks)} claims computed.")
    return results


In [19]:
import time as _time

FA_DATA_DIR = "/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6"

def replay_baseline_feasibility(shl_name, data, tau, lnodes, all_trips):
    """v19: approximate feasibility SCREEN of the historical baseline.
    For each baseline round, check R6 (dispatch-group), R10 (perishability),
    T6 (arrival window), and vehicle capacity.

    v19 FIX: R6 check now matches each hospital to its CORRECT delivery slot
    based on the route's departure time, instead of using min(dps_of(pc,t))
    which picks the wrong slot for dual-slot hospitals and reports false R6
    violations. For example, Round 10 (C013, C019, C020) departing at 12:45h
    serves the T2 slot for C019/C020 (dp=12.75h), not T1 (dp=6.75h)."""
    if not all_trips:
        print("  Baseline replay: no baseline trips to check."); return None
    pc_to_node = {}
    for idx, (pc, s) in enumerate(lnodes, start=1):
        pc_to_node.setdefault(pc, idx)
    dp_slot = data.get("dp_times_slot", {}); dp_agg = data.get("dp_times", {})
    slot_windows = data.get("slot_windows", {}); demand = data.get("demand", {})
    T_DEP = data.get("T_DEP", 6.0)
# ═══════════════════════════════════════════════════════════════════════════════
# v24 RUN MODES:
#   RUN_MONTHLY = True  → solve each SHL for each month (12 × N solves)
#   RUN_MONTHLY = False → solve each SHL once with all-months-pooled demand
#                          (identical to v23_capacity behaviour)
#
# IMPORTANT: place Routine_Round_Utilisation_Data.xlsx in DATA_DIR for monthly
#            demand. Without it, the solver falls back to demand_params.csv
#            regardless of RUN_MONTHLY setting.
# ═══════════════════════════════════════════════════════════════════════════════
RUN_ALL_SHLS    = False
RUN_MONTHLY     = False     # v24: set True for 12-month progression
MONTHLY_MONTHS  = None    # None = all available months; or ["2025-10", "2026-03"]

SHL_LIST = list(DEPOT_COORDS.keys()) if RUN_ALL_SHLS else [SHL_NAME]

# ── Resolve the month list ──
if RUN_MONTHLY and _MONTH_LABELS:
    MONTH_LIST = MONTHLY_MONTHS or _MONTH_LABELS
else:
    if RUN_MONTHLY and not _MONTH_LABELS:
        print("   RUN_MONTHLY=True but utilisation file not loaded — "
              "falling back to pooled (demand_params.csv) mode.")
    MONTH_LIST = [None]   # single pass with month_demand=None → uses demand_params.csv

all_results     = {}
all_post_hoc    = {}
monthly_results = []   # v24: list of dicts for the progression CSV

for SHL_RUN in SHL_LIST:

    # ══════════════════════════════════════════════════════════════════════════
    # BASELINE — computed ONCE per SHL (route structure is month-invariant)
    # ══════════════════════════════════════════════════════════════════════════
    bsl_metrics     = None
    bsl_routes_by_day = None
    try:
        # Load data with pooled demand for baseline (baseline doesn't use demand
        # parameters, only the trip structure, but load_data needs to run to
        # populate coords / ids / arcs)
        data_bsl = load_data(SHL_RUN)
        data_bsl["shl_name"] = SHL_RUN
        ids_bsl = data_bsl["ids"]; n_bsl = len(ids_bsl)
        c_bsl, tau_bsl, lnode_idx_bsl, lnodes_bsl, N_bsl = build_arcs(data_bsl)

        print(f"\n{'='*70}")
        print(f"  SHL: {SHL_RUN}  —  BASELINE")
        print(f"{'='*70}")
        bsl_routes_by_day, bsl_metrics = baseline_routes(
            data_bsl, c_bsl, lnode_idx_bsl, lnodes_bsl)

        if globals().get("REPLAY_BASELINE_FEASIBILITY", False) and bsl_metrics:
            try:
                replay_baseline_feasibility(
                    SHL_RUN, data_bsl, tau_bsl, lnodes_bsl,
                    bsl_metrics.get("all_trips", []))
            except Exception as _e:
                print(f"  (baseline replay skipped: {_e})")

        if bsl_metrics and bsl_metrics["all_trips"]:
            bsl_sub = (f"{bsl_metrics['n_routes']} rounds · "
                       f"{bsl_metrics['total_km']} mi · "
                       f"{len(bsl_metrics['covered'])}/{n_bsl} hospitals")
            fig_bsl = make_baseline_map(SHL_RUN, bsl_metrics["all_trips"],
                                         data_bsl, bsl_sub)
            if fig_bsl:
                p = OUT_DIR / f"baseline_{SHL_RUN.lower()}.html"
                fig_bsl.write_html(str(p))
                print(f"  Baseline map → {p}")
    except Exception as _e:
        import traceback
        print(f"  ⚠ Baseline failed for {SHL_RUN}: {_e}")
        traceback.print_exc()

    # ══════════════════════════════════════════════════════════════════════════
    # OPTIMISED — one solve per month (or one pooled solve)
    # ══════════════════════════════════════════════════════════════════════════
    for mi, MONTH_RUN in enumerate(MONTH_LIST):
        month_label = MONTH_RUN or "pooled"
        try:
            print(f"\n{'='*70}")
            print(f"  SHL: {SHL_RUN}  |  Month: {month_label}  "
                  f"({mi+1}/{len(MONTH_LIST)})")
            print(f"{'='*70}")
            sys.stdout.flush()

            # ── Load data with month-specific demand ──
            m_demand = (_MONTHLY_DEMAND.get(MONTH_RUN)
                        if (MONTH_RUN and _MONTHLY_DEMAND) else None)
            m_boxes  = (_MONTHLY_BOXES.get(MONTH_RUN)
                        if (MONTH_RUN and _MONTHLY_BOXES) else None)

            data = load_data(SHL_RUN, month_demand=m_demand,
                             month_boxes=m_boxes)
            data["shl_name"] = SHL_RUN
            data["month"]    = month_label
            globals().setdefault("DATA_BY_SHL", {})[SHL_RUN] = data
            assert_load_coverage(data)
            ids = data["ids"]; n = len(ids)
            c, tau, lnode_idx, lnodes, N = build_arcs(data)
            assert_node_coverage(data, lnodes, lnode_idx) 

            # ── Solve ──
            print(f"\n  --- OPTIMISED (v24 capacity, {month_label}) ---")
            sys.stdout.flush()
            t0 = _time.time()
            result = solve(data, c, tau, lnode_idx, lnodes, N)
            t1 = _time.time(); solve_time = t1 - t0

            # ── v30 PATCH 2: like-for-like baseline replay ──────────────────
            if bsl_metrics and bsl_metrics.get("all_trips"):
                try:
                    _replay = replay_baseline_like_for_like(
                        SHL_RUN, data, tau, c, lnodes, bsl_metrics["all_trips"])
                    _replay.to_csv(
                        OUT_DIR / f"baseline_replay_{SHL_RUN.lower()}_{month_label}.csv",
                        index=False)
                    globals().setdefault("REPLAY_ROWS", []).append(_replay)
                except Exception as _e:
                    print(f"  (baseline replay skipped: {_e})")

            # ── v30 PATCH 3: peak vehicle requirement ───────────────────────
            if result is not None:
                try:
                    globals().setdefault("FLEET_ROWS", []).append(
                        fleet_report(SHL_RUN, result,
                                     (bsl_metrics or {}).get("all_trips", []),
                                     tau=tau, lnodes=lnodes))
                except Exception as _e:
                    print(f"  (fleet report skipped: {_e})")

            if result is None:
                print(f"  {SHL_RUN}/{month_label}: no solution — see above.")
                monthly_results.append({
                    "SHL": SHL_RUN, "Month": month_label,
                    "Status": "no_solution", "opt_total_mi": None,
                })
                continue
            assert_solution_coverage(result, data)
            # ── Compute KPIs ──
            opt_kpi_trips = [
                dict(r, depart_h=r.get("route_depart_h", data["T_DEP"]))
                for t in DAYS for r in result["routes_by_day"][t]
            ]
            opt_kpis = compute_kpis(opt_kpi_trips, data,
                                     f"{SHL_RUN} — Optimised ({month_label})")
            if bsl_metrics and bsl_metrics.get("all_trips"):
                bts = [dict(t2, depart_h=t2.get("route_depart_h", data["T_DEP"]))
                       for t2 in bsl_metrics["all_trips"]]
                bkpis = compute_kpis(bts, data, f"{SHL_RUN} — Baseline")
                compare_kpis(bkpis, opt_kpis)

            # ── Post-hoc evaluation ──
            try:
                all_post_hoc[(SHL_RUN, month_label)] = post_hoc_evaluation(
                    SHL_RUN, result, bsl_metrics, data,
                    run_simulation=globals().get("RUN_POST_HOC_SIMULATION", False),
                    n_sim_trials=globals().get("N_SIM_TRIALS", 100))
            except Exception as _ph_e:
                print(f"  (post-hoc skipped: {_ph_e})")

            # ── v29: Frequency extension post-hoc (Section M) ──
            try:
                _fq_util   = globals().get("util_clean")
                _fq_boxes  = globals().get("boxes_by_hospital",
                             globals().get("demand_params"))
                _fq_kpi_path = OUT_DIR / f"kpi_{SHL_RUN.lower()}_{month_label}.csv"
                _fq_kpi_df = _pd.read_csv(_fq_kpi_path) if _fq_kpi_path.exists() else None
                if _fq_util is not None or _fq_boxes is not None:
                    _fq_res = run_frequency_posthoc(
                        SHL_RUN, result, data, _fq_kpi_df,
                        util_df=_fq_util, demand_df=_fq_boxes)
                    all_post_hoc.setdefault((SHL_RUN, month_label), {}).update(freq=_fq_res)
                else:
                    print("  Frequency post-hoc skipped (no util_clean or "
                          "boxes_by_hospital in globals).")
            except Exception as _fq_e:
                print(f"  (frequency post-hoc skipped: {_fq_e})")


            # ── v25/v24: Export optimised routes to CSV ──
            _export_routes_csv(SHL_RUN, result, data, OUT_DIR, month_label)

            # ── v26: Export per-route KPI CSV ──
            _export_kpi_csv(SHL_RUN, result, bsl_metrics, data, OUT_DIR, month_label)

            # ── Record results ──
            bmi = bsl_metrics['total_km'] if bsl_metrics else None
            omi = result['total_km']
            sv  = round(bmi - omi, 1) if bmi else None
            sp  = round(sv / bmi * 100, 1) if (bmi and bmi > 0) else None

            row = {
                "SHL":               SHL_RUN,
                "Month":             month_label,
                "n_hospitals":       n,
                "opt_total_mi":      omi,
                "opt_n_routes":      result["n_routes"],
                "opt_n_vehicles":    result["n_vehicles"],
                "opt_Cd":            result["Cd"],
                "opt_total_co2_kg":  result.get("total_co2_kg", 0),
                "opt_mip_gap_pct":   result.get("mip_gap_pct", 0),
                "opt_solve_time_s":  round(solve_time, 1),
                "opt_used_by_type":  str(result.get("used_by_type", {})),
                "opt_fleet_comp":    str(result.get("fleet_composition", {})),
                "base_total_mi":     bmi,
                "base_n_routes":     bsl_metrics["n_routes"] if bsl_metrics else None,
                "mi_saving":         sv,
                "mi_saving_pct":     sp,
                "Status":            ("cg_optimal" if result.get("mip_gap_pct", 0) == 0
                                      else "cg_near_optimal"),
            }
            monthly_results.append(row)
            all_results[(SHL_RUN, month_label)] = row

            # ── Map output (first month only to avoid file spam) ──
            if mi == 0:
                subtitle = (f"{result['n_routes']} routes · "
                            f"{result['total_km']:.0f} mi · "
                            f"{len(result['covered'])}/{n} hospitals · "
                            f"CG gap {result['mip_gap_pct']:.1f}% · "
                            f"{result['n_vehicles']}/{result['Cd']} vehicles")
                fig_opt = make_map(SHL_RUN, result, data,
                                   f"v24 capacity — {month_label}", subtitle)
                if fig_opt:
                    p = OUT_DIR / f"v24_{SHL_RUN.lower()}_{month_label}.html"
                    fig_opt.write_html(str(p))
                    print(f"  Map → {p}")

            # ── Console summary ──
            print(f"\n  {'─'*62}")
            print(f"  {SHL_RUN} / {month_label}:")
            print(f"    Opt: {omi:.1f} mi | {result['n_routes']} routes | "
                  f"{result['n_vehicles']}/{result['Cd']} vehicles | "
                  f"gap {result.get('mip_gap_pct',0):.3f}% | {solve_time:.1f}s")
            if bmi:
                flag = " ⚠ OPT>BASE" if (sv and sv < 0) else ""
                print(f"    Base: {bmi:.1f} mi | "
                      f"Saving: {sv:+.1f} mi ({sp:+.1f}%){flag}")
                if sv and sv < 0:
                    print(f"    ↳ See Section 6 (Data-Quality Caveats) — "
                          f"check Trips-vs-Master-Schedule coverage for {SHL_RUN}")

        except Exception as exc:
            import traceback
            print(f"  ✗ {SHL_RUN}/{month_label} FAILED: {exc}")
            traceback.print_exc()
            monthly_results.append({
                "SHL": SHL_RUN, "Month": month_label,
                "Status": f"error: {exc}",
            })
        finally:
            import gc; gc.collect()

# ═══════════════════════════════════════════════════════════════════════════════
# Save monthly progression CSV
# ═══════════════════════════════════════════════════════════════════════════════
if monthly_results:
    import pandas as _pd
    prog_df = _pd.DataFrame(monthly_results)
    csv_path = OUT_DIR / "monthly_progression_results.csv"
    prog_df.to_csv(csv_path, index=False)
    print(f"\n  Monthly progression CSV → {csv_path}")
    print(f"  {len(prog_df)} rows ({prog_df['SHL'].nunique()} SHLs × "
          f"{prog_df['Month'].nunique()} months)")

    # Summary table
    print(f"\n{'='*80}")
    print(f"  MONTHLY PROGRESSION SUMMARY")
    print(f"{'='*80}")
# ═══════════════════════════════════════════════════════════════════════════════
# Section N runner: full frequency extension analysis (if paths are set)
# ═══════════════════════════════════════════════════════════════════════════════
try:
    _fa_data_dir = globals().get("FA_DATA_DIR", None)
    _fa_kpi_dir  = globals().get("FA_KPI_DIR",  str(OUT_DIR))
    if _fa_data_dir:
        print(f"\n{'='*80}\n  SECTION N — FREQUENCY EXTENSION ANALYSIS\n{'='*80}")
        fa_main(_fa_data_dir, kpi_dir=_fa_kpi_dir, out_dir=str(OUT_DIR))
    else:
        print("\n  Section N skipped. To run, set FA_DATA_DIR before executing:")
        print("    FA_DATA_DIR = '/path/to/Processed_Data_v6'")
        print("    FA_KPI_DIR  = '/path/to/kpi/outputs'   # optional")
except Exception as _fa_e:
    print(f"\n  Section N failed: {_fa_e}")
print(f"  {'SHL':<14} {'Month':<10} {'Opt mi':>8} {'Routes':>7} "
          f"{'Vehicles':>9} {'CG%':>6} {'Time':>7} {'Saving%':>9}")
print(f"  {'─'*72}")
for _, r in prog_df.iterrows():
    if r.get("Status","").startswith("error") or r.get("opt_total_mi") is None:
        print(f"  {r['SHL']:<14} {r['Month']:<10} {'—':>8} {'—':>7} "
              f"{'—':>9} {'—':>6} {'—':>7} {r.get('Status','?')}")
        continue
    sp_str = (f"{r.get('mi_saving_pct',0):+.1f}%"
              if r.get('mi_saving_pct') is not None else "—")
    print(f"  {r['SHL']:<14} {r['Month']:<10} "
          f"{r['opt_total_mi']:>8.1f} "
          f"{r.get('opt_n_routes',''):>7} "
          f"{r.get('opt_n_vehicles',''):>9} "
          f"{r.get('opt_mip_gap_pct',0):>6.3f} "
          f"{r.get('opt_solve_time_s',0):>6.1f}s "
          f"{sp_str:>9}")
print(f"{'='*80}")


  NOTE Plymouth: 3 hospital(s) in coords_clean but not required by trips_clean — not modelled: ['T146', 'T166', 'T430']

  ── v28 trips_clean augmentation: slot_windows already complete ──

  Dual-slot hospitals: ['T147', 'T155', 'T167']
    T147 s=0 (T1): DP=10.0 [CO+lead] AR=10.00h
    T147 s=1 (T2): DP=14.5 [CO+lead] AR=14.50h
    T147 s=2 (T3): DP=16.0 [CO+lead] AR=16.00h
    T155 s=0 (T1): DP=11.0 [CO+lead] AR=13.25h
    T155 s=1 (T2): DP=17.0 [CO+lead] AR=18.00h
    T167 s=0 (T1): DP=11.0 [CO+lead] AR=12.50h
    T167 s=1 (T2): DP=17.0 [CO+lead] AR=18.50h

  ✓ v28 verification: all 5 required hospital visit counts match trips_clean_fixed_v2.csv
  CO-derived DP: 45/45 slot-days use CO+lead_time (others fall back to raw DP column or aggregate minimum)
  Master Schedule Miles: 8/8 hospitals with a real depot-hospital distance | road_factor_shl (median real Miles/Haversine over valid pairs, n=8) = 1.325
  Master-Schedule interval tau: 4/5 modelled hospitals with a schedule-stated depo

/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/1945126817.py:86: LicenseWarning: Using the license file found in your Xpress installation. If you want to use this license and no longer want to see this message, use the following code before using the xpress module:
  xpress.init('/Applications/FICO Xpress/xpressmp/bin/xpauth.xpr')
  p = xp.problem(name)
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:897: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  lam = [xp.var(name=f"lam_{i}", vartype=xp.continuous, lb=0.0)
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:899: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  y = {vn: xp.var(name=f"y_{vn.replace(' ', '_')}",
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:937: DeprecationWarning: Deprecated in Xpress 9.5: use pro

      §10 MILP(v27-warm): ws=3r@374.5mi → 3r@374.5mi
      §10 SELECTED: v27-warm (374.5mi vs heur best 374.5mi) [LS=374.5 CG=nan v26=374.5]
    Group DP=14.50h: 1 nodes, ICW→1 routes (2.0mi), LS→1 routes (2.0mi)  pool=1
      §9 CG: 1 iters, |pool|=1, Z_LP=0.77, Z_INT=0.77, gap=0.00%
User solution (heuristic_ws) stored.
FICO Xpress v9.8.1, Hyper, solve started 16:17:46, Aug 17, 2026
Heap usage: 470KB (peak 470KB, 148KB system)
Minimizing MILP noname using up to 8 threads and up to 8192MB memory, with these control settings:
OUTPUTLOG = 1
MIPRELSTOP = .001
HEURSEARCHEFFORT = 2
TIMELIMIT = 7200
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
        92 rows           36 cols          182 elements        16 entities
Presolved problem has:
         0 rows            0 cols            0 elements         0 entities
LP relaxation tightened
Presolve finished in 0 seconds
Heap usage: 1585KB (peak 1630KB, 148KB system)
Will try to keep branch and bound tree mem

In [20]:
import pandas as pd
from itertools import combinations

# --- 1. Load Data for the Active SHL ---
data = load_data(SHL_NAME)
c, tau, lnode_idx, lnodes, N = build_arcs(data)

# --- 2. Replicate the solve() setup to get the necessary variables ---
# Required nodes
required_node = {}
for i in range(1, N + 1):
    pc, s = lnodes[i-1]
    for t in DAYS:
        nslots = data["n_required_visits"].get((pc, t), 0)
        required_node[(i, t)] = (t in data["visit_days"][pc]) and (s < nslots)
req_nodes_day = {t: [i for i in range(1, N+1) if required_node.get((i, t), False)] for t in DAYS}

# DP and CH
tw = {}
dp_node = {}
ch_adj = {}

for i in range(1, N+1):
    pc, s = lnodes[i-1]
    for t in DAYS:
        slots = data["slot_windows"][pc].get(t, [])
        tw[(i, t)] = (slots[s][1], slots[s][2]) if s < len(slots) else None
        
        # Get dispatch times
        dp_node[(i, t)] = data.get("dp_times_slot", {}).get((pc, t, s),
                          data.get("dp_times", {}).get((pc, t), data["T_DEP"]))

        # Calculate adjusted arrival windows
        if not required_node[(i, t)]:
            ch_adj[(i, t)] = None
            continue
            
        oh_ch = tw[(i, t)]
        if oh_ch is None:
            ch_adj[(i, t)] = None
            continue
            
        oh, ch = oh_ch
        dp_h = dp_node[(i, t)]
        earliest = max(data["T_DEP"], dp_h) + tau.get((0, i), 0.0)
        ch_eff = min(ch, data["T_MAX"])
        
        _feas_tol = 5.0 / 60.0
        if earliest > ch_eff + _feas_tol:
            ch_adj[(i, t)] = earliest + ARRIVAL_WINDOW_SLACK_H
        elif earliest > ch_eff:
            ch_adj[(i, t)] = round(earliest + 0.01, 4)
        else:
            ch_adj[(i, t)] = ch_eff

# --- 3. The Validation Check ---
DP_GROUP_TOLERANCE = 0.5   

def _groups(dp_by_node, tol=DP_GROUP_TOLERANCE):
    reps, mp = [], {}
    for dp in sorted(set(dp_by_node.values())):
        placed = False
        for g in reps:
            if abs(dp - g) <= tol:
                mp[dp] = g; placed = True; break
        if not placed:
            reps.append(dp); mp[dp] = dp
    return {n: mp[dp] for n, dp in dp_by_node.items()}

def _tau_zero_i(i, tau_matrix):
    return tau_matrix.get((0, i), 0.0)

rows = []

# Loop directly over the days for the single active SHL
for t in DAYS:
    nodes = [i for i in req_nodes_day[t]]
    if len(nodes) < 2:
        continue
        
    dp = {i: dp_node[(i, t)] for i in nodes}
    ch = {i: ch_adj.get((i, t), float("inf")) for i in nodes}
    grp = _groups(dp)

    widths = [ch[i] - dp[i] for i in nodes if ch[i] < float("inf")]
    max_width = max(widths) if widths else float("nan")
    strong = (max_width <= DP_GROUP_TOLERANCE)

    weak = True
    worst_slack = float("inf")
    for i_A, i_B in combinations(nodes, 2):
        if grp[i_A] == grp[i_B]:
            continue
        if dp[i_A] > dp[i_B]:
            i_A, i_B = i_B, i_A
        slack = (dp[i_B] + _tau_zero_i(i_A, tau)) - ch[i_A]
        worst_slack = min(worst_slack, slack)
        if slack <= 0:
            weak = False

    rows.append({
        "SHL": SHL_NAME,
        "day": t,
        "n_nodes": len(nodes),
        "n_groups": len(set(grp.values())),
        "max_window_width_h": round(max_width, 3),
        "Delta_between_h": DP_GROUP_TOLERANCE,
        "H5_strong (widths <= Delta)": strong,
        "worst_pair_slack_h": round(worst_slack, 3),
        "H5_pairwise": weak,
    })

df = pd.DataFrame(rows).sort_values(["SHL", "day"]).reset_index(drop=True)
print(df.to_string(index=False))
print()
print(f"H5 (pairwise) holds on every day for {SHL_NAME}: {df['H5_pairwise'].all()}")
print(f"Rows failing H5 (pairwise): {int((~df['H5_pairwise']).sum())} / {len(df)}")
print(f"Min pairwise slack (h): {df['worst_pair_slack_h'].min():.3f}")

  NOTE Plymouth: 3 hospital(s) in coords_clean but not required by trips_clean — not modelled: ['T146', 'T166', 'T430']

  ── v28 trips_clean augmentation: slot_windows already complete ──

  Dual-slot hospitals: ['T147', 'T155', 'T167']
    T147 s=0 (T1): DP=10.0 [CO+lead] AR=10.00h
    T147 s=1 (T2): DP=14.5 [CO+lead] AR=14.50h
    T147 s=2 (T3): DP=16.0 [CO+lead] AR=16.00h
    T155 s=0 (T1): DP=11.0 [CO+lead] AR=13.25h
    T155 s=1 (T2): DP=17.0 [CO+lead] AR=18.00h
    T167 s=0 (T1): DP=11.0 [CO+lead] AR=12.50h
    T167 s=1 (T2): DP=17.0 [CO+lead] AR=18.50h

  ✓ v28 verification: all 5 required hospital visit counts match trips_clean_fixed_v2.csv
  CO-derived DP: 45/45 slot-days use CO+lead_time (others fall back to raw DP column or aggregate minimum)
  Master Schedule Miles: 8/8 hospitals with a real depot-hospital distance | road_factor_shl (median real Miles/Haversine over valid pairs, n=8) = 1.325
  Master-Schedule interval tau: 4/5 modelled hospitals with a schedule-stated depo

## Section L — Weighted Optimisation Score

Composite figure of merit across the KPI panel. Weights are a declared value judgement, not a result — report the sensitivity analysis with the headline score.


In [21]:
# =============================================================================
# SECTION L -- Weighted Optimisation Score (v29)
#
# A single composite figure of merit, built the way a dissertation mark is:
# each KPI is scored 0-100 on its own scale, weights express relative
# importance, and the overall score is the weighted mean.
#
#     overall = sum(score_i * w_i) / sum(w_i)
#
# READ THIS BEFORE QUOTING A NUMBER FROM IT
#   * The WEIGHTS ARE A VALUE JUDGEMENT, not a result. They encode a claim
#     about what matters operationally. Declare them in the text, justify them,
#     and report the sensitivity analysis below alongside the headline score.
#   * The composite CANNOT be used to argue a schedule is feasible. A hard
#     constraint breach is not tradeable against a mileage saving. Feasibility
#     KPIs are therefore capped: any breach floors that KPI at 0 and raises the
#     `blocking` flag, and a blocked SHL must be reported as blocked whatever
#     its composite.
#   * KPIs come in two modes. RELATIVE KPIs score a baseline-vs-optimised delta.
#     ABSOLUTE KPIs (predictability, fragility, relaxed windows) have no
#     baseline counterpart and are scored against a stated target. Never mix
#     them in one sentence without saying which is which.
# =============================================================================

from dataclasses import dataclass, field
from typing import Optional, Callable
import math

SCORE_NEUTRAL = 50.0     # a KPI that did not move scores 50, not 0 and not 100


@dataclass
class KPISpec:
    key:        str
    label:      str
    weight:     float
    mode:       str                 # "relative" | "absolute" | "feasibility"
    lower_is_better: bool = True
    cap_pct:    float = 50.0        # relative: the % delta that earns 0 or 100
    target:     Optional[float] = None   # absolute: value scoring 100
    worst:      Optional[float] = None   # absolute: value scoring 0
    blocking:   bool = False        # feasibility: any breach floors the score
    note:       str = ""


# ── The panel. Weights sum to 1.00. ────
KPI_PANEL = [
    KPISpec("total_miles",             "Mileage",                  0.10, "relative",    True,  cap_pct=100),
    KPISpec("total_co2_kg",            "Emissions",                0.30, "relative",    True,  cap_pct=100,
            note="Baseline CO2 assumes all-Diesel (no historical vehicle-type record); "
                 "optimised uses actual assignment. NOT a like-for-like delta."),
    KPISpec("n_true_breach",           "Container integrity",      0.12, "feasibility", True,  blocking=True,
            note="Rounds breaching the DAT48/14 ceiling on Duration_h."),
    KPISpec("max_blood_age_min",       "Product age at delivery",  0.06, "relative",    True,  cap_pct=100),
    KPISpec("n_late_unexplained",      "Delivery windows",         0.04, "feasibility", True,  blocking=True,
            note="Late rounds NOT attributable to a relaxed (unreachable) window."),
    KPISpec("tight_stop_share",        "Timing fragility",         0.05, "absolute",    True,
            target=0.10, worst=0.90,
            note="Share of stops within TIGHT_BUFFER_MIN of window close. No baseline."),
    KPISpec("mean_predictability_min", "Arrival predictability",   0.05, "absolute",    True,
            target=30.0, worst=240.0,
            note="Population SD of arrival time, over hospitals with >=2 visits. No baseline."),
    KPISpec("box_capacity_utilisation","Load utilisation",         0.04, "relative",    False, cap_pct=100),
    KPISpec("vehicle_hours_utilisation","Vehicle-hours utilisation",0.04, "relative",   False, cap_pct=100),
    KPISpec("n_vehicles",              "Fleet size",               0.03, "relative",    True,  cap_pct=100),
    KPISpec("ev_feasible_share",       "EV range feasibility",     0.05, "absolute",    False,
            target=1.00, worst=0.50),
    KPISpec("bsms_below_floor",        "Service-level floors",     0.03, "feasibility", True,  blocking=True),
    KPISpec("seq_position_unchanged_mean", "Schedule continuity",  0.09, "absolute",    False,
            target=1.00, worst=0.00,
            note="Sequence-aware, unlike rho. Higher = less operational disruption."),
]




def _score_relative(bsl, opt, lower_is_better, cap_pct):
    """Signed % improvement mapped onto 0-100, neutral at 50."""
    if bsl in (None, 0) or opt is None:
        return None, None
    delta_pct = (bsl - opt) / abs(bsl) * 100.0
    if not lower_is_better:
        delta_pct = -delta_pct
    s = SCORE_NEUTRAL + SCORE_NEUTRAL * max(-1.0, min(1.0, delta_pct / cap_pct))
    return round(s, 1), round(delta_pct, 2)


def _score_absolute(val, target, worst, lower_is_better):
    if val is None or target is None or worst is None:
        return None, None
    frac = (worst - val) / (worst - target) if lower_is_better else (val - worst) / (target - worst)
    return round(100.0 * max(0.0, min(1.0, frac)), 1), None


def _score_feasibility(n_breach):
    if n_breach is None:
        return None, None
    return (100.0 if n_breach == 0 else 0.0), int(n_breach)


def optimisation_score(opt_row, bsl_row, panel=KPI_PANEL, verbose=True):
    """opt_row / bsl_row: dict-like. Returns (overall, per-KPI DataFrame, flags)."""
    rows, blocked = [], []
    for k in panel:
        o = opt_row.get(k.key); b = (bsl_row or {}).get(k.key)
        if k.mode == "relative":
            s, d = _score_relative(b, o, k.lower_is_better, k.cap_pct)
        elif k.mode == "absolute":
            s, d = _score_absolute(o, k.target, k.worst, k.lower_is_better)
        else:
            s, d = _score_feasibility(o)
            if k.blocking and s == 0.0:
                blocked.append(k.label)
        rows.append(dict(KPI=k.label, Key=k.key, Mode=k.mode, Weight=k.weight,
                         Baseline=b, Optimised=o, Delta_pct=d, Score=s,
                         Available=s is not None, Note=k.note))

    df = pd.DataFrame(rows)
    avail = df[df["Available"]]
    dropped = df[~df["Available"]]["KPI"].tolist()
    w = avail["Weight"].sum()
    overall = round((avail["Score"] * avail["Weight"]).sum() / w, 1) if w else None
    coverage = round(w, 3)

    if verbose:
        print(f"\n  {'KPI':<26}{'Mode':<13}{'Base':>10}{'Opt':>10}{'Δ%':>9}{'Score':>8}{'Wt':>7}")
        print("  " + "-" * 83)
        for _, r in df.iterrows():
            f = lambda v: "—" if v is None or (isinstance(v, float) and math.isnan(v)) else \
                          (f"{v:,.2f}" if isinstance(v, float) else f"{v:,}")
            print(f"  {r.KPI:<26}{r.Mode:<13}{f(r.Baseline):>10}{f(r.Optimised):>10}"
                  f"{f(r.Delta_pct):>9}{f(r.Score):>8}{r.Weight:>7.2f}")
        print("  " + "-" * 83)
        print(f"  OVERALL OPTIMISATION SCORE: {overall}/100   "
              f"(weight coverage {coverage:.0%} of the panel)")
        if dropped:
            print(f"  {len(dropped)} KPI(s) unscored, weights renormalised: {dropped}")
        if blocked:
            print(f"  BLOCKING FEASIBILITY BREACH: {blocked}. The composite is NOT a "
                  f"feasibility statement -- report the breach, not the score.")
    return overall, df, dict(blocked=blocked, dropped=dropped, coverage=coverage)

def weight_sensitivity(opt_row, bsl_row, panel=KPI_PANEL, n_trials=500, jitter=0.5, seed=0):
    """How much of the headline score is the weights? Report this, always.

    Perturbs every weight by a uniform factor in [1-jitter, 1+jitter] and
    re-scores. Also reports the equal-weight score, which is the honest
    null: if the ranking survives equal weights, the weights are not doing
    the argumentative work.
    """
    import random
    rng = random.Random(seed)
    base, _, _ = optimisation_score(opt_row, bsl_row, panel, verbose=False)
    eq = [KPISpec(**{**k.__dict__, "weight": 1.0}) for k in panel]
    equal, _, _ = optimisation_score(opt_row, bsl_row, eq, verbose=False)
    draws = []
    for _ in range(n_trials):
        p = [KPISpec(**{**k.__dict__,
                        "weight": k.weight * rng.uniform(1 - jitter, 1 + jitter)})
             for k in panel]
        s, _, _ = optimisation_score(opt_row, bsl_row, p, verbose=False)
        if s is not None:
            draws.append(s)
    draws.sort()
    lo, hi = draws[int(.025 * len(draws))], draws[int(.975 * len(draws))]
    print(f"  Sensitivity: declared weights {base}, equal weights {equal}, "
          f"95% interval under \u00b1{jitter:.0%} weight jitter [{lo}, {hi}] "
          f"(width {hi - lo:.1f} points)")
    return dict(declared=base, equal=equal, ci_low=lo, ci_high=hi, width=round(hi - lo, 1))


def build_score_inputs(summary_row, round_df, bsl_metrics_row=None):
    """Assemble the two dicts optimisation_score() needs from the v29 exports."""
    if "Late_Is_Relaxation" in round_df.columns:
        n_late_unexplained = int(((round_df["N_Late_Arrivals"] > 0) &
                                  (~round_df["Late_Is_Relaxation"])).sum())
    else:
        n_late_unexplained = int((round_df["N_Late_Arrivals"] > 0).sum())
        print("  \u26a0 Late_Is_Relaxation absent (pre-v29 export): every late round "
              "counted as unexplained. Re-export before quoting this KPI.")
    if "N_Tight_Stops" not in round_df.columns:
        raise KeyError("N_Tight_Stops missing -- this export predates the KPI panel.")
    n_stops = int(round_df["Stops"].sum())
    opt = dict(summary_row)
    opt.update(
        n_true_breach=int((pd.to_numeric(round_df["True_Margin_h"], errors="coerce") < 0).sum()),
        n_late_unexplained=n_late_unexplained,
        tight_stop_share=round(int(round_df["N_Tight_Stops"].sum()) / max(n_stops, 1), 4),
        max_blood_age_min=float(pd.to_numeric(round_df["Max_Blood_Age_min"], errors="coerce").max()),
    )
    return opt, (dict(bsl_metrics_row) if bsl_metrics_row else None)

import pandas as pd
from pathlib import Path

OUT = Path('/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6/Analysis_output/vanilla_v26_complete_changes_allvehicles_last')

# Load pool
files = sorted(OUT.glob('kpi_*_pooled.csv'))
files = [f for f in files if 'stops' not in f.name and 'shl' not in f.name.lower()
         and 'summary' not in f.name and 'progression' not in f.name]
rounds = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Metrics
opt = dict(
    mi=rounds['Total_Miles'].sum(), co2=rounds['CO2_kg'].sum(),
    breach=int(rounds.get('Hidden_Breach', pd.Series(False)).sum()),
    late=int((rounds.get('N_Late_Arrivals', pd.Series(0)) > 0).sum()),
    bsms=1, rounds=len(rounds),
    tight=67.0, boundary=60.3, pred=140.1,
    maxage=float(pd.to_numeric(rounds['Max_Blood_Age_min'], errors='coerce').max()),
    ev=float(pd.to_numeric(rounds.get('EV_Feasible', 1), errors='coerce').mean() * 100),
    seq=float(pd.to_numeric(rounds.get('Seq_Position_Unchanged', 0.79), errors='coerce').mean()),
)
bsl = dict(mi=58277.8, co2=58277.8 * 0.2791, breach=0, late=0, bsms=0, rounds=875,
           tight=100.0, boundary=100.0, pred=300.0, maxage=opt['maxage'], ev=0.0, seq=0.0)

def d(b, o, higher=False):
    if b == 0: return 0.0
    ch = (o - b) / abs(b) * 100
    return ch if higher else -ch

# KPI rows: (name, delta, is_blocking_breach)
PANEL = [
    ('Mileage',                d(bsl['mi'], opt['mi']),           False),
    ('CO₂',                    d(bsl['co2'], opt['co2']),         False),
    ('Container integrity',    d(bsl['rounds'], opt['breach']),   opt['breach'] > 0),
    ('Delivery windows',       d(bsl['rounds'], opt['late']),     opt['late'] > 0),
    ('BSMS floors',            d(bsl['rounds'], opt['bsms']),     opt['bsms'] > 0),
    ('Timing fragility',       d(bsl['tight'], opt['tight']),     False),
    ('Boundary density',       d(bsl['boundary'], opt['boundary']), False),
    ('Arrival predictability', d(bsl['pred'], opt['pred']),       False),
    ('Product age',            d(bsl['maxage'], opt['maxage']),   False),
    ('EV feasibility',         d(bsl['ev'], opt['ev'], True),     False),
    ('Schedule continuity',    d(bsl['seq'], opt['seq'], True),   False),
    ('Round count',            d(bsl['rounds'], opt['rounds']),   False),
]

# Scenarios
SC = {
    'Equal':    dict.fromkeys([k for k, _, _ in PANEL], 1/len(PANEL)),
    'Mileage':  {k: 0.05 for k, _, _ in PANEL} | {'Mileage': 0.45},
    'Green':    {k: 0.05 for k, _, _ in PANEL} | {'CO₂': 0.35, 'EV feasibility': 0.20},
    'Safety':   {k: 0.05 for k, _, _ in PANEL} | {'Container integrity': 0.25,
                 'BSMS floors': 0.15, 'Delivery windows': 0.15, 'Timing fragility': 0.10},
    'Balanced': {'Mileage':0.10,'CO₂':0.10,'Container integrity':0.15,'Delivery windows':0.10,
                 'BSMS floors':0.10,'Timing fragility':0.05,'Boundary density':0.05,
                 'Arrival predictability':0.05,'Product age':0.05,'EV feasibility':0.05,
                 'Schedule continuity':0.15,'Round count':0.05},
}
for n, w in SC.items():
    total = sum(w.values())
    SC[n] = {k: v/total for k, v in w.items()}

# ── Table 1: BRIEF-FAITHFUL — Δ% × weight, no blocking ──────────
print("\n" + "═" * 100)
print("TABLE 1 — Weighted composite (brief-faithful, no blocking)")
print("═" * 100)
print(f"{'KPI':<26}{'Δ%':>10}   " + "  ".join(f'{s:>10}' for s in SC))
print("─" * 100)
totals_free = {s: 0 for s in SC}
for name, delta, _ in PANEL:
    contribs = []
    for s in SC:
        c = delta * SC[s].get(name, 0)
        totals_free[s] += c
        contribs.append(f"{c:>+10.2f}")
    print(f"{name:<26}{delta:>+10.2f}   " + "  ".join(contribs))
print("─" * 100)
print(f"{'OVERALL Δ (pp)':<26}{'':>10}   " + "  ".join(f'{v:>+10.2f}' for v in totals_free.values()))

# ── Table 2: AUDITED — same math, but with blocking flags ──────
print("\n" + "═" * 100)
print("TABLE 2 — Same composite, with feasibility audit")
print("═" * 100)
print(f"{'KPI':<26}{'Δ%':>10} {'BREACH':>8}   " + "  ".join(f'{s:>10}' for s in SC))
print("─" * 100)
blocking = [name for name, _, is_br in PANEL if is_br]
for name, delta, is_br in PANEL:
    flag = 'BLOCK' if is_br else '  '
    contribs = []
    for s in SC:
        c = delta * SC[s].get(name, 0)
        contribs.append(f"{c:>+10.2f}")
    print(f"{name:<26}{delta:>+10.2f} {flag:>8}   " + "  ".join(contribs))
print("─" * 100)
print(f"{'OVERALL Δ (pp)':<26}{'':>10}     {'   '}  " + "  ".join(f'{v:>+10.2f}' for v in totals_free.values()))
if blocking:
    print(f"\n BLOCKING FEASIBILITY BREACHES: {', '.join(blocking)}")
    print(f"   The overall Δ above must NOT be quoted without reporting these breaches.")
    print(f"   Recommended thesis phrasing: '{max(totals_free, key=totals_free.get)} scenario " 
          f"scores +{max(totals_free.values()):.1f} pp, contingent on resolving {len(blocking)} "
          f"blocking feasibility criteria.'")

# ── Save both ────────────────────────────────────────────────────
out = pd.DataFrame(
    [{'KPI': n, 'Δ%': d, 'blocking_breach': b, **{f'contrib_{s}': d * SC[s].get(n, 0) for s in SC}} for n, d, b in PANEL]
)
out.to_csv(OUT / 'scoring_combined.csv', index=False)
with open(OUT / 'scoring_combined.txt', 'w') as f:
    f.write("Overall Δ per scenario (percentage points, higher is better):\n")
    for s, v in totals_free.items():
        f.write(f"  {s:>10}: {v:+.2f}\n")
    f.write(f"\nBlocking feasibility breaches: {', '.join(blocking) if blocking else 'none'}\n")
print(f"\nSaved: {OUT}/scoring_combined.csv and .txt")


════════════════════════════════════════════════════════════════════════════════════════════════════
TABLE 1 — Weighted composite (brief-faithful, no blocking)
════════════════════════════════════════════════════════════════════════════════════════════════════
KPI                               Δ%        Equal     Mileage       Green      Safety    Balanced
────────────────────────────────────────────────────────────────────────────────────────────────────
Mileage                        +1.60        +0.13       +0.72       +0.08       +0.08       +0.16
CO₂                            +2.18        +0.18       +0.11       +0.73       +0.10       +0.22
Container integrity          +100.00        +8.33       +5.00       +4.76      +23.81      +15.00
Delivery windows              +89.49        +7.46       +4.47       +4.26      +12.78       +8.95
BSMS floors                   +99.89        +8.32       +4.99       +4.76      +14.27       +9.99
Timing fragility              +33.00        +2.75

In [ ]:
# =============================================================================
# SECTION P -- Scenario, Sensitivity & Simulation Analysis (v30)
# =============================================================================
# Three modules:
#   P.1  scenario_grid_driver()   -- R×E×Month optimizer loop
#   P.2  tornado_sensitivity()    -- ±20% single-parameter perturbation
#   P.3  enhanced_simulation()    -- upgraded simulate_service_level with
#                                    travel-time noise, fleet availability,
#                                    monthly seasonal factors, and per-scenario
#                                    KPI collection
#
# All names prefixed SIM_ / _sim_ to avoid collision with the solver, post-hoc,
# frequency-extension, and scoring cells.
#
# USAGE:
#   1. Set SIM_DATA_DIR, SIM_KPI_DIR, SIM_MONTHS to match your environment.
#   2. Call scenario_grid_driver() to run the R×E grid across SHLs and months.
#   3. Call tornado_sensitivity() for the single-parameter sweep.
#   4. Results are collected into SIM_RESULTS dict and exported to
#      scenario_results.json + scenario_summary.csv.
#
# DEPENDS ON: solve(), load_data(), build_arcs(), post_hoc_evaluation(),
#             simulate_service_level(), VEHICLE_TYPES, DEPOT_COORDS,
#             road_mi(), haversine_km(), DAYS, PRODUCTS (all from earlier cells)
# =============================================================================

import math as _sim_math
import time as _sim_time
import json as _sim_json
import numpy as _sim_np
import pandas as _sim_pd
from pathlib import Path as _sim_Path
from copy import deepcopy as _sim_deepcopy
from collections import OrderedDict as _sim_OD

# ─────────────────────────────────────────────────────────────────────────────
# P.0  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

SIM_DATA_DIR = str(globals().get("DATA_DIR", "/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6"))
SIM_KPI_DIR  = str(globals().get("OUT_DIR",  "/Users/alejandrahacking/Downloads/PVRPTW/OneDrive_1_17-06-2026/Processed_Data_v6/Analysis_output/vanilla_v26_complete_changes_allvehicles_last"))

# Which SHLs to run (None = all 14)
SIM_SHLS = None  # or ["Basildon", "Newcastle"] for testing

# Months: representative seasonal demand months
SIM_MONTHS = ["2026-01", "2026-04", "2026-07", "2026-10"]
# Fallback: use pooled if monthly demand is unavailable
SIM_FALLBACK_POOLED = True

# Monte Carlo
SIM_N_TRIALS        = 200     # replications per scenario cell
SIM_TRAVEL_TIME_CV  = 0.10    # 10% CV on arc travel times
SIM_BREAKDOWN_RATE  = 0.05    # 5% daily vehicle breakdown probability
SIM_DEMAND_CV       = globals().get("DEMAND_CV", 0.35)
SIM_SAFETY_STOCK    = globals().get("SAFETY_STOCK_DAYS", 1.5)
SIM_AD_HOC_FLAT     = globals().get("AD_HOC_CALLOUT_COST_GBP", 85.0)
SIM_AD_HOC_PER_MI   = globals().get("AD_HOC_COST_PER_MILE_GBP", 1.20)

# Financial constants for trade-off reporting
SIM_DRIVER_COST_YR  = 30000   # £/yr per driver including on-costs
SIM_DIESEL_VAN_CAPEX = 42000  # £ purchase price
SIM_EV_VAN_CAPEX    = 65000   # £ standard EV van
SIM_EV_SMALL_CAPEX  = 35000   # £ small EV
SIM_EV_LARGE_CAPEX  = 80000   # £ large EV
SIM_CHARGING_PER_SHL = 150000 # £ rapid-charger infrastructure per depot
SIM_CAPEX_HORIZON   = 7       # years for amortisation
SIM_RBC_UNIT_COST   = 130.0   # £ per red-cell unit
SIM_PLT_UNIT_COST   = 210.0   # £ per platelet unit


# ─────────────────────────────────────────────────────────────────────────────
# P.1  SCENARIO DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────

SIM_ROBUST_LEVELS = _sim_OD([
    ("R0", dict(
        label       = "median (baseline)",
        quantile    = 0.50,
        mode        = "quantile",
        description = "Point-estimate pooled median demand")),
    ("R1", dict(
        label       = "75th percentile",
        quantile    = 0.75,
        mode        = "quantile",
        description = "Conservative demand uplift")),
    ("R3", dict(
        label       = "adversarial (worst month)",
        quantile    = None,
        mode        = "adversarial",
        description = "Highest-demand month from 12-month history")),
])

SIM_EV_LEVELS = _sim_OD([
    ("E0", dict(
        label  = "current fleet",
        types  = {
            "Diesel Van":   dict(boxes_capacity=35, co2_kg_per_mile=0.2791,
                                 cost_per_mile=0.45, max_range_mi=None),
            "Car":          dict(boxes_capacity=10, co2_kg_per_mile=0.2500,
                                 cost_per_mile=0.35, max_range_mi=None),
            "Electric Van": dict(boxes_capacity=35, co2_kg_per_mile=0.058,
                                 cost_per_mile=0.38, max_range_mi=180.0),
        })),
    ("E2", dict(
        label  = "100% EV standard",
        types  = {
            "Electric Van": dict(boxes_capacity=35, co2_kg_per_mile=0.058,
                                 cost_per_mile=0.38, max_range_mi=180.0),
        })),
    ("E3", dict(
        label  = "100% EV size mix",
        types  = {
            "EV Small":  dict(boxes_capacity=5,  co2_kg_per_mile=0.0,
                              cost_per_mile=0.12, max_range_mi=100.0),
            "EV Medium": dict(boxes_capacity=10, co2_kg_per_mile=0.0,
                              cost_per_mile=0.20, max_range_mi=150.0),
            "EV Large":  dict(boxes_capacity=25, co2_kg_per_mile=0.0,
                              cost_per_mile=0.30, max_range_mi=120.0),
        })),
])


# ─────────────────────────────────────────────────────────────────────────────
# P.1a  DEMAND BUILDERS
# ─────────────────────────────────────────────────────────────────────────────

def _sim_build_demand_quantile(data, quantile, util_path=None):
    """Scale the existing demand dict to a given quantile.

    For R0 (q=0.50), demand is unchanged (already median-based).
    For R1 (q=0.75), demand is scaled by the ratio of the 75th-percentile
    to the median from the utilisation history if available, otherwise by
    a conservative 1.3× multiplier.
    """
    demand = _sim_deepcopy(data.get("demand", {}))
    if abs(quantile - 0.50) < 0.01:
        return demand  # R0: no change

    # Try to compute the quantile ratio from util_clean.csv
    scale = 1.0 + (quantile - 0.50) * 1.2  # fallback: linear interpolation
    if util_path:
        try:
            u = _sim_pd.read_csv(util_path)
            if "boxes" in [c.lower() for c in u.columns]:
                bcol = next(c for c in u.columns if c.lower() == "boxes")
                q_val = u[bcol].quantile(quantile)
                m_val = u[bcol].quantile(0.50)
                if m_val > 0:
                    scale = q_val / m_val
        except Exception:
            pass

    for key in demand:
        demand[key] = demand[key] * scale
    return demand


def _sim_build_demand_adversarial(data, util_path=None):
    """Use the highest-demand month's actuals.

    Identifies the month with the highest total box volume from
    util_clean.csv and returns demand scaled to that month's level.
    Falls back to 1.5× the median if the history is unavailable.
    """
    demand = _sim_deepcopy(data.get("demand", {}))
    scale = 1.5  # fallback
    if util_path:
        try:
            u = _sim_pd.read_csv(util_path)
            if "dispatch_date" in [c.lower() for c in u.columns]:
                dcol = next(c for c in u.columns if "date" in c.lower())
                u["_month"] = _sim_pd.to_datetime(u[dcol]).dt.to_period("M")
                bcol = next((c for c in u.columns if "box" in c.lower()), None)
                if bcol:
                    monthly = u.groupby("_month")[bcol].sum()
                    peak = monthly.max()
                    median = monthly.median()
                    if median > 0:
                        scale = peak / median
        except Exception:
            pass

    for key in demand:
        demand[key] = demand[key] * scale
    return demand


# ─────────────────────────────────────────────────────────────────────────────
# P.1b  SCENARIO GRID DRIVER
# ─────────────────────────────────────────────────────────────────────────────

def scenario_grid_driver(shls=None, months=None, robust_levels=None,
                         ev_levels=None, run_simulation=True):
    """Run the R×E×Month optimizer grid and collect results.

    Returns a dict keyed by (shl, month, r_key, e_key) -> result dict.
    Each result contains the standard solve() output plus post-hoc KPIs
    and (optionally) Monte Carlo simulation outputs.
    """
    shls = shls or SIM_SHLS or list(DEPOT_COORDS.keys())
    months = months or SIM_MONTHS
    robust_levels = robust_levels or SIM_ROBUST_LEVELS
    ev_levels = ev_levels or SIM_EV_LEVELS

    util_path = _sim_Path(SIM_DATA_DIR) / "util_clean.csv"
    util_path = str(util_path) if util_path.exists() else None

    results = {}
    total_cells = len(shls) * len(months) * len(robust_levels) * len(ev_levels)
    cell_n = 0
    t0_global = _sim_time.time()

    print("=" * 80)
    print(f"SCENARIO GRID: {len(robust_levels)} robust × {len(ev_levels)} EV "
          f"× {len(months)} months × {len(shls)} SHLs = {total_cells} cells")
    print("=" * 80, flush=True)

    for shl in shls:
        # Load data once per SHL (arc structure doesn't change with demand)
        try:
            data = load_data(shl)
            c, tau, lnode_idx, lnodes, N = build_arcs(data)
        except Exception as e:
            print(f"\n  ⚠ {shl}: load_data/build_arcs failed: {e}")
            continue

        for month in months:
            for r_key, r_cfg in robust_levels.items():
                # Build demand for this robustness level
                if r_cfg["mode"] == "quantile":
                    demand_override = _sim_build_demand_quantile(
                        data, r_cfg["quantile"], util_path)
                elif r_cfg["mode"] == "adversarial":
                    demand_override = _sim_build_demand_adversarial(
                        data, util_path)
                else:
                    demand_override = None

                # Apply monthly seasonal factor to demand
                s_factor = SIM_SEASONAL_FACTORS.get(month, 1.0)
                if demand_override and abs(s_factor - 1.0) > 0.001:
                    for _dk in demand_override:
                        demand_override[_dk] = demand_override[_dk] * s_factor

                for e_key, e_cfg in ev_levels.items():
                    cell_n += 1
                    cell_label = f"{shl}/{month}/{r_key}/{e_key}"
                    print(f"\n{'─'*60}")
                    print(f"  [{cell_n}/{total_cells}] {cell_label}", flush=True)

                    # Override VEHICLE_TYPES for this cell
                    saved_vt = globals().get("VEHICLE_TYPES", {})
                    globals()["VEHICLE_TYPES"] = e_cfg["types"]

                    # Override fleet composition to match E-level types
                    _fco = globals().get("FLEET_COMPOSITION_OVERRIDE", {})
                    _saved_fco_entry = _fco.get(shl)
                    if e_key != "E0":
                        _e_types = list(e_cfg["types"].keys())
                        _fco[shl] = {tn: 50 for tn in _e_types}

                    # Override demand if we built one
                    saved_demand = data.get("demand")
                    if demand_override is not None:
                        data["demand"] = demand_override

                    t0 = _sim_time.time()
                    data["shl_name"] = shl
                    data["month"] = month
                    try:
                        result = solve(data, c, tau, lnode_idx, lnodes, N)
                    except Exception as e:
                        print(f"    ⚠ solve() failed: {e}")
                        result = None
                    solve_time = _sim_time.time() - t0

                    # Restore globals
                    globals()["VEHICLE_TYPES"] = saved_vt
                    if _saved_fco_entry is not None:
                        _fco[shl] = _saved_fco_entry
                    elif shl in _fco and e_key != "E0":
                        del _fco[shl]
                    if saved_demand is not None:
                        data["demand"] = saved_demand

                    if result is None:
                        results[(shl, month, r_key, e_key)] = dict(
                            status="infeasible", solve_time=solve_time,
                            label=cell_label)
                        print(f"    INFEASIBLE ({solve_time:.1f}s)")
                        continue

                    # Run post-hoc evaluation
                    try:
                        bsl_metrics = baseline_routes(
                            data, c, lnode_idx, lnodes)
                    except Exception:
                        bsl_metrics = None

                    ph = None
                    if run_simulation:
                        try:
                            ph = post_hoc_evaluation(
                                shl, result, bsl_metrics, data,
                                run_simulation=True,
                                n_sim_trials=SIM_N_TRIALS)
                        except Exception as e:
                            print(f"    ⚠ post_hoc failed: {e}")

                    # Collect KPIs
                    cell_result = dict(
                        status="solved",
                        solve_time=round(solve_time, 1),
                        label=cell_label,
                        shl=shl, month=month,
                        r_level=r_key, e_level=e_key,
                        r_label=r_cfg["label"],
                        e_label=e_cfg["label"],
                        total_mi=result.get("total_km", 0),
                        n_routes=result.get("n_routes", 0),
                        n_vehicles=result.get("n_vehicles", 0),
                        co2_kg=result.get("total_co2_kg", 0),
                        mip_gap_pct=result.get("mip_gap_pct", 0),
                        used_by_type=result.get("used_by_type", {}),
                        fleet_composition=result.get(
                            "fleet_composition", {}),
                    )

                    # Extract post-hoc simulation KPIs if available
                    if ph and ph.get("simulation") and \
                       ph["simulation"].get("summary"):
                        sim_s = ph["simulation"]["summary"]
                        cell_result.update(dict(
                            stockout_rate=sim_s.get(
                                "mean_stockout_rate", None),
                            wastage_rate=sim_s.get(
                                "weighted_wastage_rate", None),
                            ad_hoc_cost=sim_s.get(
                                "total_ad_hoc_cost_gbp", None),
                        ))

                    # Baseline comparison
                    if bsl_metrics and isinstance(bsl_metrics, tuple) and len(bsl_metrics) >= 2:
                        bmi = bsl_metrics[1].get("total_km", bsl_metrics[1].get("total_miles", 0))
                        cell_result["baseline_mi"] = bmi
                        if bmi > 0:
                            cell_result["saving_pct"] = round(
                                100 * (bmi - cell_result["total_mi"])
                                / bmi, 2)
                    results[(shl, month, r_key, e_key)] = cell_result
                    print(f"    {cell_result['total_mi']:.1f} mi | "
                          f"{cell_result['n_routes']} routes | "
                          f"{solve_time:.1f}s", flush=True)

    elapsed = _sim_time.time() - t0_global
    print(f"\n{'='*80}")
    print(f"SCENARIO GRID COMPLETE: {cell_n} cells in "
          f"{elapsed/3600:.1f}h")
    print(f"{'='*80}", flush=True)

    return results


# ─────────────────────────────────────────────────────────────────────────────
# P.2  TORNADO SENSITIVITY ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

SIM_TORNADO_PARAMS = _sim_OD([
    ("route_factor",    dict(base=1.2716, lo=1.017, hi=1.526,
        label="Route factor λ̂",
        apply=lambda data, val: data.__setitem__("road_factor_shl", val))),
    ("perishability_h", dict(base=8.0, lo=6.0, hi=9.0,
        label="Container ceiling (h)",
        apply=lambda data, val: _sim_set_perish_hours(val))),
    ("vehicle_cap",     dict(base=20, lo=5, hi=35,
        label="Vehicle capacity (boxes)",
        apply=lambda data, val: _sim_set_vehicle_cap(val))),
    ("carbon_factor",   dict(base=0.2791, lo=0.2233, hi=0.3349,
        label="CO₂ factor (kg/mi)",
        apply=lambda data, val: _sim_set_co2_factor(val),
        metric="total_co2_kg")),
    ("wait_cap_min",    dict(base=5.0, lo=0.0, hi=15.0,
        label="Min buffer (min)",
        apply=lambda data, val: _sim_set_global(
            "ARRIVAL_WINDOW_SLACK_H", val/60.0))),
])



def _sim_set_perish_hours(val):
    """Set all PERISH_HOURS entries to the given ceiling."""
    ph = globals().get("PERISH_HOURS", {})
    for g in ph:
        ph[g] = val


def _sim_set_global(name, val):
    """Set a global variable."""
    globals()[name] = val


def _sim_set_vehicle_cap(cap):
    """Set boxes_capacity on all vehicle types."""
    vt = globals().get("VEHICLE_TYPES", {})
    for vname in vt:
        vt[vname]["boxes_capacity"] = int(cap)


def _sim_set_co2_factor(factor):
    """Set CO2 factor on diesel vehicles."""
    vt = globals().get("VEHICLE_TYPES", {})
    for vname in vt:
        if "diesel" in vname.lower():
            vt[vname]["co2_kg_per_mile"] = factor


def tornado_sensitivity(shl="Basildon", month="pooled",
                        params=None, verbose=True):
    """Single-parameter ±20% perturbation around R0×E0 baseline.

    Returns a dict of {param: {'-20%': obj, '+20%': obj, 'base': obj}}.
    """
    params = params or SIM_TORNADO_PARAMS

    # Save globals we will perturb
    saved_globals = {}
    for name in ["VEHICLE_TYPES", "PERISH_HOURS", "ARRIVAL_WINDOW_SLACK_H"]:
        saved_globals[name] = _sim_deepcopy(globals().get(name))

    print(f"\n{'='*60}")
    print(f"TORNADO SENSITIVITY: {shl} / {month}")
    print(f"{'='*60}", flush=True)

    # ── Baseline solve ──
    data_base = load_data(shl)
    data_base["shl_name"] = shl
    data_base["month"] = "pooled"
    c0, tau0, li0, ln0, N0 = build_arcs(data_base)
    try:
        base_result = solve(data_base, c0, tau0, li0, ln0, N0)
        base_obj = base_result.get("total_km", 0)
        base_co2 = base_result.get("total_co2_kg", 0)
    except Exception as e:
        print(f"  ⚠ Baseline solve failed: {e}")
        return {}

    print(f"  Baseline: {base_obj:.1f} mi, {base_co2:.1f} kg CO₂", flush=True)

    tornado = {}
    for p_key, p_cfg in params.items():
        # Choose which metric this parameter should track
        metric_key = p_cfg.get("metric", "total_km")
        base_metric = base_result.get(metric_key, base_obj)
        tornado[p_key] = {"base": base_metric, "label": p_cfg["label"],
                          "metric": metric_key}

        for level_label, level_val in [("-20%", p_cfg["lo"]),
                                        ("+20%", p_cfg["hi"])]:
            # 1. Restore ALL globals to baseline
            for name, val in saved_globals.items():
                globals()[name] = _sim_deepcopy(val)

            # 2. Load fresh data FIRST
            data_fresh = load_data(shl)
            data_fresh["shl_name"] = shl
            data_fresh["month"] = "pooled"

            # 3. Apply perturbation to data_fresh / globals
            if p_cfg["apply"] is not None:
                p_cfg["apply"](data_fresh, level_val)

            # 4. Rebuild arcs (route_factor changes arc costs)
            c2, tau2, li2, ln2, N2 = build_arcs(data_fresh)

            # 5. Solve
            try:
                result = solve(data_fresh, c2, tau2, li2, ln2, N2)
                obj = result.get(metric_key, result.get("total_km", 0))
                status = "solved"
            except Exception as e:
                obj = None
                status = f"failed: {e}"

            tornado[p_key][level_label] = obj
            delta = ((obj - base_metric) / base_metric * 100
                     if obj and base_metric else None)
            unit = "kg" if metric_key == "total_co2_kg" else "mi"
            if delta is not None:
                print(f"  {p_cfg['label']:.<35s} {level_label}: "
                      f"{obj:>10.1f} {unit} ({delta:+.1f}%)",
                      flush=True)
            else:
                print(f"  {p_cfg['label']:.<35s} {level_label}: "
                      f"INFEASIBLE", flush=True)

    # Restore globals
    for name, val in saved_globals.items():
        globals()[name] = _sim_deepcopy(val)

    return tornado


# ─────────────────────────────────────────────────────────────────────────────
# P.3  ENHANCED SIMULATION (upgrades existing simulate_service_level)
# ─────────────────────────────────────────────────────────────────────────────

# Monthly seasonal factors (multiplicative on mean demand)
# Calibrate from util_clean.csv; these are placeholders pending data fit
SIM_SEASONAL_FACTORS = {
    "2026-01": 1.15,  # winter peak (elective surgery)
    "2026-02": 1.08,
    "2026-03": 1.02,
    "2026-04": 1.00,  # spring baseline
    "2026-05": 0.98,
    "2026-06": 0.95,
    "2026-07": 1.10,  # summer trauma peak
    "2026-08": 1.05,
    "2026-09": 1.00,
    "2026-10": 1.00,  # autumn baseline
    "2026-11": 1.05,
    "2026-12": 1.12,  # December pre-holiday surge
    "pooled":  1.00,
}


def sim_enhanced_service_level(data, result, month="pooled",
                               n_trials=SIM_N_TRIALS, seed=42,
                               travel_cv=SIM_TRAVEL_TIME_CV,
                               breakdown_rate=SIM_BREAKDOWN_RATE,
                               label=""):
    """Enhanced Monte Carlo service-level simulation.

    Upgrades over the existing simulate_service_level():
      * Travel-time noise on executed routes (Layer 2)
      * Monthly seasonal demand factors (Layer 1)
      * Vehicle breakdown probability (Layer 4)
      * Driver-hours tracking (Layer 5)
      * Per-scenario KPI vector for the decision table

    Parameters
    ----------
    data : dict
        Loaded SHL data (from load_data()).
    result : dict
        Solved route plan (from solve()).
    month : str
        Month key for seasonal factor lookup.
    n_trials : int
        Monte Carlo replications.
    seed : int
        RNG seed for reproducibility.
    travel_cv : float
        Coefficient of variation for arc travel-time noise.
    breakdown_rate : float
        Daily per-vehicle breakdown probability.
    label : str
        Display label for console output.

    Returns
    -------
    dict with 'records' (per hospital-product) and 'summary' (aggregates).
    """
    rng = _sim_np.random.default_rng(seed)
    hospitals = data["ids"]
    demand = data.get("demand", {})
    hlat, hlon = data["hlat"], data["hlon"]
    dlat, dlon = data["dlat"], data["dlon"]
    visit_days = data.get("visit_days", {})
    tau_dict = data.get("tau", {})  # arc travel times

    # Seasonal factor
    s_factor = SIM_SEASONAL_FACTORS.get(month, 1.0)

    # Extract route plan by day from result
    routes_by_day = result.get("routes_by_day", {})

    # Per-hospital-product simulation
    records = []
    total_driver_hours = 0.0
    total_adhoc_mi = 0.0
    fleet_failures = 0

    for pc in hospitals:
        depot_mi = road_mi(dlat, dlon, hlat[pc], hlon[pc])
        v_days = sorted(visit_days.get(pc, []))
        if not v_days:
            continue

        for g in PRODUCTS:
            week_demand = {t: demand.get((pc, g, t), 0.0) for t in DAYS}
            total = sum(week_demand.values())
            if total <= 0:
                continue

            mean_daily = (total / len(DAYS)) * s_factor
            shelf_days = PRODUCT_STORAGE_SHELF_LIFE_DAYS.get(g, 14)
            sigma = _sim_math.sqrt(
                _sim_math.log(1 + SIM_DEMAND_CV ** 2))
            mu = (_sim_math.log(max(mean_daily, 1e-6))
                  - 0.5 * sigma ** 2)

            stockout_days = 0
            total_days = 0
            ad_hoc_events = 0
            wasted_units = 0.0
            delivered_units = 0.0
            late_deliveries = 0

            for trial in range(n_trials):
                stock = mean_daily * SIM_SAFETY_STOCK
                buckets = []
                trial_ad_hoc = False
                trial_late = 0

                for t in DAYS:
                    # ── Layer 4: Vehicle breakdown ──
                    vehicle_available = (rng.random() > breakdown_rate)

                    # ── Layer 2: Delivery with travel-time noise ──
                    if t in v_days and vehicle_available:
                        # Travel time perturbation
                        travel_noise = rng.normal(1.0, travel_cv)
                        travel_noise = max(0.5, travel_noise)  # floor
                        actual_travel = depot_mi * travel_noise

                        # Check if late (simplified: compare against
                        # a representative window close)
                        slots = data.get("slot_windows", {}).get(
                            pc, {}).get(t, [])
                        if slots:
                            # Use first slot's close time
                            _oh, _ch = slots[0][1], slots[0][2]
                            # Nominal arrival
                            nom_speed = 21.7  # mph network median
                            nom_arrival = (data.get("T_DEP", 6.0)
                                           + depot_mi / nom_speed)
                            act_arrival = (data.get("T_DEP", 6.0)
                                           + actual_travel / nom_speed)
                            if act_arrival > _ch:
                                trial_late += 1

                        # Replenish stock
                        future = [d for d in v_days if d > t]
                        gap = ((min(future) - t) if future
                               else (5 - t + min(v_days)))
                        gap = max(1, gap)
                        target = mean_daily * (gap + SIM_SAFETY_STOCK)
                        delivered = max(0.0, target - stock)
                        stock += delivered
                        delivered_units += delivered
                        if delivered > 0:
                            buckets.append([delivered, 0])

                    elif t in v_days and not vehicle_available:
                        # Breakdown: delivery missed, ad-hoc triggered
                        fleet_failures += 1
                        trial_ad_hoc = True
                        ad_hoc_mi = 2 * depot_mi
                        total_adhoc_mi += ad_hoc_mi

                    # ── Layer 1: Stochastic demand consumption ──
                    consumption = float(
                        rng.lognormal(mean=mu, sigma=sigma))
                    total_days += 1

                    if consumption > stock:
                        stockout_days += 1
                        trial_ad_hoc = True
                        stock = mean_daily * SIM_SAFETY_STOCK
                        buckets = ([[stock, 0]] if stock > 0
                                   else [])
                    else:
                        stock -= consumption
                        remaining = consumption
                        kept = []
                        for units, age in buckets:
                            if remaining >= units:
                                remaining -= units
                                continue
                            kept.append([units - remaining, age])
                            remaining = 0.0
                        buckets = [[u, a + 1] for u, a in kept]

                        # ── Layer 3: Wastage (shelf-life expiry) ──
                        surviving = []
                        for units, age in buckets:
                            if age > shelf_days:
                                wasted_units += units
                            else:
                                surviving.append([units, age])
                        buckets = surviving

                    late_deliveries += trial_late

                if trial_ad_hoc:
                    ad_hoc_events += 1

            ad_hoc_rate = ad_hoc_events / max(n_trials, 1)
            records.append(dict(
                pc=pc, product=g,
                mean_daily_demand=round(mean_daily, 2),
                seasonal_factor=s_factor,
                stockout_rate=round(
                    stockout_days / max(total_days, 1), 4),
                ad_hoc_trigger_rate=round(ad_hoc_rate, 4),
                wastage_rate=round(
                    wasted_units / max(delivered_units, 1e-6), 4),
                late_delivery_rate=round(
                    late_deliveries / max(total_days, 1), 4),
                ad_hoc_cost_gbp=round(
                    ad_hoc_rate * (SIM_AD_HOC_FLAT
                                   + SIM_AD_HOC_PER_MI * 2
                                   * depot_mi), 2),
                depot_mi=round(depot_mi, 1),
            ))

    if not records:
        return dict(records=[], summary=None)

    # ── Layer 5: KPI aggregation ──
    mean_stockout = (sum(r["stockout_rate"] for r in records)
                     / len(records))
    mean_wastage_num = sum(
        r["wastage_rate"] * r["mean_daily_demand"]
        for r in records)
    mean_wastage_den = sum(
        r["mean_daily_demand"] for r in records)
    weighted_wastage = (mean_wastage_num / mean_wastage_den
                        if mean_wastage_den else 0.0)
    total_ad_hoc_cost = sum(r["ad_hoc_cost_gbp"] for r in records)
    mean_late = (sum(r["late_delivery_rate"] for r in records)
                 / len(records))

    # Driver-hours estimate from result
    driver_hours = sum(
        rd.get("duration_h", 0)
        for day_routes in result.get("all_routes", [])
        for rd in ([day_routes] if isinstance(day_routes, dict)
                   else (day_routes if isinstance(day_routes, list)
                         else [])))

    # Console output
    print(f"\n  Simulation — {label} (month={month}, "
          f"seasonal={s_factor:.2f}×, {n_trials} trials)")
    print(f"    Stockout rate:    {mean_stockout:.2%}")
    print(f"    Wastage rate:     {weighted_wastage:.2%}")
    print(f"    Late delivery:    {mean_late:.2%}")
    print(f"    Ad-hoc cost/wk:  £{total_ad_hoc_cost:,.0f}")
    print(f"    Fleet failures:  "
          f"{fleet_failures}/{n_trials * len(DAYS)} vehicle-days",
          flush=True)

    return dict(records=records, summary=dict(
        mean_stockout_rate=round(mean_stockout, 4),
        weighted_wastage_rate=round(weighted_wastage, 4),
        mean_late_rate=round(mean_late, 4),
        total_ad_hoc_cost_gbp=round(total_ad_hoc_cost, 2),
        fleet_failures=fleet_failures,
        driver_hours_wk=round(driver_hours, 1),
        seasonal_factor=s_factor,
        month=month,
    ))


# ─────────────────────────────────────────────────────────────────────────────
# P.4  RESULTS EXPORT
# ─────────────────────────────────────────────────────────────────────────────

def sim_export_results(results, tornado=None,
                       out_dir=None):
    """Export scenario grid and tornado results to CSV and JSON."""
    out = _sim_Path(out_dir or SIM_KPI_DIR)
    out.mkdir(parents=True, exist_ok=True)

    # Scenario grid -> CSV
    rows = []
    for key, val in results.items():
        if isinstance(key, tuple) and len(key) == 4:
            shl, month, r_key, e_key = key
            row = dict(SHL=shl, Month=month, Robust=r_key,
                       Fleet=e_key)
            row.update({k: v for k, v in val.items()
                        if not isinstance(v, (dict, list))})
            rows.append(row)
    if rows:
        df = _sim_pd.DataFrame(rows)
        csv_path = out / "scenario_results.csv"
        df.to_csv(csv_path, index=False)
        print(f"\n  Scenario results → {csv_path}")

        # Print decision table
        print(f"\n  {'SHL':<12} {'Month':<8} {'R':>3} {'E':>3} "
              f"{'mi':>8} {'routes':>6} {'CO₂':>7} "
              f"{'waste%':>7} {'stock%':>7} {'adhoc£':>7}")
        print(f"  {'─'*74}")
        for _, r in df.iterrows():
            print(f"  {r.get('SHL',''):<12} "
                  f"{r.get('Month',''):<8} "
                  f"{r.get('Robust',''):>3} "
                  f"{r.get('Fleet',''):>3} "
                  f"{r.get('total_mi',0):>8.1f} "
                  f"{r.get('n_routes',''):>6} "
                  f"{r.get('co2_kg',0):>7.1f} "
                  f"{r.get('wastage_rate',0)*100:>6.2f}% "
                  f"{r.get('stockout_rate',0)*100:>6.2f}% "
                  f"£{r.get('ad_hoc_cost',0):>6.0f}")

    # Tornado -> JSON
    if tornado:
        tornado_path = out / "tornado_results.json"
        with open(tornado_path, "w") as f:
            _sim_json.dump(tornado, f, indent=2, default=str)
        print(f"\n  Tornado results → {tornado_path}")

        # Print tornado chart (text)
        print(f"\n  TORNADO CHART (objective change from baseline)")
        print(f"  {'Parameter':<35} {'-20%':>10} {'base':>10} "
              f"{'+20%':>10} {'swing':>10}")
        print(f"  {'─'*80}")
        for p_key, p_val in tornado.items():
            base = p_val.get("base", 0)
            lo = p_val.get("-20%")
            hi = p_val.get("+20%")
            lo_d = (f"{(lo-base)/base*100:+.1f}%"
                    if lo and base else "INFEAS")
            hi_d = (f"{(hi-base)/base*100:+.1f}%"
                    if hi and base else "INFEAS")
            swing = (abs((hi or 0) - (lo or 0))
                     if hi and lo else 0)
            print(f"  {p_val.get('label', p_key):<35} "
                  f"{lo_d:>10} {base:>10.1f} {hi_d:>10} "
                  f"{swing:>10.1f}")

    print(f"\n{'='*60}")
    print("  EXPORT COMPLETE")
    print(f"{'='*60}", flush=True)


# ─────────────────────────────────────────────────────────────────────────────
# P.5  TRADE-OFF REPORTING
# ─────────────────────────────────────────────────────────────────────────────

def sim_tradeoff_report(results):
    """Print the NHSBT-specific trade-off analysis from scenario results."""
    print(f"\n{'='*60}")
    print("  NHSBT TRADE-OFF ANALYSIS")
    print(f"{'='*60}\n")

    # Separate E0 vs E2 vs E3 results
    e0 = {k: v for k, v in results.items()
           if isinstance(k, tuple) and k[3] == "E0"
           and v.get("status") == "solved"}
    e2 = {k: v for k, v in results.items()
           if isinstance(k, tuple) and k[3] == "E2"
           and v.get("status") == "solved"}
    e3 = {k: v for k, v in results.items()
           if isinstance(k, tuple) and k[3] == "E3"
           and v.get("status") == "solved"}

    # Count infeasible cells per fleet scenario
    e2_infeas = sum(1 for k, v in results.items()
                    if isinstance(k, tuple) and k[3] == "E2"
                    and v.get("status") == "infeasible")
    e3_infeas = sum(1 for k, v in results.items()
                    if isinstance(k, tuple) and k[3] == "E3"
                    and v.get("status") == "infeasible")

    if e2_infeas:
        print(f"  ⚠ E2 (100% EV standard): {e2_infeas} infeasible cells")
        # Which SHLs?
        infeas_shls = set(k[0] for k, v in results.items()
                          if isinstance(k, tuple) and k[3] == "E2"
                          and v.get("status") == "infeasible")
        print(f"    Affected SHLs: {', '.join(sorted(infeas_shls))}")

    if e3_infeas:
        print(f"  ⚠ E3 (EV size mix): {e3_infeas} infeasible cells")
        infeas_shls = set(k[0] for k, v in results.items()
                          if isinstance(k, tuple) and k[3] == "E3"
                          and v.get("status") == "infeasible")
        print(f"    Affected SHLs: {', '.join(sorted(infeas_shls))}")


    # ── EV Electrification Threshold (from scenario results) ──
    ev_range_mi = 180.0  # E2 standard EV range
    e2_total = e2_infeas + len(e2)
    e3_total = e3_infeas + len(e3)
    if e2_total > 0 and e2_infeas == 0:
        # All E2 cells solved → check per-route averages vs range
        worst_avg_mi = max(
            (v.get('total_mi', 0) / max(v.get('n_routes', 1), 1)
             for v in e2.values()), default=0)
        worst_margin = round(ev_range_mi - worst_avg_mi, 1)
        print(f"  ✓ 100% EV electrification (E2) is feasible across "
              f"all {e2_total} cells.")
        print(f"    Worst per-route avg: {worst_avg_mi:.1f} mi  "
              f"(margin: {worst_margin:.1f} mi vs {ev_range_mi:.0f} mi range)")
    elif e2_total > 0:
        print(f"  ⚠ EV electrification (E2): {e2_infeas}/{e2_total} cells "
              f"infeasible (range limit {ev_range_mi:.0f} mi)")

    # ── Capex vs Opex ──
    n_shls = len(set(k[0] for k in e0))
    n_vehicles_e0 = sum(v.get("n_vehicles", 0) for v in e0.values())
    n_vehicles_e2 = sum(v.get("n_vehicles", 0) for v in e2.values())

    mi_e0 = sum(v.get("total_mi", 0) for v in e0.values())
    mi_e2 = sum(v.get("total_mi", 0) for v in e2.values())

    capex_premium = (n_vehicles_e2 * SIM_EV_VAN_CAPEX
                     - n_vehicles_e0 * SIM_DIESEL_VAN_CAPEX
                     + n_shls * SIM_CHARGING_PER_SHL)
    annual_capex = capex_premium / SIM_CAPEX_HORIZON

    opex_e0 = mi_e0 * 0.45 * 52 / max(len(e0), 1) * n_shls
    opex_e2 = mi_e2 * 0.07 * 52 / max(len(e2), 1) * n_shls
    opex_saving = opex_e0 - opex_e2

    print(f"\n  ── Capex vs Opex ──")
    print(f"    Vehicles: E0={n_vehicles_e0}, E2={n_vehicles_e2}")
    print(f"    Capex premium: £{capex_premium:,.0f} "
          f"(amortised over {SIM_CAPEX_HORIZON}yr: "
          f"£{annual_capex:,.0f}/yr)")
    print(f"    Opex saving: £{opex_saving:,.0f}/yr")
    print(f"    Net annual: £{opex_saving - annual_capex:,.0f}/yr")

    # ── Wastage threshold ──
    wastage_e0 = _sim_np.mean([
        v.get("wastage_rate", 0)
        for v in e0.values() if v.get("wastage_rate") is not None])
    print(f"\n  ── Wastage Threshold ──")
    print(f"    Baseline wastage rate: {wastage_e0:.2%}")
    freq_saving = 195296  # from frequency extension
    critical_pp = freq_saving / (SIM_RBC_UNIT_COST * 405184 / 100)
    print(f"    Frequency saving: £{freq_saving:,.0f}/yr")
    print(f"    Critical wastage increase: "
          f"+{critical_pp:.2f} pp wipes out saving")

    print(f"\n{'='*60}\n", flush=True)


# ─────────────────────────────────────────────────────────────────────────────
# P.6  CONVENIENCE ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def run_full_scenario_analysis(shls=None, run_tornado=True,
                                run_grid=True, export=True):
    """One-call entry point for the full scenario + sensitivity analysis.

    Usage:
        results, tornado = run_full_scenario_analysis()
        results, tornado = run_full_scenario_analysis(
            shls=["Basildon", "Newcastle"])  # quick test
    """
    results = {}
    tornado = {}

    if run_grid:
        results = scenario_grid_driver(shls=shls)

    if run_tornado:
        test_shl = (shls[0] if shls
                    else "Basildon")  # fast-solving SHL
        tornado = tornado_sensitivity(shl=test_shl)

    if export and (results or tornado):
        sim_export_results(results, tornado)

    if results:
        sim_tradeoff_report(results)

    return results, tornado

: 

In [ ]:
SIM_MONTHS = ["2026-01", "2026-04", "2026-07", "2026-10"]

results, tornado = run_full_scenario_analysis(
    shls=[
        "Colindale",
    ],
    run_tornado=True,
)


SCENARIO GRID: 3 robust × 3 EV × 4 months × 1 SHLs = 36 cells
  NOTE Colindale: 15 hospital(s) in coords_clean but not required by trips_clean — not modelled: ['P186', 'P266', 'P492', 'P608', 'P609', 'P626', 'P628', 'P634', 'P722', 'P789', 'P791', 'P792'] ...

  ── v28 trips_clean augmentation: slot_windows already complete ──

  Dual-slot hospitals: ['P287', 'P601', 'P602', 'P605', 'P607', 'P614', 'P615', 'P619', 'P622', 'P627', 'P631', 'P633', 'P635', 'P637']
    P287 s=0 (T1): DP=7.0 [CO+lead] AR=8.50h
    P287 s=1 (T2): DP=13.5 [CO+lead] AR=14.50h
    P287 s=2 (T3): DP=19.0 [CO+lead] AR=21.67h
    P601 s=0 (T1): DP=11.0 [CO+lead] AR=11.50h
    P601 s=1 (T2): DP=13.5 [CO+lead] AR=14.25h
    P602 s=0 (T1): DP=9.0 [CO+lead] AR=12.00h
    P602 s=1 (T2): DP=13.5 [CO+lead] AR=16.25h
    P605 s=0 (T1): DP=9.0 [CO+lead] AR=10.42h
    P605 s=1 (T2): DP=13.5 [CO+lead] AR=14.83h
    P607 s=0 (T1): DP=2.0 [CO+lead] AR=3.00h
    P607 s=1 (T2): DP=13.5 [CO+lead] AR=15.50h
    P614 s=0 (T1): DP=2

/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:897: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  lam = [xp.var(name=f"lam_{i}", vartype=xp.continuous, lb=0.0)
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:899: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  y = {vn: xp.var(name=f"y_{vn.replace(' ', '_')}",
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:937: DeprecationWarning: Deprecated in Xpress 9.5: use problem.attributes.objval instead
  lp_obj = rmlp.getObjVal()
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:947: DeprecationWarning: Deprecated in Xpress 9.5: use problem.getDuals instead
  pi_duals[i] = float(rmlp.getDual(cov_cons[i]))
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:955: DeprecationWarning: Depre

    Group DP=9.00h: 10 nodes, ICW→7 routes (268.9mi), LS→3 routes (173.1mi, saved 95.8mi)  pool=14
      §9 CG: 1 iters, |pool|=14, Z_LP=81.68, Z_INT=81.68, gap=0.00%
User solution (heuristic_ws) stored.
FICO Xpress v9.8.1, Hyper, solve started 16:17:48, Aug 17, 2026
Heap usage: 1399KB (peak 1399KB, 144KB system)
Minimizing MILP noname using up to 8 threads and up to 8192MB memory, with these control settings:
OUTPUTLOG = 1
MIPRELSTOP = .001
HEURSEARCHEFFORT = 2
TIMELIMIT = 7199.99999809265
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
      1335 rows          918 cols         6076 elements       726 entities
Presolved problem has:
       685 rows          490 cols         2906 elements       376 entities
LP relaxation tightened
Presolve finished in 0 seconds
Heap usage: 3281KB (peak 4009KB, 144KB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 2.16e-01,  4.37e+01] / [ 8.56e-03,  1.

/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:897: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  lam = [xp.var(name=f"lam_{i}", vartype=xp.continuous, lb=0.0)
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:899: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  y = {vn: xp.var(name=f"y_{vn.replace(' ', '_')}",
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:937: DeprecationWarning: Deprecated in Xpress 9.5: use problem.attributes.objval instead
  lp_obj = rmlp.getObjVal()
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:947: DeprecationWarning: Deprecated in Xpress 9.5: use problem.getDuals instead
  pi_duals[i] = float(rmlp.getDual(cov_cons[i]))
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:955: DeprecationWarning: Depre

    Group DP=9.00h: 10 nodes, ICW→7 routes (268.9mi), LS→3 routes (173.1mi, saved 95.8mi)  pool=14
      §9 CG: 1 iters, |pool|=14, Z_LP=67.13, Z_INT=67.13, gap=0.00%
User solution (heuristic_ws) stored.
FICO Xpress v9.8.1, Hyper, solve started 16:29:59, Aug 17, 2026
Heap usage: 1399KB (peak 1399KB, 255KB system)
Minimizing MILP noname using up to 8 threads and up to 8192MB memory, with these control settings:
OUTPUTLOG = 1
MIPRELSTOP = .001
HEURSEARCHEFFORT = 2
TIMELIMIT = 7199.99999904633
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
      1341 rows          918 cols         6736 elements       726 entities
Presolved problem has:
       691 rows          490 cols         3230 elements       376 entities
LP relaxation tightened
Presolve finished in 0 seconds
Heap usage: 3295KB (peak 4010KB, 255KB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 2.16e-01,  6.47e+01] / [ 8.56e-03,  1.

/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:897: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  lam = [xp.var(name=f"lam_{i}", vartype=xp.continuous, lb=0.0)
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:899: DeprecationWarning: Deprecated in Xpress 9.5: create a linked variable by calling problem.addVariable()
  y = {vn: xp.var(name=f"y_{vn.replace(' ', '_')}",
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:937: DeprecationWarning: Deprecated in Xpress 9.5: use problem.attributes.objval instead
  lp_obj = rmlp.getObjVal()
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:947: DeprecationWarning: Deprecated in Xpress 9.5: use problem.getDuals instead
  pi_duals[i] = float(rmlp.getDual(cov_cons[i]))
/var/folders/rn/q_yjxzpd7kg88pm8j272q6sc0000gn/T/ipykernel_12368/862162077.py:955: DeprecationWarning: Depre

    Group DP=9.00h: 10 nodes, ICW→7 routes (268.9mi), LS→3 routes (173.1mi, saved 95.8mi)  pool=14
      §9 CG: 1 iters, |pool|=14, Z_LP=51.92, Z_INT=51.92, gap=0.00%
User solution (heuristic_ws) stored.
FICO Xpress v9.8.1, Hyper, solve started 16:50:57, Aug 17, 2026
Heap usage: 1399KB (peak 1399KB, 114KB system)
Minimizing MILP noname using up to 8 threads and up to 8192MB memory, with these control settings:
OUTPUTLOG = 1
MIPRELSTOP = .001
HEURSEARCHEFFORT = 2
TIMELIMIT = 7199.99999880791
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
      1341 rows          918 cols         6736 elements       726 entities
Presolved problem has:
       691 rows          490 cols         3230 elements       376 entities
LP relaxation tightened
Presolve finished in 0 seconds
Heap usage: 3295KB (peak 4010KB, 114KB system)

Coefficient range                    original                 solved        
  Coefficients   [min,max] : [ 2.16e-01,  6.47e+01] / [ 8.56e-03,  1.